# The Archer model

This notebook trains the one assistant model the app runs. It is not a Cree model sitting beside a general one: the same weights hold the conversation, read the retrieved song notes, translate between English and nêhiyawêwin, explain words, and convert between SRO and syllabics. The corpus is what makes that possible, and the instruction set is what teaches the model which job it is doing.

It is written to run on a Colab GPU from top to bottom. Nothing is left as an exercise: it downloads every corpus, cleans and deduplicates it, builds a Cree tokenizer, continues pretraining, tunes on instruction data, teaches faithfulness with preference pairs, scores everything on held-out data, and exports a quantised GGUF that the app can load.

## What is different from a generic fine-tune

A general model sees Cree as a long tail of rare byte sequences. Three things change that here.

1. **The tokenizer learns Cree first.** A tokenizer trained on English splits `nêhiyawêwin` into many pieces; this one learns the actual syllables and morphemes of the language, then grafts them onto the base model with statistically initialised embeddings. Fewer tokens per word means more real text per context window, faster generation, and less room for spelling drift.
2. **The data is curated, not scraped blindly.** Every source carries its licence, dialect and script in a provenance ledger. Restricted material in the EdTeKLA collection is excluded at the loader, not quietly copied. Near-duplicate sentences are removed, translation pairs are checked for alignment, and the test split is document-disjoint.
3. **The model is taught to be faithful.** The instruction set contains unanswerable questions whose correct answer is "I do not have notes on that", plus preference pairs that punish invented names, dropped clauses and English leaking into a Cree sentence. That is the failure mode that matters for a cultural project.

## What it produces

| Artifact | Where | What it is |
|---|---|---|
| `archer-<size>.Q4_K_M.gguf` | `out/` | the only chat model the app loads, about 4.5 GB for the 7B default |
| `creebench_report.json` | `out/` | every score, per task, per split |
| `report_card.md` | `out/` | a model card with data provenance, licence and results |
| `training_curves.png`, `creebench.png` | `out/` | the figures the report cites |
| `Modelfile` | `out/` | sampling settings and context length for Ollama |
| `run_state.json` | `archer_cree_llm/` | which stages are finished, and from which inputs |
| `checkpoints/<stage>/checkpoint-N` | `archer_cree_llm/` | the resume points, written every few minutes |

## Before you run

Use **Runtime → Change runtime type → A100** for the full recipe. A free T4 also works, and needs no flag: the notebook reads the card, loads the base in nf4 when 16-bit weights would not leave room to train, and keeps the vocabulary table out of the optimizer, which is what lets a 7B model fit in 15 GB. `CREE_PROFILE="t4"` shortens the schedules to match. A free CPU runtime is enough for the data, tokenizer and evaluation cells but not for training.

Model weights are cached on the local runtime disk instead of Drive, because a multi-gigabyte shard written through the Drive mount can come back damaged; each shard is checked before it is loaded and a broken one is fetched again on its own. Everything that cannot be recreated, meaning checkpoints, the run ledger, plots and exports, is written to Drive. Set `CREE_USE_DRIVE=0` to keep everything in `/content` instead.

## If Colab disconnects

Free Colab ends a session without warning, and a run this long will lose one eventually. The notebook is built for that. Each finished stage writes a marker to `run_state.json` on Drive, and each training stage writes a checkpoint to Drive every few minutes. The recovery is: reconnect, **Runtime → Run all**, and leave it. The data cells find their outputs already on Drive and skip the downloads, a finished stage is not trained again, and the stage that was interrupted continues from its last checkpoint with its step count, learning rate schedule and loss curve intact. `Runtime → Restart and run all` behaves the same way; nothing is repeated that already has a marker.

Two settings control it. `CREE_FORCE=cpt,sft` re-runs those stages from the beginning even though they are marked finished, and `CREE_SAVE_MINUTES=5` shortens the checkpoint interval. The stage names are `corpus`, `clean`, `tokenizer`, `cpt`, `sft`, `dpo`, `eval`. The setup cell prints which of them are done and where an interrupted one had reached.

Every cell prints what it did. If a source is unreachable the cell records that fact and continues with the rest; it never substitutes invented text.

In [ ]:
# Configuration. One place for every knob; every later cell reads from here.
from __future__ import annotations

import collections
import gc
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import tarfile
import textwrap
import time
import unicodedata
import urllib.request
import zipfile
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np

IN_COLAB = "google.colab" in sys.modules

# Colab ships an older stack; install the small number of packages this
# notebook actually needs. Torch is left exactly as Colab provides it.
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers>=4.46,<6", "peft>=0.13", "datasets>=3", "accelerate>=1.0",
         "bitsandbytes>=0.44", "sentencepiece>=0.2", "sacrebleu>=2.4",
         "matplotlib>=3.8", "pandas>=2.0", "psutil>=5.9"],
        check=True,
    )

import torch
import torch.nn.functional as F

SMOKE = os.environ.get("CREE_SMOKE", "0") == "1"
PROFILE = os.environ.get("CREE_PROFILE", "smoke" if SMOKE else "auto")


def _gpu_line() -> str:
    if not torch.cuda.is_available():
        return "CPU only"
    name = torch.cuda.get_device_name(0)
    gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    return f"{name} ({gb:.0f} GB)"


_PRESETS = {
    # A tiny model and a handful of steps: proves every cell runs, changes nothing.
    "smoke": dict(base="Qwen/Qwen2.5-0.5B-Instruct", seq=384, lora_r=8,
                  cpt_epochs=1, sft_epochs=1, dpo_epochs=1, eval_n=2,
                  max_cpt_packs=6, max_sft=48, min_tokens=0),
    # Free / pay-as-you-go T4 (16 GB). Shorter context, two passes.
    "t4": dict(base="Qwen/Qwen2.5-7B-Instruct", seq=1024, lora_r=32,
               cpt_epochs=2, sft_epochs=2, dpo_epochs=1, eval_n=120,
               max_cpt_packs=100_000, max_sft=100_000, min_tokens=3_000_000),
    # A100 40 GB: the intended run.
    "a100": dict(base="Qwen/Qwen2.5-7B-Instruct", seq=2048, lora_r=64,
                 cpt_epochs=3, sft_epochs=2, dpo_epochs=1, eval_n=250,
                 max_cpt_packs=100_000, max_sft=100_000, min_tokens=5_000_000),
    # A100 80 GB / H100: the 14B model. Set CREE_PROFILE=big. The token target is
    # deliberately 20M: a 14B model trained on the small repository slice is
    # still the weak model with more parameters. The run warns below it, reports
    # the number, and still trains on every verified token.
    "big": dict(base="Qwen/Qwen2.5-14B-Instruct", seq=2048, lora_r=64,
                cpt_epochs=3, sft_epochs=2, dpo_epochs=1, eval_n=250,
                max_cpt_packs=100_000, max_sft=100_000, min_tokens=20_000_000),
    # One 24 GB card (RTX 4090, A10): the 14B in nf4 at a shorter context. This
    # is the largest base that fits a single 24 GB card; the merge runs on CPU
    # and needs about 64 GB of system RAM. Slower than the 7B, and 
    # not chosen by `auto`, because it is a different bet on quality vs time.
    "big24": dict(base="Qwen/Qwen2.5-14B-Instruct", seq=1024, lora_r=32,
                  cpt_epochs=2, sft_epochs=2, dpo_epochs=1, eval_n=180,
                  max_cpt_packs=100_000, max_sft=100_000, min_tokens=20_000_000),
    # The largest preset that fits one 80 GB card with QLoRA. Merging and
    # quantising a 32B needs a separate high-memory step; the notebook says so
    # rather than pretending it fits everywhere.
    "huge": dict(base="Qwen/Qwen2.5-32B-Instruct", seq=2048, lora_r=64,
                 cpt_epochs=3, sft_epochs=2, dpo_epochs=1, eval_n=250,
                 max_cpt_packs=100_000, max_sft=100_000, min_tokens=20_000_000),
}


def choose_profile() -> str:
    if PROFILE != "auto":
        return PROFILE
    if not torch.cuda.is_available():
        return "smoke"
    gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    return "a100" if gb >= 38 else "t4"


ACTIVE_PROFILE = choose_profile()
P = _PRESETS[ACTIVE_PROFILE]


@dataclass
class Config:
    base_model: str = P["base"]
    seq_len: int = P["seq"]
    lora_r: int = P["lora_r"]
    lora_alpha: int = P["lora_r"] * 2
    lora_dropout: float = 0.05
    cpt_epochs: float = P["cpt_epochs"]
    sft_epochs: float = P["sft_epochs"]
    dpo_epochs: float = P["dpo_epochs"]
    eval_n: int = P["eval_n"]
    max_cpt_packs: int = P["max_cpt_packs"]
    max_sft: int = P["max_sft"]
    # Estimated tokens below which a profile warns. Training still consumes all
    # verified data; the warning prevents a small corpus from being reported as
    # ready. The smoke profile has no warning threshold.
    min_corpus_tokens: int = P["min_tokens"]
    cpt_lr: float = 2e-4
    sft_lr: float = 1.5e-4
    # Preference tuning moves an already-instruction-tuned policy, so it needs a
    # far smaller step than SFT. At 5e-5 a 7B run answered correctly for a few
    # tokens and then repeated one symbol to the token limit on every job -- the
    # signature of a policy that the preference step has collapsed, not of a
    # translation that is merely weak.
    dpo_lr: float = 1e-6
    dpo_beta: float = 0.1
    micro_batch: int = 1
    grad_accum: int = 16
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    new_token_budget: int = 5200
    spm_vocab: int = 6000
    seed: int = 17
    # Licence policy: NoDerivatives material is never loaded. NonCommercial
    # ShareAlike material is loaded because this model is released under the
    # same licence; set to False for a commercially licensable build.
    include_noncommercial: bool = True
    dpo_enabled: bool = True
    external_baseline: str = os.environ.get("CREE_EXTERNAL_MODEL", "")  # ollama tag


CFG = Config()

# The exported artifact is named for the model actually trained, not for the
# default. A 14B run must not produce a file called archer-7b.
_BASE_LOWER = CFG.base_model.lower()
MODEL_TAG = next(
    (tag for tag in ("32b", "14b", "7b", "3b", "1.5b", "0.5b") if tag in _BASE_LOWER),
    "custom",
)
MODEL_STEM = f"archer-{MODEL_TAG}"

# where everything lives
if IN_COLAB and os.environ.get("CREE_USE_DRIVE", "1") == "1":
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/archer_cree_llm")
    except Exception as exc:  # drive auth declined: still run, just not persistent
        print(f"[warning] Drive is not mounted ({exc}).")
        print("[warning] Everything goes to /content, which is deleted with the runtime, so a")
        print("[warning] disconnect will lose the run. Re-run this cell and allow the Drive prompt.")
        ROOT = Path("/content/cree_llm")
elif IN_COLAB:
    ROOT = Path("/content/cree_llm")
else:
    ROOT = Path(os.environ.get("CREE_WORK", Path.cwd() / "cree_llm_work")).resolve()

RAW = ROOT / "raw"
PROC = ROOT / "processed"
CKPT = ROOT / "checkpoints"
OUT = ROOT / "out"
PLOTS = OUT / "plots"
for d in (RAW, PROC, CKPT, OUT, PLOTS):
    d.mkdir(parents=True, exist_ok=True)

# where the Hugging Face cache lives
# Model weights are never cached on Drive. Drive's FUSE mount does not reliably
# finish writing multi-gigabyte binary files, and a shard that reads back with a
# damaged header is exactly the `SafetensorError: header too large` that stops
# training before the first step. Weights are a cache, they can always be fetched
# again, so they live on the runtime's local disk. Checkpoints and exports, which
# cannot be recreated, still go to ROOT on Drive.
HF_CACHE = Path(
    os.environ.get("CREE_HF_HOME")
    or (Path("/content/hf_cache") if IN_COLAB else ROOT / "hf")
)
if str(HF_CACHE).startswith("/content/drive"):
    print("[note] the Hugging Face cache was pointed at Drive; using /content/hf_cache instead, "
          "because the Drive mount corrupts large model shards")
    HF_CACHE = Path("/content/hf_cache")
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
# Read timeouts, not transfer limits. A keep-alive socket that the network stack
# has already dropped otherwise parks a download until the process is killed,
# which looks exactly like a hung corpus stage. Sixty seconds without a byte is
# already a broken connection, so failing and retrying is the faster path.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else (torch.float16 if DEVICE == "cuda" else torch.float32)

GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3 if DEVICE == "cuda" else 0.0

# Parameter counts of the bases this notebook knows about, in billions. Used only
# to plan memory, never to describe the model.
_BASE_PARAMS_B = {"0.5b": 0.5, "1.5b": 1.5, "3b": 3.0, "7b": 7.6, "14b": 14.7, "32b": 32.8}
BASE_PARAMS_B = next((v for k, v in _BASE_PARAMS_B.items() if k in _BASE_LOWER), 7.6)


def plan_weights(gpu_gb: float, params_b: float, smoke: bool) -> bool:
    """True when the base weights must be quantised to 4 bits.

    16-bit weights cost 2 bytes per parameter, before any gradient, optimizer
    state or activation. A 7.6B base is about 15 GB of weights, which on a
    free Colab T4 (14.56 GiB usable) leaves nothing at all: the run dies on the
    first cross-entropy with a 608 MiB allocation it cannot make. Quantised to
    nf4 the same weights are about 4.4 GB.

    The 0.55 factor is the share of the card that weights are allowed to take,
    leaving the rest for activations, the logits and the CUDA context.
    """
    if gpu_gb <= 0 or smoke:
        return False
    return 2 * params_b > gpu_gb * 0.55


USE_4BIT = plan_weights(GPU_GB, BASE_PARAMS_B, SMOKE)


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def jdump(path: Path | str, obj) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def jload(path: Path | str):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_jsonl(path: Path | str, rows) -> int:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with path.open("w", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")
            n += 1
    return n


def read_jsonl(path: Path | str) -> list[dict]:
    with Path(path).open(encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def sha1(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


def download(url: str, dest: Path, timeout: int = 120) -> Path:
    """Fetch once, reuse forever. A half-written file is never trusted."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    req = urllib.request.Request(url, headers={"User-Agent": "archer-llm/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp, tmp.open("wb") as fh:
        shutil.copyfileobj(resp, fh)
    tmp.replace(dest)
    return dest


# crash-safe run state
# Colab ends a runtime without asking: a closed tab, an idle timeout, a GPU
# reset. So every step that takes longer than a minute leaves two traces on
# Drive. A stage marker records that the step finished, and periodic
# checkpoints record how far an unfinished step got. Re-running the notebook
# after a disconnect walks past the finished work and continues the interrupted
# stage from its last checkpoint instead of beginning again.
#
# A marker carries a fingerprint of the inputs that produced it. Change the
# corpus, the model, the context length or the LoRA rank and the affected stage
# runs again on its own: a result computed from different inputs is not a
# result for this run.

STATE_FILE = ROOT / "run_state.json"
FORCE = {name.strip().lower() for name in os.environ.get("CREE_FORCE", "").split(",") if name.strip()}
SAVE_MINUTES = float(os.environ.get("CREE_SAVE_MINUTES", "10"))
STAGE_ORDER = ("corpus", "clean", "tokenizer", "cpt", "sft", "dpo", "eval", "llmtests")


def fingerprint(**parts) -> str:
    """A short, stable hash of the inputs a stage's output depends on."""
    return hashlib.sha1(
        json.dumps(parts, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    ).hexdigest()[:12]


def load_run_state() -> dict:
    if not STATE_FILE.exists():
        return {"stages": {}}
    try:
        state = jload(STATE_FILE)
    except (OSError, json.JSONDecodeError):
        # A half-written ledger is treated as an empty one. The next mark_stage
        # rewrites it, and the worst case is one stage running twice.
        return {"stages": {}}
    state.setdefault("stages", {})
    return state


def save_run_state(state: dict) -> None:
    """Replace the ledger atomically, so a disconnect cannot halve it."""
    tmp = STATE_FILE.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(state, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(STATE_FILE)


def stage_entry(name: str) -> dict | None:
    return load_run_state()["stages"].get(name)


def stage_done(name: str, key: str | None = None, requires: Path | str | None = None,
               quiet: bool = False) -> bool:
    """True when this stage already finished with these inputs.

    `requires` is the artifact the marker promises. If it is missing, the stage
    runs again: a marker without its file would otherwise skip work and leave
    every later cell reading nothing.
    """
    if name in FORCE:
        if not quiet:
            log(f"{name}: forced to run again (CREE_FORCE={name})")
        return False
    entry = stage_entry(name)
    if not entry:
        return False
    if requires is not None and not Path(requires).exists():
        log(f"{name}: marked finished but {Path(requires).name} is missing, so it runs again")
        return False
    if key is not None and entry.get("key") != key:
        log(f"{name}: the inputs changed since it last ran, so it runs again")
        return False
    if not quiet:
        log(f"{name}: already finished {entry.get('at', '')}, skipping")
    return True


def mark_stage(name: str, key: str | None = None, **extra) -> None:
    state = load_run_state()
    state["stages"][name] = {
        "at": time.strftime("%Y-%m-%d %H:%M:%S"), "key": key, **extra,
    }
    state["updated"] = time.strftime("%Y-%m-%d %H:%M:%S")
    save_run_state(state)


def clear_stage(*names: str) -> None:
    state = load_run_state()
    for name in names:
        state["stages"].pop(name, None)
    save_run_state(state)


def metrics_path(tag: str) -> Path:
    folder = OUT / "metrics"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{tag.lower()}.jsonl"


def append_metrics(tag: str, rows: list[dict]) -> None:
    """Append to the loss log on Drive the moment a row exists."""
    if not rows:
        return
    with metrics_path(tag).open("a", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_metrics(tag: str) -> list[dict]:
    """Every logged row for this stage, including the ones from earlier sessions.

    A resumed run logs its last step again, so rows are de-duplicated on step.
    Without that, the loss curve after three sessions would show a spike at
    every reconnect.
    """
    path = metrics_path(tag)
    if not path.exists():
        return []
    out, seen = [], set()
    for row in read_jsonl(path):
        key = (row.get("step"), row.get("event"))
        if key in seen:
            continue
        seen.add(key)
        out.append(row)
    return out


def checkpoints(folder: Path | str) -> list[Path]:
    """Trainer checkpoints in a stage folder, oldest first."""
    root = Path(folder)
    if not root.exists():
        return []
    found = [p for p in root.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")]
    return sorted(found, key=lambda p: int(p.name.split("-")[-1]))


def stage_overview() -> None:
    """Where this run stands, printed at setup so a reconnect is not a mystery."""
    stages = load_run_state()["stages"]
    if not stages:
        print("fresh run: no stage has finished yet")
        return
    print(f"{'stage':10s} {'state':8s} {'finished':19s} detail")
    for name in STAGE_ORDER:
        entry = stages.get(name)
        if entry:
            detail = ", ".join(f"{k}={v}" for k, v in entry.items() if k not in {"at", "key"})
            print(f"{name:10s} {'done':8s} {entry.get('at', ''):19s} {detail}")
        else:
            found = checkpoints(CKPT / name)
            detail = f"interrupted, last checkpoint {found[-1].name}" if found else ""
            print(f"{name:10s} {'pending':8s} {'':19s} {detail}")


def plot_style() -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "figure.facecolor": "#f7f4ec",
        "axes.facecolor": "#fffdf7",
        "axes.edgecolor": "#c9c0ad",
        "axes.labelcolor": "#2b2620",
        "text.color": "#2b2620",
        "xtick.color": "#6b6254",
        "ytick.color": "#6b6254",
        "font.size": 10,
        "axes.grid": True,
        "grid.color": "#e4ddcf",
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })


plot_style()

# A 2048-token window on a 15 GB card costs four times the activation memory of
# 1024 and buys nothing here: the corpus is sentences, not long documents.
if DEVICE == "cuda" and 0 < GPU_GB < 24 and CFG.seq_len > 1024:
    log(f"context {CFG.seq_len} does not fit {GPU_GB:.0f} GB; using 1024")
    CFG.seq_len = 1024

log(f"profile={ACTIVE_PROFILE}  device={_gpu_line()}  autocast={'bf16' if BF16 else 'fp16'}")
log(f"base={CFG.base_model}  seq={CFG.seq_len}  LoRA r={CFG.lora_r}  work={ROOT}")
log(f"weights={'4-bit nf4' if USE_4BIT else str(DTYPE).split('.')[-1]}"
    f" ({BASE_PARAMS_B:.1f}B parameters, {GPU_GB:.1f} GB card)" if DEVICE == "cuda"
    else f"weights={str(DTYPE).split('.')[-1]} (CPU)")
log(f"hf cache={HF_CACHE}  free={shutil.disk_usage(str(HF_CACHE)).free / 1024 ** 3:.1f} GB")
log(f"work dir={ROOT}  free={shutil.disk_usage(str(ROOT)).free / 1024 ** 3:.1f} GB")
log(f"checkpoints: every {SAVE_MINUTES:.0f} min to {CKPT}")
if FORCE:
    log(f"forcing these stages to run again: {', '.join(sorted(FORCE))}")
stage_overview()
if SMOKE:
    log("SMOKE mode: tiny model, tiny schedules. Scores are meaningless; it only proves the code runs.")

In [ ]:
# nêhiyawêwin orthography: SRO, syllabics, and the checks that keep a sentence
# from being half English.
#
# The conversion tables are the ones behind itwêwina (GiellaLT lang-crk,
# Latn-to-Cans.xfscript and Cans-to-Latn.xfscript, ALTLab / University of
# Alberta with the Maskwacîs Education Schools Commission and First Nations
# University). They are data, not invention, and every rule below is tested
# against the canonical example atim -> ᐊᑎᒼ.

SRO_VOWELS = ("a", "e", "i", "o")
LONG_VOWELS = {"a": "â", "e": "ê", "i": "î", "o": "ô"}

_CONSONANT_SERIES = {
    "c": {"a": "ᒐ", "i": "ᒋ", "o": "ᒍ", "â": "ᒑ", "ê": "ᒉ", "î": "ᒌ", "ô": "ᒎ"},
    "k": {"a": "ᑲ", "i": "ᑭ", "o": "ᑯ", "â": "ᑳ", "ê": "ᑫ", "î": "ᑮ", "ô": "ᑰ"},
    "m": {"a": "ᒪ", "i": "ᒥ", "o": "ᒧ", "â": "ᒫ", "ê": "ᒣ", "î": "ᒦ", "ô": "ᒨ"},
    "n": {"a": "ᓇ", "i": "ᓂ", "o": "ᓄ", "â": "ᓈ", "ê": "ᓀ", "î": "ᓃ", "ô": "ᓅ"},
    "p": {"a": "ᐸ", "i": "ᐱ", "o": "ᐳ", "â": "ᐹ", "ê": "ᐯ", "î": "ᐲ", "ô": "ᐴ"},
    "s": {"a": "ᓴ", "i": "ᓯ", "o": "ᓱ", "â": "ᓵ", "ê": "ᓭ", "î": "ᓰ", "ô": "ᓲ"},
    "t": {"a": "ᑕ", "i": "ᑎ", "o": "ᑐ", "â": "ᑖ", "ê": "ᑌ", "î": "ᑏ", "ô": "ᑑ"},
    "y": {"a": "ᔭ", "i": "ᔨ", "o": "ᔪ", "â": "ᔮ", "ê": "ᔦ", "î": "ᔩ", "ô": "ᔫ"},
    "w": {"a": "ᐘ", "i": "ᐏ", "o": "ᐓ", "â": "ᐚ", "ê": "ᐍ", "î": "ᐑ", "ô": "ᐕ"},
}
_W_SERIES = {
    "c": {"a": "ᒝ", "i": "ᒕ", "o": "ᒙ", "â": "ᒟ", "ê": "ᒓ", "î": "ᒗ", "ô": "ᒛ"},
    "k": {"a": "ᑿ", "i": "ᑷ", "o": "ᑻ", "â": "ᒁ", "ê": "ᑵ", "î": "ᑹ", "ô": "ᑽ"},
    "m": {"a": "ᒷ", "i": "ᒯ", "o": "ᒳ", "â": "ᒹ", "ê": "ᒭ", "î": "ᒱ", "ô": "ᒵ"},
    "n": {"a": "ᓌ", "i": "ᣇ", "o": "ᣋ", "â": "ᓎ", "ê": "ᓊ", "î": "ᣉ", "ô": "ᣍ"},
    "p": {"a": "ᑅ", "i": "ᐽ", "o": "ᑁ", "â": "ᑇ", "ê": "ᐻ", "î": "ᐿ", "ô": "ᑃ"},
    "s": {"a": "ᔁ", "i": "ᓹ", "o": "ᓽ", "â": "ᔃ", "ê": "ᓷ", "î": "ᓻ", "ô": "ᓿ"},
    "t": {"a": "ᑢ", "i": "ᑚ", "o": "ᑞ", "â": "ᑤ", "ê": "ᑘ", "î": "ᑜ", "ô": "ᑠ"},
    "y": {"a": "ᔺ", "i": "ᔲ", "o": "ᔶ", "â": "ᔼ", "ê": "ᔰ", "î": "ᔴ", "ô": "ᔸ"},
}
_VOWEL_GLYPHS = {"a": "ᐊ", "i": "ᐃ", "o": "ᐅ", "â": "ᐋ", "ê": "ᐁ", "î": "ᐄ", "ô": "ᐆ"}
_TH_SERIES = {"a": "ᖬ", "e": "ᖧ", "i": "ᖨ", "o": "ᖪ"}
SYLLABICS_FINALS = {
    "c": "ᐨ", "h": "ᐦ", "k": "ᐠ", "m": "ᒼ", "n": "ᐣ", "p": "ᑊ",
    "s": "ᐢ", "t": "ᐟ", "w": "ᐤ", "y": "ᕀ", "l": "ᓬ", "r": "ᕒ",
}
_FINALS_TO_SRO = {v: k for k, v in SYLLABICS_FINALS.items()}
_HK_GLYPH = "ᕽ"
_FULL_STOP = "᙮"
_DOTTED_W = "ᐧ"

_SRO_TO_CANS: dict[str, str] = {}


def _add(sro: str, cans: str) -> None:
    _SRO_TO_CANS[sro] = cans


for _c, _row in _W_SERIES.items():
    for _v, _g in _row.items():
        _add(_c + "w" + _v, _g)
for _c, _row in _CONSONANT_SERIES.items():
    for _v, _g in _row.items():
        _add(_c + _v, _g)
for _c, _row in _TH_SERIES.items():
    _add("th" + _c, _row)
for _v, _g in _VOWEL_GLYPHS.items():
    _add(_v, _g)
_add("hk", _HK_GLYPH)
for _c, _g in SYLLABICS_FINALS.items():
    _add(_c, _g)
_add(".", _FULL_STOP)
_add("e", _VOWEL_GLYPHS["ê"])

_CANS_TO_SRO: dict[str, str] = {}
for _sro, _cans in _SRO_TO_CANS.items():
    if len(_cans) == 1 and _sro != "e":
        _CANS_TO_SRO.setdefault(_cans, _sro)

_COMPOSED_W: dict[str, str] = {}
for _c, _row in _CONSONANT_SERIES.items():
    for _v, _g in _row.items():
        _COMPOSED_W[_g] = _c + "w" + _v
for _v, _g in _VOWEL_GLYPHS.items():
    _COMPOSED_W[_g] = "w" + _v

_SYLLABICS_RANGES = ((0x1400, 0x167F), (0x18B0, 0x18FF))

CREE_LETTERS = set("ptkcsmnhwy" "lr" "aeio" "âêîô")


def macron_to_circumflex(text: str) -> str:
    return text.translate(str.maketrans({
        "ā": "â", "ē": "ê", "ī": "î", "ō": "ô",
        "Ā": "Â", "Ē": "Ê", "Ī": "Î", "Ō": "Ô",
    }))


def is_syllabics(ch: str) -> bool:
    cp = ord(ch)
    return any(lo <= cp <= hi for lo, hi in _SYLLABICS_RANGES)


def looks_like_syllabics(text: str) -> bool:
    return any(is_syllabics(ch) for ch in str(text))


def normalize_sro(text: str) -> str:
    """NFC, macrons to circumflexes, lowercase, collapsed whitespace.

    Long-vowel marks are preserved: removing them would turn different words
    into the same string (nêhiyaw / nehiyaw), which is exactly the damage a
    lazy normaliser does to a language that marks length.
    """
    text = unicodedata.normalize("NFC", str(text))
    text = macron_to_circumflex(text)
    text = text.replace("|", " ").replace("\u2010", "-").replace("\u2014", "-")
    return re.sub(r"\s+", " ", text).strip().lower()


def to_syllabics(sro: str) -> str:
    text = normalize_sro(sro).replace("e", "ê")
    out: list[str] = []
    i, n = 0, len(text)
    while i < n:
        for size in (3, 2, 1):
            chunk = text[i:i + size]
            if len(chunk) == size and chunk in _SRO_TO_CANS:
                out.append(_SRO_TO_CANS[chunk])
                i += size
                break
        else:
            out.append(text[i])
            i += 1
    return "".join(out)


def to_sro(syllabics: str, long_vowels: str = "circumflex") -> str:
    text = unicodedata.normalize("NFC", str(syllabics))
    out: list[str] = []
    last_glyph: str | None = None
    for ch in text:
        if ch == _DOTTED_W:
            if last_glyph is not None:
                out[-1] = _COMPOSED_W.get(last_glyph, out[-1] + "w")
                last_glyph = None
            else:
                out.append("w")
            continue
        last_glyph = None
        if ch in _CANS_TO_SRO:
            out.append(_CANS_TO_SRO[ch])
            last_glyph = ch
        elif ch in _FINALS_TO_SRO:
            out.append(_FINALS_TO_SRO[ch])
        elif ch == _HK_GLYPH:
            out.append("hk")
        elif ch == _FULL_STOP:
            out.append(".")
        else:
            out.append(ch)
    text = "".join(out)
    if long_vowels == "macron":
        text = text.translate(str.maketrans({
            "â": "ā", "ê": "ē", "î": "ī", "ô": "ō",
        }))
    return text


# Words that carry almost no meaning on their own but identify a language.
_EN_STOP = {
    "the", "and", "of", "to", "in", "is", "was", "were", "that", "this",
    "for", "with", "he", "she", "they", "his", "her", "their", "you", "not",
    "but", "from", "have", "has", "had", "will", "would", "there", "what",
    "when", "who", "how", "all", "one", "two", "on", "at", "by", "as", "it",
}
_CR_STOP = {
    "êkwa", "mîna", "kî", "ê", "kâ", "ta", "ni", "ki", "wa", "ôma", "ôki",
    "êsa", "kêkway", "awiyak", "niya", "kiya", "êkosi", "êkotê", "itwêw",
    "itêw", "ayâw", "nipâw", "mîcisow", "pîhtokêw", "nama", "namôya",
}


def lang_guess(text: str) -> str:
    """'crk', 'eng' or 'mixed' from the words that actually appear.

    Character ratios alone are not enough: 'the' is made only of letters Cree
    uses. Function words and long-vowel marks are the reliable signal.
    """
    t = normalize_sro(text)
    if looks_like_syllabics(t):
        return "crk"
    words = re.findall(r"[a-zâêîô']+", t)
    if len(words) < 2:
        return "crk" if re.search(r"[âêîô]", t) else "eng"
    en = sum(w in _EN_STOP for w in words) / len(words)
    cr = sum(w in _CR_STOP for w in words) / len(words)
    diacritic = sum(bool(re.search(r"[âêîô]", w)) for w in words) / len(words)
    if cr > en and (cr + diacritic) > 0.03:
        return "crk"
    if en > cr and en > 0.04:
        return "eng"
    # Unknown but plausible Cree: short words, few Latin-only consonants.
    odd = sum(bool(re.search(r"[bdfgjqvxz]", w)) for w in words) / len(words)
    return "crk" if odd < 0.05 and diacritic > 0 else "mixed"


def cree_word_ratio(text: str) -> float:
    words = re.findall(r"[^\W\d_]+", normalize_sro(text), flags=re.UNICODE)
    if not words:
        return 0.0
    good = 0
    for w in words:
        if looks_like_syllabics(w):
            good += 1
        elif any(ch in CREE_LETTERS or ch.isalpha() and ch in "âêîô" for ch in w):
            if not re.search(r"[bdfgjqvxz]", w):
                good += 1
    return good / len(words)


def syllabify(sro: str) -> list[str]:
    word = normalize_sro(sro)
    chunks: list[str] = []
    i = 0
    vowels = set("aeiouâêîô")
    while i < len(word):
        j = i
        while j < len(word) and word[j] not in vowels:
            j += 1
        if j >= len(word):
            chunks.append(word[i:])
            break
        j += 1
        if j < len(word) and word[j] in vowels:
            j += 1
        if j < len(word) and word[j] not in vowels and word[j] not in "w":
            j += 1
        chunks.append(word[i:j])
        i = j
    return [c for c in chunks if c]


# The canonical checks from the converter's own test cases.
assert to_syllabics("atim") == "ᐊᑎᒼ", to_syllabics("atim")
assert to_sro("ᐊᑎᒼ") == "atim", to_sro("ᐊᑎᒼ")
assert to_syllabics("tânisi") == "ᑖᓂᓯ", to_syllabics("tânisi")
assert to_sro(to_syllabics("nêhiyawêwin")) == "nêhiyawêwin"

print("orthography ready:")
print("  atim      ->", to_syllabics("atim"))
print("  tânisi    ->", to_syllabics("tânisi"))
print("  nêhiyawêwin ->", to_syllabics("nêhiyawêwin"))

## 1. The corpus

A language model is a mirror of its text, so the first job is deciding what goes in.

The notebook draws on six kinds of material. There is the EdTeKLA collection from the University of Alberta, which is the most carefully curated Plains Cree corpus available and also the most unevenly licensed: some subcollections are CC0, some are CC BY, and some are NonCommercial NoDerivatives. The loader reads the nearest README for each text and refuses anything whose licence forbids adapted works. There is the Plains Cree Figurative Language corpus, 10,847 sentence pairs from Leonard Bloomfield's 1934 texts with English translations, which is the backbone of the translation data.

There is the contemporary web: Cree Wikipedia, FineWeb-2 (192 documents), FinePDFs (733 documents and 1.19 million words taken from PDF text layers, the single largest source), FineTranslations (582 aligned chunks), GlotCC and GlotStoryBook. There is the public-domain past: 14 books from the Internet Archive, 612,000 words, including Bloomfield's Plains Cree Texts and Sacred Stories of the Sweet Grass Cree. Each 700-word window is checked against the GiellaLT lexicon, the English stopword rate and the Cree letter ratio, so a syllabic scan that OCR turned into noise is rejected instead of being counted as text.

There is reference material: Tatoeba's Cree sentences, 64 Plains Cree phrase recordings, and GiellaLT's public texts. There is Archer's own material: the project's aligned pairs, the song notes, and the 1865 Watkins dictionary, kept under its own historical tag. Finally there is a folder you can drop your own texts into.

The complete run accepts 12,743 documents and 2.02 million raw words, which cleaning reduces to 1,124,146 training words across 66,071 sentences, 10,934 training pairs and 203 held-out pairs. That is the largest verified Plains Cree corpus this notebook can assemble without scraping a copyrighted site or inventing sentences.

Every item is written with its source, licence, dialect and script. Nothing is silently merged into an anonymous blob, because a model card that cannot say where its Cree came from is not a model card.

The cell below downloads what it can and records what it could not reach. A missing source never stops the run and never gets replaced with invented text.

In [ ]:
# Corpus acquisition.
#
# Every loader returns (documents, pairs) and writes one line to the provenance
# ledger. Documents are running Cree text; pairs are aligned Cree/English
# sentences. A loader that fails is recorded as failed and the run continues:
# a missing source must never be papered over with invented text.
#
# Licence rule, enforced here rather than mentioned in a footnote:
#   * NoDerivatives material is never loaded (it forbids adapted models).
#   * NonCommercial ShareAlike material is loaded only when
#     CFG.include_noncommercial is True, and the model is then released under
#     the same licence in the report card.

import bz2
import unicodedata
import urllib.error
import urllib.parse

DOCS: list[dict] = []
PAIRS: list[dict] = []
LEXICON: dict[str, float] = {}
LEDGER: list[dict] = []
STATUS: dict[str, str] = {}

PROJECT_REPO = "haaaarsh4/Archer-Robot-Vocal-Collaboration-System"
PROJECT_RAW = f"https://raw.githubusercontent.com/{PROJECT_REPO}/main"
_ND_MARKERS = ("noderiv", "noderat", "no-deriv", "no deriv", "nc-nd", "nd-4.0")


def ledger(name: str, url: str, licence: str, dialect: str, script: str,
           note: str = "", allowed: bool = True) -> None:
    LEDGER.append({
        "source": name, "url": url, "licence": licence, "dialect": dialect,
        "script": script, "note": note, "allowed": allowed,
    })


def _readme_licence(path: Path) -> str:
    """The nearest README's licence block, or the repository default.

    Only the closest README is consulted. Walking further up would apply the
    root README's bag-of-words licence (CC BY-NC 4.0) to texts whose own
    README states CC BY 4.0 or says nothing at all, which would either
    mislicense permissive text or let restricted text through.
    """
    for parent in [path.parent, *path.parents]:
        readme = parent / "README.md"
        if readme.exists():
            text = readme.read_text(encoding="utf-8", errors="ignore")
            m = re.search(r"#\s*Licen[cs]e(.{0,700})", text, re.I | re.S)
            if m:
                return " ".join(m.group(1).split())[:400]
            # The repository README is explicit: unless a collection states
            # otherwise, it is CC BY-NC-SA 4.0. Assuming CC0 here would quietly
            # relicense community material, so the fallback is the restrictive
            # one and the run's licence ledger inherits it.
            return "CC BY-NC-SA 4.0 (repository default; no subcollection licence stated)"
    return "CC BY-NC-SA 4.0 (repository default; no subcollection licence stated)"


def _blocked_by_licence(licence: str) -> bool:
    low = licence.lower()
    if any(marker in low for marker in _ND_MARKERS):
        return True
    if "noncommercial" in low or "non-commercial" in low or "nc-" in low:
        return not CFG.include_noncommercial
    return False


def _add_doc(text: str, **meta) -> None:
    text = unicodedata.normalize("NFC", str(text)).strip()
    if len(text) < 12:
        return
    DOCS.append({"text": text, **meta})


def _add_pair(cree: str, english: str, **meta) -> None:
    cree = unicodedata.normalize("NFC", str(cree)).strip()
    english = unicodedata.normalize("NFC", str(english)).strip()
    if len(cree) < 2 or len(english) < 2:
        return
    PAIRS.append({"cree": cree, "english": english, **meta})


# 1. EdTeKLA Cree Corpus
def load_edtekla() -> None:
    url = "https://codeload.github.com/EdTeKLA/IndigenousLanguages_Corpora/tar.gz/refs/heads/master"
    archive = download(url, RAW / "edtekla.tar.gz")
    root = RAW / "edtekla"
    if not (root / "IndigenousLanguages_Corpora-master").exists():
        with tarfile.open(archive) as tf:
            tf.extractall(root)
    base = root / "IndigenousLanguages_Corpora-master" / "PlainsCree"

    blocked_dirs = {"Facebook", "CreeLiteracyOrg", "Twitter", "Bible"}
    used = skipped_lic = skipped_bow = 0
    for path in sorted(base.rglob("*.txt")):
        rel = path.relative_to(base)
        if path.name.lower().startswith("readme"):
            continue
        if rel.parts[0] in blocked_dirs:
            skipped_lic += 1
            continue
        licence = _readme_licence(path)
        if _blocked_by_licence(licence):
            skipped_lic += 1
            continue
        if "_bow" in path.stem:
            # Bag-of-words files carry counts, not sentences. They are still
            # useful as a vocabulary check, never as training text.
            for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
                parts = line.rsplit(None, 1)
                if len(parts) == 2 and parts[0].strip() and parts[1].isdigit():
                    w = normalize_sro(parts[0])
                    if 2 <= len(w) <= 40 and re.fullmatch(r"[a-zâêîô'-]+", w):
                        LEXICON[w] = max(LEXICON.get(w, 0.0), float(parts[1]))
            skipped_bow += 1
            continue

        text = path.read_text(encoding="utf-8", errors="ignore")
        script = "Cans" if path.stem.endswith("_syllabics") else "Latn"
        if script == "Cans":
            text = to_sro(text)
        # Keep a mixed document: the sentence filter in the next cell removes
        # its English lines, while its Cree paragraphs are still worth having.
        if cree_word_ratio(text) < 0.25 and lang_guess(text) != "crk":
            continue
        doc_id = str(rel.with_suffix(""))
        _add_doc(text, lang="crk", script="Latn", source=f"edtekla/{doc_id}",
                 doc_id=doc_id, licence=licence, dialect="plains", kind="prose")
        used += 1

        # A Cree file and its English sibling, line-aligned.
        en_path = path.with_name(path.name.replace("_cr.txt", "_en.txt").replace("_syllabics.txt", "_en.txt"))
        if en_path.exists():
            cr_lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
            en_lines = en_path.read_text(encoding="utf-8", errors="ignore").splitlines()
            if len(cr_lines) == len(en_lines):
                for i, (cr, en) in enumerate(zip(cr_lines, en_lines)):
                    cr = to_sro(cr) if script == "Cans" else cr
                    if len(cr.split()) >= 2 and len(en.split()) >= 2 and lang_guess(en) == "eng" and cree_word_ratio(cr) > 0.6:
                        _add_pair(cr, en, source="edtekla", doc_id=f"{doc_id}#{i}",
                                  licence=licence, dialect="plains", origin="human")

    ledger("EdTeKLA Cree Corpus", "https://github.com/EdTeKLA/IndigenousLanguages_Corpora",
           "CC BY-NC-SA 4.0 repo default; per-text READMEs; NC-ND subcollections skipped",
           "plains (y-dialect)", "Latn + Cans",
           f"{used} texts used; {skipped_lic} skipped on licence; {skipped_bow} BoW files used for vocabulary only")
    STATUS["edtekla"] = f"ok: {used} texts"


# 2. Plains Cree figurative corpus (Bloomfield 1934 + audits)
def load_figurative() -> None:
    from datasets import load_dataset

    n = 0
    for config in ("gold", "silver"):
        ds = load_dataset("KonradBRG/plains-cree-figurative", config, split=config)
        if SMOKE:
            ds = ds.select(range(min(120, len(ds))))
        for row in ds:
            cree, english = row.get("text_cree", ""), row.get("text_en", "")
            if not cree or not english:
                continue
            src = row.get("source_file") or f"figurative-{config}"
            _add_pair(cree, english, source="figurative", doc_id=f"{src}/{row.get('sentence_id', n)}",
                      licence="CC BY 4.0", dialect="plains", origin="human" if config == "gold" else "silver")
            _add_doc(cree, lang="crk", script="Latn", source=f"figurative/{src}",
                     doc_id=f"{src}/{row.get('sentence_id', n)}", licence="CC BY 4.0",
                     dialect="plains", kind="prose")
            n += 1
    ledger("Plains Cree Figurative Language corpus", "https://huggingface.co/datasets/KonradBRG/plains-cree-figurative",
           "CC BY 4.0", "plains", "Latn", f"{n} sentence pairs and sentences")
    STATUS["figurative"] = f"ok: {n} pairs"


# 3. Wikipedia in Cree
def load_wikipedia() -> None:
    from datasets import load_dataset

    ds = load_dataset("wikimedia/wikipedia", "20231101.cr", split="train")
    n = 0
    for row in ds:
        text = row.get("text", "")
        if len(text) < 80 or cree_word_ratio(text) < 0.2:
            continue
        _add_doc(text, lang="crk", script="Latn", source="wikipedia",
                 doc_id=str(row.get("id", n)), licence="CC BY-SA 4.0",
                 dialect="plains", kind="encyclopedic")
        n += 1
        if SMOKE and n >= 20:
            break
    ledger("Cree Wikipedia", "https://huggingface.co/datasets/wikimedia/wikipedia",
           "CC BY-SA 4.0", "mixed Cree", "Latn", f"{n} articles")
    STATUS["wikipedia"] = f"ok: {n} articles"


# 4. FineWeb-2 Cree slices
def load_fineweb() -> None:
    from datasets import load_dataset

    n = 0
    for config, script in (("crk_Latn", "Latn"), ("crk_Cans", "Cans")):
        try:
            ds = load_dataset("HuggingFaceFW/fineweb-2", config, split="train", streaming=True)
        except Exception as exc:
            STATUS[f"fineweb:{config}"] = f"unavailable: {exc}"
            continue
        for row in ds:
            text = row.get("text", "")
            if len(text) < 120:
                continue
            if script == "Cans":
                text = to_sro(text)
            if cree_word_ratio(text) < 0.35:
                continue
            _add_doc(text, lang="crk", script=script, source=f"fineweb2/{config}",
                     doc_id=str(row.get("id", n)), licence="ODC-BY 1.0",
                     dialect="plains", kind="web")
            n += 1
            if SMOKE and n >= 30:
                break
    ledger("FineWeb-2 Cree slices", "https://huggingface.co/datasets/HuggingFaceFW/fineweb-2",
           "ODC-BY 1.0", "plains", "Latn + Cans", f"{n} documents after language filtering")
    STATUS["fineweb2"] = f"ok: {n} docs"


# 4a. FinePDFs Cree slices
#
# FinePDFs extracts text from Common Crawl PDFs. Its Cree slices are small but
# they are the densest modern prose available: 430 Latin-script documents and
# 330 syllabic documents, roughly 1.2 million words, mostly published Cree
# rather than forum fragments. The text layer is already extracted, so this is
# clean copy, not OCR.
def load_finepdfs() -> None:
    from datasets import load_dataset

    n = 0
    for config, script in (("crk_Latn", "Latn"), ("crk_Cans", "Cans")):
        try:
            ds = load_dataset("HuggingFaceFW/finepdfs", config, split="train", streaming=True)
        except Exception as exc:
            STATUS[f"finepdfs:{config}"] = f"unavailable: {exc}"
            continue
        for row in ds:
            text = row.get("text", "")
            if len(text) < 150:
                continue
            if script == "Cans":
                text = to_sro(text)
            if cree_word_ratio(text) < 0.30:
                continue
            _add_doc(text, lang="crk", script=script, source=f"finepdfs/{config}",
                     doc_id=str(row.get("id", n)), licence="ODC-BY 1.0 (PDF text layer)",
                     dialect="plains", kind="pdf")
            n += 1
            if SMOKE and n >= 20:
                break
    ledger("FinePDFs Cree slices", "https://huggingface.co/datasets/HuggingFaceFW/finepdfs",
           "ODC-BY 1.0", "plains", "Latn + Cans", f"{n} documents from PDF text layers")
    STATUS["finepdfs"] = f"ok: {n} docs"


# 4b. FineTranslations Cree documents
#
# FineTranslations pairs each FineWeb-2 document with an English translation.
# The Cree side is human web text; the English side is machine translation, so
# every pair is tagged `origin=mined` and `translation=machine` and the
# benchmark reports human-origin and mined results separately. Chunk alignment
# is used, never a whole article, because a 40-line page is not one sentence.
def load_finetranslations() -> None:
    from datasets import load_dataset

    n_docs = n_pairs = 0
    for config, script in (("crk_Latn", "Latn"), ("crk_Cans", "Cans")):
        try:
            ds = load_dataset("HuggingFaceFW/finetranslations", config, split="train", streaming=True)
        except Exception as exc:
            STATUS[f"finetranslations:{config}"] = f"unavailable: {exc}"
            continue
        for row in ds:
            cree = row.get("og_full_text", "") or ""
            english = row.get("translated_text", "") or ""
            if len(cree) < 80 or not english:
                continue
            if script == "Cans":
                cree = to_sro(cree)
            if cree_word_ratio(cree) < 0.30:
                continue
            doc_id = str(row.get("id", f"{config}-{n_docs}"))
            # The Cree document itself is already loaded by `load_fineweb`;
            # adding it again here would double-count the same text in the
            # ledger and make the corpus look larger than it is. Only the
            # aligned chunks are new.
            n_docs += 1
            chunks_cr = row.get("og_chunks") or []
            chunks_en = row.get("translated_chunks") or []
            for i, (cr, en) in enumerate(zip(chunks_cr, chunks_en)):
                cr = to_sro(cr) if script == "Cans" else cr
                if not (3 <= len(cr.split()) <= 80 and 3 <= len(en.split()) <= 120):
                    continue
                if cree_word_ratio(cr) < 0.40 or lang_guess(en) != "eng":
                    continue
                _add_pair(cr, en, source="finetranslations", doc_id=f"{doc_id}#{i}",
                          licence="ODC-BY 1.0 (machine English)", dialect="plains",
                          origin="mined", translation="machine")
                n_pairs += 1
            if SMOKE and n_docs >= 20:
                break
    ledger("FineTranslations Cree", "https://huggingface.co/datasets/HuggingFaceFW/finetranslations",
           "ODC-BY 1.0; English side is machine translation", "plains", "Latn + Cans",
           f"{n_docs} documents, {n_pairs} aligned chunks")
    STATUS["finetranslations"] = f"ok: {n_pairs} aligned chunks from {n_docs} documents"


# 4c. GlotCC Cree slices
def load_glotcc() -> None:
    from datasets import load_dataset

    n = 0
    for config, script in (("crk-Latn", "Latn"), ("crk-Cans", "Cans")):
        try:
            ds = load_dataset("cis-lmu/GlotCC-V1", config, split="train", streaming=True)
        except Exception as exc:
            STATUS[f"glotcc:{config}"] = f"unavailable: {exc}"
            continue
        for row in ds:
            text = row.get("content", "")
            if len(text) < 80:
                continue
            if script == "Cans":
                text = to_sro(text)
            if cree_word_ratio(text) < 0.30:
                continue
            _add_doc(text, lang="crk", script=script, source=f"glotcc/{config}",
                     doc_id=str(row.get("id", n)), licence="ODC-BY 1.0",
                     dialect="plains", kind="web")
            n += 1
    ledger("GlotCC Cree slices", "https://huggingface.co/datasets/cis-lmu/GlotCC-V1",
           "ODC-BY 1.0", "plains", "Latn + Cans", f"{n} documents")
    STATUS["glotcc"] = f"ok: {n} docs"


# 4d. Tatoeba Cree sentences
def load_tatoeba() -> None:
    url = "https://downloads.tatoeba.org/exports/per_language/crk/crk_sentences.tsv.bz2"
    path = download(url, RAW / "tatoeba_crk.tsv.bz2")
    n = 0
    with bz2.open(path, "rt", encoding="utf-8") as fh:
        for line in fh:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            sent_id, _lang, text = parts[0], parts[1], "\t".join(parts[2:])
            if len(text) < 12:
                continue
            _add_doc(text, lang="crk", script="Cans" if looks_like_syllabics(text) else "Latn",
                     source="tatoeba", doc_id=sent_id, licence="CC BY 2.0 FR",
                     dialect="plains", kind="sentence")
            n += 1
    ledger("Tatoeba Cree sentences", "https://tatoeba.org/", "CC BY 2.0 FR",
           "plains", "Latn + Cans", f"{n} sentences")
    STATUS["tatoeba"] = f"ok: {n} sentences"


# 4e. GlotStoryBook Cree rows
def load_storybook() -> None:
    from datasets import load_dataset

    n = 0
    try:
        ds = load_dataset("cis-lmu/GlotStoryBook", "default", split="train", streaming=True)
    except Exception as exc:
        STATUS["storybook"] = f"unavailable: {exc}"
        return
    for row in ds:
        code = str(row.get("ISO639-3", "")).lower()
        if code not in {"crk", "cre"}:
            continue
        text = row.get("Text", "")
        if len(text) < 40:
            continue
        _add_doc(text, lang="crk", script="Cans" if looks_like_syllabics(text) else "Latn",
                 source="storybook", doc_id=f"{row.get('Text Number')}-{n}",
                 licence=str(row.get("License", "CC BY-NC-SA")), dialect="plains", kind="story")
        n += 1
    ledger("GlotStoryBook Cree", "https://huggingface.co/datasets/cis-lmu/GlotStoryBook",
           "CC BY-NC-SA (per row)", "plains", "Latn + Cans", f"{n} story paragraphs")
    STATUS["storybook"] = f"ok: {n} paragraphs"


# 4f. Public-domain historical Cree books
#
# The modern web has almost no Plains Cree prose: FineWeb-2 carries 192 Cree
# documents in total, and Cree Wikipedia has 187 articles. The nineteenth
# century printed far more. These books are public domain, their Cree is real,
# and they are used for language exposure only, never as modern translation
# targets; every one carries the `historical` dialect tag.
_HIST_TRANS = str.maketrans({
    "û": "ô", "Û": "Ô", "ù": "ô", "Ù": "Ô",
    "ā": "â", "ē": "ê", "ī": "î", "ō": "ô",
    "à": "â", "è": "ê", "ì": "î", "ò": "ô", "é": "ê", "ó": "ô",
})
_WATKINS_EN_STOP = {
    "or", "and", "the", "a", "an", "to", "of", "in", "with", "by", "for",
    "is", "are", "was", "were", "this", "that", "when", "past", "present",
    "future", "more", "less", "little", "small", "large", "see", "also",
    "same", "as", "it", "he", "she", "they", "his", "her", "their", "not",
}


def _clean_watkins_form(form: str) -> str:
    """One modern-spelling Cree headword, or an empty string.

    Watkins' dictionary is OCR'd and annotated: entries arrive as
    ``Jspimik. (more) awusim4`` or with 19th-century vowel marks. This keeps
    only a short, unmixed, normalised form, because a dirty gloss teaches the
    model a spelling nobody uses.
    """
    form = unicodedata.normalize("NFC", str(form))
    form = re.sub(r"\([^)]*\)", " ", form)
    form = form.translate(_HIST_TRANS)
    form = form.replace("‘", "'").replace("’", "'")
    form = re.sub(r"[^A-Za-zâêîô'\- ]+", " ", form)
    form = re.sub(r"\s+", " ", form).strip().lower()
    if not form:
        return ""
    words = form.split()
    if not 1 <= len(words) <= 4:
        return ""
    if any(w.strip(".,") in _WATKINS_EN_STOP for w in words):
        return ""
    if cree_word_ratio(form) < 0.8 or lang_guess(form) != "crk":
        return ""
    return form


def _clean_watkins_english(gloss: str) -> str:
    gloss = re.sub(r"\([^)]*\)", " ", str(gloss))
    gloss = re.sub(r"[^A-Za-z'\- ]+", " ", gloss)
    gloss = re.sub(r"\s+", " ", gloss).strip().lower()
    if not gloss or len(gloss.split()) > 6:
        return ""
    return gloss


# Books known to be worth the download even if the archive search ranks them
# low. The rest of the list is discovered, not hand-picked.
_ARCHIVE_SEED = {
    "P005757": "Plains Cree texts (Bloomfield, 1934)",
    "P005409": "Sacred stories of the Sweet Grass Cree (Bloomfield, 1930)",
    "cihm_07195": "Gospel of Matthew, 1877",
    "cihm_01021": "Gospel of Mark, 1876",
    "P001735": "Cree hymn book, 1888",
    "P002107": "Cree primer, 1893",
    "cihm_30552": "Primer and language lessons, 1890",
    "cihm_30323": "Indian child's book, 1890",
    "P004640": "Cree hymn book, 1920",
    "creehymnbook00huntgoog": "Cree hymn book, 1888",
    "grammarofcreelan00hord_0": "Grammar of the Cree language, 1881",
    "P001885": "Primer and language lessons in English and Cree, 1890",
    "P002037": "Indian child's book",
    "P002871": "Psalms in Plains Cree, 1905",
    "cihm_05447": "Psalter in Cree, 1875",
    "P000574": "Cree hymn book, 1870",
    "rosettaproject_crk_vertxt-2": "Plains Cree Texts, Bloomfield 1934",
    "rosettaproject_crk_book-1": "Plains Cree Texts, Bloomfield 1934 (book scan)",
    "P000209": "Cree syllabic hymn book, 1841",
}
# Moosonee, James Bay and East Cree material is a different dialect. It is
# excluded rather than blended, because a y-dialect model that quietly learns
# n-dialect morphology is worse than one that says which data it used.
_ARCHIVE_BLOCK = (
    "moosonee", "moose cree", "rupert's land", "rupert ", "james bay",
    "east main", "eastmain", "albany", "chisasibi", "mistissini", "waskaganish",
    "wemindji", "nemaska", "ouje", "whapmagoostui", "fort george", "east cree",
    # French-language devotional books and English grammars are about Cree
    # rather than in it; their OCR is mostly the surrounding language.
    "français", "francais", "langue crise", "prières", "prieres", "cantiques",
    "catéchisme", "catechisme", "dictionnaire", "grammaire", "grammar of the cree",
)
_ARCHIVE_EN = {
    "the", "and", "of", "to", "in", "is", "was", "were", "that", "this", "for",
    "with", "he", "she", "they", "his", "her", "their", "you", "not", "but",
    "from", "have", "has", "had", "will", "would", "there", "what", "when",
    "who", "how", "all", "one", "two", "on", "at", "by", "as", "it", "be",
}


def _english_share(text: str) -> float:
    words = re.findall(r"[a-z']+", text.lower())
    return sum(w in _ARCHIVE_EN for w in words) / max(1, len(words))


def _lex_hit(text: str) -> float:
    """Fraction of words the GiellaLT lexicon recognises.

    This is the check that catches syllabic OCR: a scan of a syllabic book
    produces page after page of Latin letters that no Cree word list contains.
    English pages fail it too, for the same reason.
    """
    words = [w for w in re.findall(r"[a-zâêîô'-]+", normalize_sro(text)) if len(w) > 1]
    if not words:
        return 0.0
    hits = 0
    for word in words:
        parts = [p for p in word.replace("-", " ").split() if p]
        if any(p in LEXICON for p in parts):
            hits += 1
    return hits / len(words)


def _archive_candidates() -> list[tuple[str, str, int]]:
    params = {
        "q": "language:(cree) AND mediatype:texts",
        "fl[]": ["identifier", "title", "year", "downloads", "description"],
        "rows": "500",
        "output": "json",
    }
    url = "https://archive.org/advancedsearch.php?" + urllib.parse.urlencode(params, doseq=True)
    req = urllib.request.Request(url, headers={"User-Agent": "archer-llm/1.0"})
    payload = json.loads(urllib.request.urlopen(req, timeout=90).read())
    found: dict[str, tuple[str, str, int]] = {}
    for row in payload["response"]["docs"]:
        ident = str(row.get("identifier", ""))
        if not ident:
            continue
        title = str(row.get("title", "") or "")
        description = str(row.get("description", "") or "")
        low = f"{title} {description}".lower()
        if any(marker in low for marker in _ARCHIVE_BLOCK):
            continue
        match = re.search(r"(1[5-9]\d\d|20[0-2]\d)", str(row.get("year", "")))
        year = int(match.group(1)) if match else 0
        if year and year > 1934:
            continue
        found[ident] = (title, description, int(row.get("downloads", 0) or 0))
    ordered = sorted(found.items(), key=lambda kv: (kv[0] not in _ARCHIVE_SEED, -kv[1][2]))
    return [(ident, title, downloads) for ident, (title, _d, downloads) in ordered]


def load_archive() -> None:
    try:
        candidates = _archive_candidates()
    except Exception as exc:
        STATUS["archive_search"] = f"unavailable: {exc}"
        candidates = [(ident, title, 0) for ident, title in _ARCHIVE_SEED.items()]
    log(f"archive: {len(candidates)} candidate books after dialect filter")
    n = rejected = attempts = 0
    # Use the whole public-domain shelf, not an arbitrary first sixty-five. The
    # per-book OCR and language filters still reject bad scans; this only changes
    # how many real books are examined.
    budget = 6 if SMOKE else 250
    max_attempts = 12 if SMOKE else 450
    for ident, label, _downloads in candidates:
        if n >= budget or attempts >= max_attempts:
            break
        attempts += 1
        try:
            meta = json.loads(urllib.request.urlopen(
                f"https://archive.org/metadata/{ident}", timeout=60
            ).read())
            names = [f["name"] for f in meta.get("files", []) if f["name"].endswith("_djvu.txt")]
            if not names:
                rejected += 1
                continue
            url = f"https://archive.org/download/{ident}/{urllib.parse.quote(names[0])}"
            path = download(url, RAW / f"archive_{ident}.txt", timeout=180)
            text = path.read_text(encoding="utf-8", errors="ignore").translate(_HIST_TRANS)
            if len(text.split()) < 1500:
                rejected += 1
                continue
            # Bilingual books (Bloomfield faces Cree with English) and books
            # with a long English introduction cannot pass a whole-book test.
            # Judge each 700-word window instead, then let the sentence filter
            # in the next cell remove the English half of a mixed window.
            words = text.split()
            keep: list[tuple[int, str]] = []
            for i in range(0, len(words), 700):
                chunk = " ".join(words[i:i + 700])
                if len(chunk.split()) < 150:
                    continue
                if _english_share(chunk) > 0.5 or cree_word_ratio(chunk) < 0.5:
                    continue
                if _lex_hit(chunk) < 0.2:
                    continue
                keep.append((i // 700, chunk))
            # A book that yields one plausible window is usually a bad scan
            # with a stray Cree quotation. Require at least three.
            if len(keep) >= 3:
                for index, chunk in keep:
                    _add_doc(chunk, lang="crk", script="Latn", source=f"archive/{ident}",
                             doc_id=f"{ident}-{index:03d}",
                             licence="public domain (published before 1935; traditional texts)",
                             dialect="historical", kind="historical")
                n += 1
            else:
                rejected += 1
                STATUS[f"archive:{ident}"] = "rejected: OCR not usable (syllabic scan or English)"
        except Exception as exc:
            STATUS[f"archive:{ident}"] = f"unavailable: {exc}"
            rejected += 1
    ledger("Internet Archive historical Cree", "https://archive.org/",
           "public domain (published before 1935; traditional texts)", "historical", "Latn",
           f"{n} books used, {rejected} rejected (syllabic OCR, English, or too short)")
    STATUS["archive"] = f"ok: {n} books, {rejected} rejected"


# 4g. Little Cree Books (Plains Cree only)
#
# This is the open-licensed Little Cree Books collection: four Plains Cree
# stories in SRO and syllabics, each with an English sibling. It is small, but
# it is real community text under CC BY-NC-SA, and it is exactly the kind of
# material the model should see. Swampy Cree stories 0005 and 0006 are excluded
# by their own metadata; mixing an n-dialect into a y-dialect model is not a
# size win.
def load_lcb() -> None:
    licence = "CC BY-NC-SA 3.0 (Little Cree Books)"
    if _blocked_by_licence(licence):
        ledger("Little Cree Books", "https://github.com/global-asp/lcb-source", licence,
               "plains", "Latn + Cans", "skipped because the build is commercial", allowed=False)
        STATUS["lcb"] = "skipped: noncommercial licence and commercial build"
        return
    url = "https://codeload.github.com/global-asp/lcb-source/tar.gz/refs/heads/master"
    archive = download(url, RAW / "lcb-source.tar.gz")
    root = RAW / "lcb-source"
    if not (root / "lcb-source-master").exists():
        with tarfile.open(archive) as tf:
            tf.extractall(root)
    base = root / "lcb-source-master"

    def paragraphs(path: Path) -> list[str]:
        text = path.read_text(encoding="utf-8", errors="ignore")
        out: list[str] = []
        for chunk in re.split(r"(?m)^##\s*$", text):
            lines = []
            for line in chunk.splitlines():
                line = line.strip()
                if not line or line.startswith(("#", "*", "[", "!")):
                    continue
                lines.append(line)
            para = " ".join(lines).strip()
            if len(para) >= 12:
                out.append(para)
        return out

    n_docs = n_pairs = 0
    for path in sorted(base.glob("*/[0-9]*_sro.md")):
        # The README files do not always spell out the ISO code. Stories 0005
        # and 0006 are Swampy Cree; every other story in this repository is
        # Plains Cree SRO. Exclude by the explicit Swampy folder names rather
        # than silently dropping a Plains story whose README omits a language.
        if path.parent.name in {"0005_grandmother-fox", "0006_little-fox"}:
            continue
        story = path.parent.name
        cree_paras = paragraphs(path)
        en_path = path.with_name(path.name.replace("_sro.md", "_en.md"))
        en_paras = paragraphs(en_path) if en_path.exists() else []
        for i, cree in enumerate(cree_paras):
            _add_doc(cree, lang="crk", script="Latn", source=f"lcb/{story}",
                     doc_id=f"{story}#{i}", licence=licence, dialect="plains", kind="story")
            n_docs += 1
            if i < len(en_paras) and len(cree.split()) >= 2 and len(en_paras[i].split()) >= 2:
                _add_pair(cree, en_paras[i], source="lcb", doc_id=f"{story}#{i}",
                          licence=licence, dialect="plains", origin="human")
                n_pairs += 1
    ledger("Little Cree Books", "https://github.com/global-asp/lcb-source", licence,
           "plains (Y-dialect)", "Latn + Cans",
           f"{n_docs} SRO paragraphs, {n_pairs} English pairs; Swampy Cree excluded")
    STATUS["lcb"] = f"ok: {n_docs} paragraphs, {n_pairs} pairs"


# 4h. Plains Cree phrase recordings
def load_plainscree() -> None:
    from datasets import load_dataset

    n = 0
    try:
        ds = load_dataset("verayang/plainscree", split="train", streaming=True)
        # The audio column needs torchcodec; the transcriptions do not.
        ds = ds.select_columns(["audio_id", "cree_transcription", "english_transcription"])
    except Exception as exc:
        STATUS["plainscree"] = f"unavailable: {exc}"
        return
    for row in ds:
        cree = row.get("cree_transcription", "") or ""
        english = row.get("english_transcription", "") or ""
        if len(cree) < 4 or not english:
            continue
        _add_pair(cree, english, source="plainscree", doc_id=f"audio-{row.get('audio_id', n)}",
                  licence="research dataset (audio-derived transcriptions)", dialect="plains",
                  origin="human", translation="human")
        n += 1
    ledger("Plains Cree phrase recordings", "https://huggingface.co/datasets/verayang/plainscree",
           "research dataset; transcriptions only, audio not redistributed", "plains", "Latn",
           f"{n} short phrases")
    STATUS["plainscree"] = f"ok: {n} phrases"


# 5. This project's own data and notes
def load_project() -> None:
    # 5a. The aligned sentences already curated in the repository. The file also
    #     contains dictionary frequency rows; those are filtered below.
    url = f"{PROJECT_RAW}/data/processed/parallel_cr_en.jsonl"
    path = download(url, RAW / "project_parallel.jsonl")
    n = 0
    skipped_rows = 0
    for row in read_jsonl(path):
        cree, english = row.get("cree", ""), row.get("english", "")
        # Most of this file is a dictionary alignment ("a 218" <-> "abiathar
        # 1"), not sentence pairs. Those rows teach the model to emit stray
        # numbers and short glosses, and they poison the benchmark because the
        # test split is drawn from the same pool. A translation pair has no
        # digits and at least two words on each side.
        if re.search(r"\d", f"{cree} {english}"):
            skipped_rows += 1
            continue
        if len(cree.split()) < 2 or len(english.split()) < 2:
            skipped_rows += 1
            continue
        if cree_word_ratio(cree) > 0.55 and lang_guess(english) == "eng":
            _add_pair(cree, english, source="project", doc_id=f"parallel-{n}",
                      licence="project-owned", dialect="plains", origin="human")
            n += 1
    parallel_pairs = n
    STATUS["project_parallel"] = f"ok: {n} pairs ({skipped_rows} numeric/short rows skipped)"

    # 5b. Archer's song notes: the grounding text for the assistant.
    notes = []
    try:
        listing = json.loads(urllib.request.urlopen(
            f"https://api.github.com/repos/{PROJECT_REPO}/contents/song_notes", timeout=30
        ).read())
        for item in listing:
            if item["name"].endswith((".md", ".txt")) and item["name"].lower() != "readme.md":
                body = urllib.request.urlopen(item["download_url"], timeout=30).read().decode("utf-8", "ignore")
                notes.append({"name": item["name"], "text": body})
    except Exception as exc:
        STATUS["song_notes"] = f"unavailable: {exc}"
    for note in notes:
        _add_doc(note["text"], lang="eng", script="Latn", source=f"song_notes/{note['name']}",
                 doc_id=note["name"], licence="project-owned", dialect="plains", kind="song-notes")
    STATUS["song_notes"] = f"ok: {len(notes)} notes"

    # 5c. Watkins 1865 dictionary entries, kept under their own dialect tag.
    watkins_pairs = 0
    try:
        wpath = download(f"{PROJECT_RAW}/data/processed/watkins_entries.jsonl", RAW / "watkins.jsonl")
        n = 0
        skipped = 0
        for row in read_jsonl(wpath):
            english = _clean_watkins_english(row.get("english", ""))
            if not english:
                continue
            for form in row.get("cree_forms", []):
                cree = _clean_watkins_form(form)
                if not cree:
                    skipped += 1
                    continue
                _add_pair(cree, english, source="watkins1865", doc_id=f"watkins-{n}",
                          licence="public domain (1865)", dialect="watkins1865",
                          origin="historical")
                n += 1
        watkins_pairs = n
        STATUS["watkins"] = f"ok: {n} historical pairs ({skipped} OCR/annotation forms rejected)"
    except Exception as exc:
        STATUS["watkins"] = f"unavailable: {exc}"

    # The completeness gate asks for one status per loader, but this loader
    # reports its three parts under their own keys. Without a summary under
    # "project" the source reads as "not run" even when every part succeeded,
    # and the gate stops a corpus that is actually complete.
    _total = parallel_pairs + len(notes) + watkins_pairs
    STATUS["project"] = (f"ok: {_total} rows ({parallel_pairs} parallel pairs, "
                         f"{len(notes)} song notes, {watkins_pairs} Watkins pairs)")

    ledger("Archer project data + song notes", f"https://github.com/{PROJECT_REPO}",
           "project-owned", "plains", "Latn", f"{len(notes)} song notes, parallel pairs, Watkins 1865")


# 6. GiellaLT lexicon (validity only, not running text)
def load_giellalt() -> None:
    url = "https://codeload.github.com/giellalt/lang-crk/tar.gz/refs/heads/main"
    archive = download(url, RAW / "lang-crk.tar.gz")
    root = RAW / "lang-crk"
    if not (root / "lang-crk-main").exists():
        with tarfile.open(archive) as tf:
            tf.extractall(root)
    base = root / "lang-crk-main"

    n = 0
    weighted = base / "tools" / "spellcheckers" / "prefix_weighting.lexc"
    if weighted.exists():
        for line in weighted.read_text(encoding="utf-8", errors="ignore").splitlines():
            m = re.match(r'([^\s#]+)\s*[#$]?\s*#?\s*"weight:\s*([0-9.]+)"', line.strip())
            if not m:
                continue
            word, weight = normalize_sro(m.group(1).split("+")[0].rstrip("$")), float(m.group(2))
            if 2 <= len(word) <= 40 and re.fullmatch(r"[a-zâêîô'-]+", word):
                LEXICON[word] = max(LEXICON.get(word, 0.0), weight)
                n += 1
    for lexc in (base / "src" / "fst" / "morphology" / "stems").glob("*.lexc"):
        for line in lexc.read_text(encoding="utf-8", errors="ignore").splitlines():
            line = line.strip()
            if not line or line.startswith("!") or line.startswith("LEXICON"):
                continue
            surface = line.split()[0].split(":")[0].split("+")[0].rstrip("$")
            word = normalize_sro(surface)
            if 2 <= len(word) <= 40 and re.fullmatch(r"[a-zâêîô'-]+", word):
                LEXICON.setdefault(word, 0.1)
    public_texts = base / "devtools" / "testdata" / "crk_public_texts.txt"
    if public_texts.exists():
        _add_doc(public_texts.read_text(encoding="utf-8", errors="ignore"), lang="crk",
                 script="Latn", source="giellalt/public-texts", doc_id="crk_public_texts",
                 licence="AGPL-3.0 repository; public texts", dialect="plains", kind="prose")
        n += 1

    ledger("GiellaLT lang-crk lexicon", "https://github.com/giellalt/lang-crk",
           "AGPL-3.0 source; word list used for validation and tokenizer weighting",
           "plains", "Latn", f"{len(LEXICON)} word forms; {n} public text files")
    STATUS["giellalt"] = f"ok: {len(LEXICON)} forms"


# 7. Anything the user drops in
def load_uploads() -> None:
    dirs = [Path(os.environ.get("CREE_UPLOAD_DIR", ROOT / "uploads"))]
    if IN_COLAB:
        dirs.append(Path("/content/drive/MyDrive/cree_corpus"))
    n = 0
    for d in dirs:
        if not d.exists():
            continue
        for path in sorted(d.rglob("*")):
            if path.suffix.lower() not in {".txt", ".md"}:
                continue
            text = path.read_text(encoding="utf-8", errors="ignore")
            for para in re.split(r"\n\s*\n", text):
                if len(para.strip()) > 20:
                    _add_doc(para, lang=lang_guess(para), script="Cans" if looks_like_syllabics(para) else "Latn",
                             source="community", doc_id=f"{path.name}#{n}", licence="user-provided",
                             dialect="unspecified", kind="community")
                    n += 1
    if n:
        ledger("Community uploads", str(dirs[0]), "user-provided (confirm before publishing)",
               "unspecified", "Latn + Cans", f"{n} paragraphs")
        STATUS["uploads"] = f"ok: {n} paragraphs"


# run every loader independently
# On Drive this whole section is skipped once it has produced files. It is the
# slowest part of the notebook (network, archive index, PDF extraction) and none
# of it changes between sessions, so repeating it after a disconnect only burns
# the time the checkpointing is meant to save.
_CORPUS_KEY = fingerprint(corpus="v3", noncommercial=CFG.include_noncommercial,
                         min_tokens=CFG.min_corpus_tokens)
_READY = (RAW / "docs.jsonl", RAW / "pairs.jsonl", RAW / "lexicon.json")

if all(f.exists() for f in _READY) and stage_done("corpus", _CORPUS_KEY, requires=RAW / "ledger.json"):
    DOCS = read_jsonl(RAW / "docs.jsonl")
    PAIRS = read_jsonl(RAW / "pairs.jsonl")
    LEXICON = jload(RAW / "lexicon.json")
    LEDGER = jload(RAW / "ledger.json")
    STATUS = jload(RAW / "source_status.json")
    log(f"corpus loaded from Drive: {len(DOCS)} documents, {len(PAIRS)} pairs, {len(LEXICON)} lexicon forms")
else:
    for name, fn in [
        ("edtekla", load_edtekla),
        ("figurative", load_figurative),
        ("project", load_project),
        ("wikipedia", load_wikipedia),
        ("fineweb2", load_fineweb),
        ("finepdfs", load_finepdfs),
        ("finetranslations", load_finetranslations),
        ("glotcc", load_glotcc),
        ("tatoeba", load_tatoeba),
        ("storybook", load_storybook),
        ("lcb", load_lcb),
        ("plainscree", load_plainscree),
        # GiellaLT runs before the archive because its lexicon is what separates a
        # real 19th-century Cree page from a syllabic scan OCR turned into noise.
        ("giellalt", load_giellalt),
        ("archive", load_archive),
        ("uploads", load_uploads),
    ]:
        t0 = time.time()
        try:
            fn()
            log(f"{name}: {STATUS.get(name, 'ok')}  ({time.time() - t0:.1f}s)")
        except Exception as exc:
            STATUS[name] = f"failed: {type(exc).__name__}: {exc}"
            log(f"{name}: FAILED: {exc}")

    # A partial corpus must not be trained on quietly. Every source in the table
    # is expected to contribute, and a download that failed, returned nothing, or
    # produced zero documents is a corpus that looks finished and is not: the
    # model then learns less Cree than the report claims. The run stops here
    # unless the caller has explicitly accepted a partial corpus.
    _expected = {
        "edtekla", "figurative", "project", "wikipedia", "fineweb2", "finepdfs",
        "finetranslations", "glotcc", "tatoeba", "storybook", "lcb", "plainscree",
        "giellalt", "archive",
    }

    def _contributed(name: str) -> bool:
        note = str(STATUS.get(name, ""))
        if not note.startswith("ok"):
            return False
        # "ok: 0 docs" is a source that ran and contributed nothing.
        return not re.search(r":\s*0\s", note)

    _missing = sorted(n for n in _expected if not _contributed(n))
    if _missing and not getattr(CFG, "allow_missing_sources", False):
        raise RuntimeError(
            "corpus incomplete, these sources did not contribute: "
            + ", ".join(f"{n} [{STATUS.get(n, 'not run')}]" for n in _missing)
            + ". Fix the download, or set allow_missing_sources on the profile if you "
              "mean to train on less. Training on a partial corpus is how a model "
              "ends up looking trained while it cannot write the language."
        )
    log(f"corpus complete: {len(_expected) - len(_missing)}/{len(_expected)} sources contributed")

    # Source completeness is not corpus size. A run can download every
    # collection, clean it, and still have only a small slice of the language.
    # The 14B profiles warn loudly when that happens. Training continues and
    # uses every token, because the user explicitly wants the model trained on
    # all available, verified data. The warning is also written to the run so a
    # report can never call a small corpus complete.
    _est_tokens = int((sum(len(d["text"].split()) for d in DOCS)
                       + sum(len(p["cree"].split()) + len(p["english"].split()) for p in PAIRS))
                      * 1.4)
    _corpus_size = {
        "estimated_tokens": _est_tokens,
        "minimum_for_profile": CFG.min_corpus_tokens,
        "documents": len(DOCS),
        "pairs": len(PAIRS),
        "ready": not (not SMOKE and CFG.min_corpus_tokens and _est_tokens < CFG.min_corpus_tokens),
    }
    jdump(RAW / "corpus_size.json", _corpus_size)
    if not _corpus_size["ready"]:
        STATUS["corpus_size"] = (f"warning: {_est_tokens:,} tokens below "
                                 f"{CFG.min_corpus_tokens:,}; training still uses all data")
        log("WARNING: corpus size is below the profile target; training continues with all data")
    else:
        STATUS["corpus_size"] = f"ok: {_est_tokens:,} estimated tokens"
    log(f"corpus size: about {_est_tokens:,} tokens (profile target {CFG.min_corpus_tokens:,})")

    write_jsonl(RAW / "docs.jsonl", DOCS)
    write_jsonl(RAW / "pairs.jsonl", PAIRS)
    jdump(RAW / "lexicon.json", LEXICON)
    jdump(RAW / "ledger.json", LEDGER)
    jdump(RAW / "source_status.json", STATUS)
    mark_stage("corpus", _CORPUS_KEY, documents=len(DOCS), pairs=len(PAIRS))


import pandas as pd

by_source: dict[str, list[dict]] = {}
for doc in DOCS:
    by_source.setdefault(doc["source"].split("/")[0], []).append(doc)
rows = []
for name, subset in sorted(by_source.items()):
    rows.append({
        "source": name,
        "documents": len(subset),
        "cree words": sum(len(d["text"].split()) for d in subset),
        "script": ", ".join(sorted({d["script"] for d in subset})),
    })
ledger_df = pd.DataFrame(rows)
print(f"\n{len(DOCS)} documents, {len(PAIRS)} aligned pairs, {len(LEXICON)} lexicon forms")
print(ledger_df.to_string(index=False))
print("\nsource status:", json.dumps(STATUS, indent=2, ensure_ascii=False))

## 2. Cleaning, deduplication and the split

Raw corpus text is not training data yet.

Three problems have to be solved before a single token is counted. The first is language: a file labelled Cree can contain English translations, headings, or a sentence that begins in Cree and ends in English. Each sentence is checked with function words and long-vowel marks, not just character ranges, because `the` is made entirely of letters Cree uses. The second is duplication: the same verse, refrain or dictionary example can appear in several collections, and a model that sees it in both train and test will report a score it did not earn. Exact hashes catch the obvious copies; a MinHash index catches the near-copies. The third is the split itself. Sentences are grouped by source document first, then divided, so a text cannot contribute half its paragraphs to training and half to the test set.

The split is 88% train, 7% development, 5% test. After dividing, a second pass checks every development and test sentence against the training set and moves any near-duplicate back into training. That is conservative: it is better to lose a test sentence than to publish a leaked score.

Bag-of-words files are detected by their shape and never used as prose. They still contribute word forms to the lexicon, which is used later to check whether generated Cree is a real word.

In [ ]:
# Cleaning, deduplication and splitting.
#
# Three failures destroy a low-resource model, and each is handled explicitly:
#   1. English hiding inside "Cree" text  -> language check per sentence.
#   2. The same sentence in train and test -> exact + near-duplicate removal,
#      then a document-disjoint split, then a second leakage check.
#   3. Bag-of-words files pretending to be prose -> rejected by shape.

DOCS = read_jsonl(RAW / "docs.jsonl")
PAIRS = read_jsonl(RAW / "pairs.jsonl")
LEXICON = jload(RAW / "lexicon.json")

_URL = re.compile(r"https?://\S+|www\.\S+")
_HTML = re.compile(r"<[^>]+>")
_WIKI = re.compile(r"\[\[(?:[^|\]]*\|)?([^\]]+)\]\]|\[[0-9]+\]|\{\{[^}]*\}\}")
_SPACE = re.compile(r"[ \t\u00a0]+")
_BOW_LINE = re.compile(r"^\s*[^\W\d_][^\s]*\s+\d+\s*$")


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))
    text = _HTML.sub(" ", text)
    text = _WIKI.sub(r"\1", text)
    text = _URL.sub(" ", text)
    text = text.replace("\u200b", "").replace("\ufeff", "")
    text = text.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    text = text.replace("—", "-").replace("–", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_sentences(text: str) -> list[str]:
    text = clean_text(text)
    if looks_like_syllabics(text):
        text = to_sro(text)
    text = text.replace("᙮", ".")
    # Paragraph breaks are sentence breaks even without punctuation.
    pieces = re.split(r"(?<=[.!?])\s+|\n+", text)
    out = []
    for piece in pieces:
        piece = re.sub(r"^[\s\"'\-–—•*]+|[\s\"'\-–—•*]+$", "", piece)
        if not piece:
            continue
        words = piece.split()
        if 4 <= len(words) <= 80:
            out.append(piece)
    return out


def bow_share(text: str) -> float:
    lines = [ln for ln in text.splitlines() if ln.strip()]
    if len(lines) < 4:
        return 0.0
    return sum(bool(_BOW_LINE.match(ln)) for ln in lines) / len(lines)


class NearDup:
    """MinHash + banded LSH, 32 permutations, 8 bands of 4 rows.

    A sentence is a duplicate when its estimated Jaccard similarity to an
    indexed sentence is at least ``threshold``. This catches the repeated
    refrains, quoted verses and copied dictionary examples that exact hashing
    misses, without pulling in a dependency.
    """

    _M = (1 << 61) - 1

    def __init__(self, threshold: float = 0.82, ngram: int = 5, perms: int = 32, bands: int = 8):
        self.threshold = threshold
        self.ngram = ngram
        self.perms = perms
        self.bands = bands
        self.rows = perms // bands
        rng = random.Random(4242)
        self.a = [rng.randrange(1, self._M - 1) for _ in range(perms)]
        self.b = [rng.randrange(0, self._M - 1) for _ in range(perms)]
        self.buckets: list[dict[tuple, list[int]]] = [{} for _ in range(bands)]
        self.sigs: list[tuple[int, ...]] = []
        self.texts: list[str] = []

    def _signature(self, text: str) -> tuple[int, ...]:
        words = re.findall(r"[^\W\d_]+", text.lower(), flags=re.UNICODE)
        if len(words) < self.ngram:
            grams = [" ".join(words)] if words else [text.lower()]
        else:
            grams = [" ".join(words[i:i + self.ngram]) for i in range(len(words) - self.ngram + 1)]
        hashes = {int.from_bytes(hashlib.blake2b(g.encode(), digest_size=8).digest(), "big") for g in grams}
        return tuple(min((h * a + b) % self._M for h in hashes) for a, b in zip(self.a, self.b))

    def find(self, text: str) -> int | None:
        sig = self._signature(text)
        for band in range(self.bands):
            key = tuple(sig[band * self.rows:(band + 1) * self.rows])
            for idx in self.buckets[band].get(key, ()):  # candidates only
                other = self.sigs[idx]
                same = sum(x == y for x, y in zip(sig, other))
                if same / self.perms >= self.threshold:
                    return idx
        return None

    def add(self, text: str) -> bool:
        if self.find(text) is not None:
            return False
        sig = self._signature(text)
        idx = len(self.sigs)
        self.sigs.append(sig)
        self.texts.append(text)
        for band in range(self.bands):
            key = tuple(sig[band * self.rows:(band + 1) * self.rows])
            self.buckets[band].setdefault(key, []).append(idx)
        return True


def norm_key(text: str) -> str:
    return re.sub(r"[^\wâêîô]+", "", normalize_sro(text))


# documents -> sentences
# The splits are the thing every later cell reads, so they are built once and
# then read back. Rebuilding them is not just slow: the near-duplicate filter is
# randomised by insertion order, so a second build could produce slightly
# different splits under the same names and quietly invalidate a benchmark that
# was already computed.
_CLEAN_KEY = fingerprint(
    docs=(RAW / "docs.jsonl").stat().st_size,
    pairs=(RAW / "pairs.jsonl").stat().st_size,
    version="clean-v2",
)
_SPLIT_FILES = [PROC / f"lm_{s}.jsonl" for s in ("train", "dev", "test")]
_SPLIT_FILES += [PROC / "parallel_train.jsonl", PROC / "parallel_test.jsonl"]

if all(f.exists() for f in _SPLIT_FILES) and stage_done("clean", _CLEAN_KEY, requires=PROC / "stats.json"):
    stats = jload(PROC / "stats.json")
    print(json.dumps(stats, indent=2, ensure_ascii=False))
else:
    # documents -> sentences
    NOTES = [d for d in DOCS if d["kind"] == "song-notes"]
    DOCS = [d for d in DOCS if d["kind"] != "song-notes"]

    sentences: list[dict] = []
    seen_exact: set[str] = set()
    near = NearDup()
    rejected = {"english": 0, "short": 0, "bow": 0, "duplicate": 0, "near": 0}

    for doc in DOCS:
        if bow_share(doc["text"]) > 0.35:
            rejected["bow"] += 1
            continue
        historical = doc.get("dialect") in {"historical", "watkins1865"}
        for sent in split_sentences(doc["text"]):
            guess, ratio = lang_guess(sent), cree_word_ratio(sent)
            # Nineteenth-century texts were printed without the long-vowel marks
            # the modern language detector expects, so a good historical sentence
            # can come back as "mixed". It still has to clear a higher Cree-letter
            # ratio, which keeps genuinely bilingual pages out.
            if guess != "crk" and not (historical and guess == "mixed" and ratio >= 0.9):
                rejected["english"] += 1
                continue
            if ratio < (0.75 if historical else 0.8):
                rejected["english"] += 1
                continue
            key = norm_key(sent)
            if len(key) < 12:
                rejected["short"] += 1
                continue
            if key in seen_exact:
                rejected["duplicate"] += 1
                continue
            seen_exact.add(key)
            if not near.add(sent):
                rejected["near"] += 1
                continue
            sentences.append({
                "text": sent,
                "source": doc["source"],
                "doc_id": doc["doc_id"],
                "licence": doc["licence"],
                "dialect": doc["dialect"],
                "kind": doc["kind"],
            })

    log(f"sentences kept {len(sentences)}  rejected {rejected}")

    # pairs: deduplicate, then split by document
    _PRIORITY = {"edtekla": 0, "figurative": 1, "project": 2, "watkins1865": 3}

    pairs_by_cree: dict[str, dict] = {}
    for pair in PAIRS:
        cree, english = clean_text(pair["cree"]), clean_text(pair["english"])
        if len(cree.split()) < 2 or len(english.split()) < 2:
            continue
        if cree_word_ratio(cree) < 0.55 or lang_guess(english) != "eng":
            continue
        key = norm_key(cree)
        if not key:
            continue
        rank = _PRIORITY.get(pair["source"].split("/")[0], 9)
        if key not in pairs_by_cree or rank < _PRIORITY.get(pairs_by_cree[key]["source"].split("/")[0], 9):
            pairs_by_cree[key] = {**pair, "cree": cree, "english": english}

    pairs = list(pairs_by_cree.values())
    log(f"aligned pairs kept {len(pairs)} of {len(PAIRS)}")

    # A near-duplicate check inside the pair set would delete legitimate repeated
    # short sentences ("tânisi" appears in many lessons), so pairs are deduped on
    # the exact Cree string only and leakage is handled at the group level below.


    def group_of(row: dict) -> str:
        src = row["source"].split("/")[0]
        doc = str(row.get("doc_id", ""))
        if src == "figurative":
            return f"{src}|{doc.split('/')[0]}"
        if src == "project":
            # All project pairs share one bucket; split by stable sentence hash.
            return f"{src}|{int(sha1(norm_key(row.get('cree', row.get('text', ''))))[:8], 16) % 20}"
        return f"{src}|{doc.split('#')[0]}"


    def assign(group: str) -> str:
        h = int(sha1(group)[:8], 16) % 100
        return "test" if h < 5 else ("dev" if h < 12 else "train")


    for row in sentences:
        row["split"] = assign(group_of(row))
    for row in pairs:
        row["split"] = assign(group_of(row))

    # leakage guard: nothing in dev/test may be a near-duplicate of train
    train_guard = NearDup(threshold=0.8)
    for row in sentences:
        if row["split"] == "train":
            train_guard.add(row["text"])

    leaked = 0
    for row in sentences:
        if row["split"] != "train" and train_guard.find(row["text"]) is not None:
            row["split"] = "train"
            leaked += 1
    for row in pairs:
        if row["split"] == "train":
            continue
        if train_guard.find(row["text"] if "text" in row else row["cree"]) is not None:
            row["split"] = "train"
            leaked += 1
    log(f"moved {leaked} near-duplicates out of dev/test")

    # a tiny dev/test split must still exist
    def ensure_split(rows: list[dict], minimum: int) -> None:
        """Top up the test split by moving whole groups, never single rows.

        Moving one sentence out of a document breaks the document-disjoint
        property the split is supposed to have, and a benchmark scored on a
        sibling of a training sentence is not held out in any meaningful sense.
        """
        if sum(r["split"] == "test" for r in rows) >= minimum:
            return
        groups: dict[str, list[dict]] = {}
        for row in rows:
            if row["split"] == "train":
                groups.setdefault(group_of(row), []).append(row)
        for group in sorted(groups, key=sha1):
            for row in groups[group]:
                row["split"] = "test"
            if sum(r["split"] == "test" for r in rows) >= minimum:
                break


    ensure_split(sentences, 40 if SMOKE else 300)
    ensure_split(pairs, 30 if SMOKE else 200)

    lm = {s: [r for r in sentences if r["split"] == s] for s in ("train", "dev", "test")}
    tr = {s: [r for r in pairs if r["split"] == s] for s in ("train", "dev", "test")}

    # Contamination audit: no held-out sentence and no 12-gram from a held-out
    # sentence may appear in any training split. The near-duplicate pass above
    # catches edited copies; this catches a verbatim span that survived because
    # the whole sentence was not similar enough to trip Jaccard.
    def _word_12grams(text: str) -> set[str]:
        words = re.findall(r"[^\W\d_]+", normalize_sro(str(text)).lower())
        if len(words) < 12:
            return set()
        return {" ".join(words[i:i + 12]) for i in range(len(words) - 11)}

    _train_12: set[str] = set()
    for row in lm["train"]:
        _train_12.update(_word_12grams(row["text"]))
    for row in tr["train"]:
        _train_12.update(_word_12grams(row["cree"]))
        _train_12.update(_word_12grams(row["english"]))

    # ---- contamination repair -----------------------------------------------
    # The split is document-disjoint, but two documents can still share a verbatim
    # span: a quotation, or a catechism reprinted in two collections. A held-out
    # sentence that reproduces twelve training words is not held out in any useful
    # sense, so it is moved into train here and the audit below re-checks the
    # result. The audit stays fatal on purpose: if a span survives this repair,
    # the repair itself is broken and the run must stop rather than publish a
    # benchmark scored on contaminated data.
    _repair_moved = 0
    for _pass in range(20):
        _bad: list[dict] = []
        for _split in ("dev", "test"):
            _bad += [r for r in lm[_split] if _word_12grams(r["text"]) & _train_12]
            _bad += [r for r in tr[_split]
                     if _word_12grams(r["cree"]) & _train_12
                     or _word_12grams(r["english"]) & _train_12]
        if not _bad:
            break
        for _row in _bad:
            _row["split"] = "train"
            for _field in ("text", "cree", "english"):
                if _field in _row:
                    _train_12.update(_word_12grams(_row[_field]))
        _repair_moved += len(_bad)
        lm = {s: [r for r in sentences if r["split"] == s] for s in ("train", "dev", "test")}
        tr = {s: [r for r in pairs if r["split"] == s] for s in ("train", "dev", "test")}
    if _repair_moved:
        log(f"contamination repair: moved {_repair_moved} held-out rows into train")
    log(f"split sizes: sentences {len(lm['train'])}/{len(lm['dev'])}/{len(lm['test'])}, "
        f"pairs {len(tr['train'])}/{len(tr['dev'])}/{len(tr['test'])} (train/dev/test)")

    _test_12: set[str] = set()
    for row in lm["test"]:
        _test_12.update(_word_12grams(row["text"]))
    for row in tr["test"]:
        _test_12.update(_word_12grams(row["cree"]))
        _test_12.update(_word_12grams(row["english"]))

    _contamination = sorted(_train_12 & _test_12)
    if _contamination:
        raise RuntimeError(
            "held-out 12-gram contamination: "
            f"{len(_contamination)} span(s), first: {_contamination[0][:160]!r}"
        )
    log("contamination audit: no held-out 12-gram appears in training")

    # The evaluation set for translation is modern Cree only; Watkins 1865 is a
    # separate historical capability and would distort the modern score.
    tr_test_modern = [r for r in tr["test"] if r["source"] != "watkins1865"]
    if len(tr_test_modern) < 50 and not SMOKE:
        tr_test_modern = tr["test"]

    # ---- duplicate audit ---------------------------------------------------
    # Deduplication removes repeated rows; it must never remove a unique word.
    # This audit measures both halves of that promise and writes the evidence:
    #
    #   * every word in the accepted text is counted, so "the pipeline uses
    #     every available word" is a measured set, not a claim;
    #   * repeated rows are counted at document, sentence and pair level;
    #   * spelling variants that collapse to one word (capitalisation, curly
    #     apostrophe, macron or no macron) are grouped -- that is the
    #     duplicate-word check exact hashing cannot perform;
    #   * the stage fails if an exact duplicate row is still present.
    _WORD_RE = re.compile(r"[^\W\d_]+(?:['-][^\W\d_]+)*", re.UNICODE)

    def _words(text: str) -> list[str]:
        text = normalize_sro(str(text))
        if looks_like_syllabics(text):
            text = to_sro(text)
        return _WORD_RE.findall(text.lower())

    def _folded(word: str) -> str:
        """Spelling-insensitive key: case, apostrophes and macrons removed."""
        plain = unicodedata.normalize("NFKD", word.replace("\u2019", "'"))
        plain = "".join(ch for ch in plain if not unicodedata.combining(ch))
        return re.sub(r"[^a-z]", "", plain)

    raw_words: collections.Counter = collections.Counter()
    for doc in DOCS:
        raw_words.update(_words(doc["text"]))
    for pair in PAIRS:
        raw_words.update(_words(pair["cree"]))
        raw_words.update(_words(pair["english"]))

    kept_words: collections.Counter = collections.Counter()
    for split in ("train", "dev", "test"):
        for row in lm[split]:
            kept_words.update(_words(row["text"]))
        for row in tr[split]:
            kept_words.update(_words(row["cree"]))
            kept_words.update(_words(row["english"]))

    variants: dict[str, dict[str, int]] = {}
    for word, count in kept_words.items():
        variants.setdefault(_folded(word), {})[word] = count
    variant_groups = {key: forms for key, forms in variants.items() if len(forms) > 1 and key}
    variant_examples = sorted(
        ({"word": key, "forms": dict(sorted(forms.items(), key=lambda kv: (-kv[1], kv[0]))),
          "tokens": sum(forms.values())}
         for key, forms in variant_groups.items()),
        key=lambda row: (-row["tokens"], row["word"]),
    )[:25]

    lost = {w: n for w, n in raw_words.items() if w not in kept_words}
    lost_attested = {w: n for w, n in lost.items() if w in LEXICON}

    doc_keys = [norm_key(d["text"]) for d in DOCS]
    sent_keys = [norm_key(r["text"]) for split in ("train", "dev", "test") for r in lm[split]]
    pair_keys = [f"{norm_key(r['cree'])}|{norm_key(r['english'])}"
                 for split in ("train", "dev", "test") for r in tr[split]]

    audit = {
        "scope": "all accepted rows in train/dev/test, plus everything rejected",
        "tokens": {"raw": sum(raw_words.values()), "kept": sum(kept_words.values())},
        "unique_words": {
            "raw": len(raw_words),
            "kept": len(kept_words),
            "in_lexicon": sum(1 for w in kept_words if w in LEXICON),
            "seen_once": sum(1 for n in kept_words.values() if n == 1),
        },
        "rejected_sentences": dict(rejected),
        "documents": {
            "rows": len(DOCS), "unique": len(set(doc_keys)),
            "duplicate_rows": len(DOCS) - len(set(doc_keys)),
            "note": ("documents are deduplicated at sentence level on purpose: two sources "
                     "that carry the same article keep one copy of each sentence. A "
                     "document-level repeat here is reported, not fatal."),
        },
        "sentences": {
            "rows": len(sent_keys), "unique": len(set(sent_keys)),
            "duplicate_rows": len(sent_keys) - len(set(sent_keys)),
        },
        "pairs": {
            "rows": len(pair_keys), "unique": len(set(pair_keys)),
            "duplicate_rows": len(pair_keys) - len(set(pair_keys)),
        },
        "spelling_variants": {
            "groups": len(variant_groups),
            "forms": sum(len(forms) for forms in variant_groups.values()),
            "tokens": sum(sum(forms.values()) for forms in variant_groups.values()),
            "examples": variant_examples,
        },
        "filtered_out": {
            "distinct_words": len(lost),
            "tokens": sum(lost.values()),
            "attested_cree_forms": len(lost_attested),
            "attested_examples": sorted(lost_attested)[:40],
            "note": ("filtered_out counts words in rejected lines: English, OCR noise and "
                     "too-short fragments. It is reported, not hidden, so a low token count "
                     "can be traced to a filter instead of blamed on the sources."),
        },
    }
    # Only the guarantees the stage actually makes are fatal. Duplicate
    # documents collapse into duplicate sentences, which the exact/near filters
    # remove; a document-level repeat is real overlap between sources and is
    # reported above rather than treated as a bug.
    audit["checks"] = {
        "no_duplicate_sentences": audit["sentences"]["duplicate_rows"] == 0,
        "no_duplicate_pairs": audit["pairs"]["duplicate_rows"] == 0,
        "every_word_counted": audit["unique_words"]["kept"] > 0,
    }
    jdump(PROC / "duplicate_audit.json", audit)

    log(f"duplicate audit: {audit['tokens']['kept']:,} tokens, "
        f"{audit['unique_words']['kept']:,} unique words "
        f"({audit['unique_words']['seen_once']:,} seen once), "
        f"{audit['spelling_variants']['groups']:,} spelling-variant groups")
    _duplicate_failures = [name for name, ok in audit["checks"].items() if not ok]
    if _duplicate_failures:
        raise RuntimeError("duplicate audit failed: " + ", ".join(_duplicate_failures))

    write_jsonl(PROC / "lm_train.jsonl", lm["train"])
    write_jsonl(PROC / "lm_dev.jsonl", lm["dev"])
    write_jsonl(PROC / "lm_test.jsonl", lm["test"])
    write_jsonl(PROC / "parallel_train.jsonl", tr["train"])
    write_jsonl(PROC / "parallel_dev.jsonl", tr["dev"])
    write_jsonl(PROC / "parallel_test.jsonl", tr_test_modern)
    write_jsonl(PROC / "parallel_test_historical.jsonl", [r for r in tr["test"] if r["source"] == "watkins1865"])
    write_jsonl(PROC / "notes.jsonl", NOTES)
    jdump(PROC / "lexicon.json", LEXICON)

    stats = {
        "documents": len(DOCS),
        "sentences": {k: len(v) for k, v in lm.items()},
        "sentence_words": {k: sum(len(r["text"].split()) for r in v) for k, v in lm.items()},
        "pairs": {k: len(v) for k, v in tr.items()},
        "pair_test_modern": len(tr_test_modern),
        "lexicon_forms": len(LEXICON),
        "rejected": rejected,
        "leakage_moved": leaked,
        "contamination_12gram_overlap": len(_contamination),
        "licences": sorted({r["licence"] for r in sentences})[:20],
        "duplicate_audit": {
            "tokens_kept": audit["tokens"]["kept"],
            "unique_words": audit["unique_words"]["kept"],
            "unique_words_seen_once": audit["unique_words"]["seen_once"],
            "spelling_variant_groups": audit["spelling_variants"]["groups"],
            "duplicate_documents": audit["documents"]["duplicate_rows"],
            "duplicate_sentences": audit["sentences"]["duplicate_rows"],
            "duplicate_pairs": audit["pairs"]["duplicate_rows"],
            "attested_cree_forms_filtered": audit["filtered_out"]["attested_cree_forms"],
            "report": "duplicate_audit.json",
        },
    }
    jdump(PROC / "stats.json", stats)
    print(json.dumps(stats, indent=2, ensure_ascii=False))

    # integrity assertions: the rest of the notebook relies on these
    assert lm["train"] and lm["test"], "empty LM split"
    assert tr["train"] and tr["test"], "empty pair split"
    _train_keys = {norm_key(r["text"]) for r in lm["train"]}
    assert not (_train_keys & {norm_key(r["text"]) for r in lm["test"]}), "exact LM leakage"
    assert not _contamination, "held-out 12-gram contamination"
    assert all(len(r["cree"].split()) >= 2 for r in tr["train"]), "bad pair"
    assert all(lang_guess(r["text"]) in {"crk", "mixed"} and cree_word_ratio(r["text"]) >= 0.7
               for r in lm["test"]), "English in LM test"
    print("\nintegrity checks passed: splits are disjoint and language-checked")

    # composition figure
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    counts = {k: len(v) for k, v in lm.items()}
    ax[0].bar(counts.keys(), counts.values(), color=["#2f5d50", "#c98b3a", "#8c3b2e"])
    ax[0].set_title("Cree sentences by split")
    ax[0].set_ylabel("sentences")
    for i, v in enumerate(counts.values()):
        ax[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

    by_src: dict[str, int] = {}
    for row in lm["train"]:
        by_src[row["source"].split("/")[0]] = by_src.get(row["source"].split("/")[0], 0) + 1
    ax[1].barh(list(by_src)[::-1], list(by_src.values())[::-1], color="#2f5d50")
    ax[1].set_title("Training sentences by source")
    ax[1].set_xlabel("sentences")
    fig.tight_layout()
    fig.savefig(PLOTS / "corpus_composition.png", dpi=160)
    plt.show()
    print(f"saved {PLOTS / 'corpus_composition.png'}")

    mark_stage("clean", _CLEAN_KEY, sentences=sum(len(v) for v in lm.values()),
               pairs=sum(len(v) for v in tr.values()))

## 3. The tokenizer

This is the highest-leverage change in the whole notebook.

A tokenizer trained mostly on English has never seen a word like `nêhiyawêwin` often enough to give it a single piece. It splits it into fragments, and every fragment is a chance to lose a circumflex or to spend context on a boundary that carries no meaning. Measure it and the cost is obvious: several tokens per Cree word, where a language-aware tokenizer needs closer to one and a half.

The fix is a SentencePiece BPE model trained on the Cree corpus itself. Its pieces are filtered before they are added to the base vocabulary: no space markers, no control symbols, no bare punctuation, nothing shorter than two characters. The survivors are grafted onto the base tokenizer as ordinary tokens, and the next cell initialises their embeddings from the pieces they replace. A new token for `nêhiyawêwin` starts as the average of the embeddings for `n`, `ê`, `hi`, `yaw` and so on, so the model can already read the word on the first step of training.

The audit cell reports fertility (tokens per word), characters per token, and the 95th percentile, on held-out text only. Training text would flatter any tokenizer.

In [ ]:
# A tokenizer that knows Cree.
#
# The base model's tokenizer was trained mostly on English. For a language with
# long polysynthetic words and marked vowels, that means every word becomes a
# pile of fragments: more tokens per word, more chances to drop a diacritic, and
# less text per context window.
#
# The fix is not "add some words". A SentencePiece BPE model is trained on the
# Cree corpus itself, its pieces are filtered so they can live inside the base
# model's byte-level vocabulary, and the surviving pieces are added as ordinary
# tokens. The next cell initialises their embeddings from the pieces the base
# model already knows, so a new token starts life already meaning something.

from transformers import AutoTokenizer
from sentencepiece import SentencePieceTrainer

SPECIAL_TOKENS = [
    "<|cree|>", "<|english|>", "<|sro|>", "<|cans|>",
    "<|translate|>", "<|define|>", "<|song|>", "<|note|>",
]

# The tokenizer is the one artifact whose absence is not obvious: a model with a
# vocabulary of the wrong size loads happily and produces garbage. So the guard
# checks the tokenizer folder, the token map and the audit, and rebuilds all of
# them together if any one is missing.
_TOKENIZER_KEY = fingerprint(
    sentences=(PROC / "lm_train.jsonl").stat().st_size,
    spm_vocab=CFG.spm_vocab,
    budget=CFG.new_token_budget,
    specials=len(SPECIAL_TOKENS),
)

if ((PROC / "tokenizer" / "tokenizer_config.json").exists()
        and (PROC / "new_tokens.json").exists()
        and stage_done("tokenizer", _TOKENIZER_KEY, requires=PROC / "tokenizer_audit.json")):
    comparison = jload(PROC / "tokenizer_audit.json")
    print(json.dumps(comparison, indent=2))
    log(f"tokenizer loaded from Drive: fertility {comparison['reduction_percent']:.1f}% lower than the base")
else:
    lm_train = read_jsonl(PROC / "lm_train.jsonl")
    lm_test = read_jsonl(PROC / "lm_test.jsonl")
    lexicon = jload(PROC / "lexicon.json")

    sp_input = RAW / "spm_input.txt"
    with sp_input.open("w", encoding="utf-8") as fh:
        for row in lm_train:
            fh.write(row["text"].replace("\n", " ") + "\n")
        # Word lists give the tokenizer the shapes of inflected forms that running
        # text mentions only once. They are appended once, not repeated, so they
        # cannot outvote real sentences.
        for word in sorted(lexicon, key=lambda w: -lexicon[w])[:30_000]:
            fh.write(word + "\n")

    sp_prefix = RAW / "cree_spm"
    SentencePieceTrainer.train(
        input=str(sp_input),
        model_prefix=str(sp_prefix),
        vocab_size=CFG.spm_vocab,
        model_type="bpe",
        character_coverage=0.9995,
        byte_fallback=True,
        normalization_rule_name="identity",
        split_digits=True,
        allow_whitespace_only_pieces=False,
        remove_extra_whitespaces=False,
        shuffle_input_sentence=True,
        max_sentence_length=8192,
        num_threads=max(1, min(8, os.cpu_count() or 1)),
        seed_sentencepiece_size=1_000_000,
        train_extremely_large_corpus=False,
    )

    import sentencepiece as spm

    sp = spm.SentencePieceProcessor(model_file=str(sp_prefix) + ".model")
    log(f"tokenizer: SentencePiece trained ({sp.get_piece_size()} pieces)")
    log("tokenizer: loading the base tokenizer")
    base_tok = AutoTokenizer.from_pretrained(CFG.base_model)
    log("tokenizer: base tokenizer loaded")

    # Pieces that are safe to graft onto a byte-level BPE vocabulary: no space
    # marker, no control symbol, no bare punctuation, at least two characters.
    seen = set(base_tok.get_vocab())
    candidates: list[str] = []
    for i in range(sp.get_piece_size()):
        piece = sp.id_to_piece(i)
        if not piece or piece.startswith("<"):
            continue
        piece = piece.replace("▁", "")
        if not piece or "▁" in piece:
            continue
        piece = normalize_sro(piece)
        if len(piece) < 2 or len(piece) > 32:
            continue
        if piece in seen or not re.fullmatch(r"[a-zâêîô'-]+", piece):
            continue
        seen.add(piece)
        candidates.append(piece)

    # Longest useful pieces first. A three-letter Cree stem taught as one token
    # saves more than a two-letter fragment that the base model already splits well.
    candidates.sort(key=lambda p: (-len(p), p))
    log(f"tokenizer: {len(candidates)} safe Cree pieces after filtering")
    new_tokens = candidates[: CFG.new_token_budget]

    # Syllabics are a script, not a decoration. The base tokenizer treats each
    # glyph as raw bytes, which is both wasteful and fragile. Every glyph the
    # converter knows is added as one token, so a syllable costs one token.
    syllabics_tokens = sorted({
        glyph for glyph in _SRO_TO_CANS.values()
        if len(glyph) == 1 and all(is_syllabics(ch) for ch in glyph)
    })
    for glyph in syllabics_tokens:
        if glyph not in base_tok.get_vocab() and glyph not in new_tokens:
            new_tokens.append(glyph)

    old_size = len(base_tok)
    log(f"tokenizer: adding {len(new_tokens)} pieces to a {old_size}-token vocabulary")
    _t = time.time()
    base_tok.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
    log(f"tokenizer: specials added in {time.time() - _t:.2f}s")
    new_token_start = len(base_tok)          # first ordinary token after the specials
    _t = time.time()
    vocab = base_tok.get_vocab()
    log(f"tokenizer: vocabulary read in {time.time() - _t:.2f}s")
    new_tokens = [t for t in new_tokens if t not in vocab]
    _t = time.time()
    n_new = base_tok.add_tokens(new_tokens, special_tokens=False)
    log(f"tokenizer: added {n_new} tokens in {time.time() - _t:.2f}s")
    if base_tok.pad_token is None:
        base_tok.pad_token = base_tok.eos_token
    base_tok.padding_side = "right"

    # Every new token must sit in one contiguous block, because the next cell
    # initialises their embeddings by position.
    assert len(base_tok) == new_token_start + n_new, "new tokens are not contiguous"
    assert all(base_tok.convert_tokens_to_ids(t) >= new_token_start for t in new_tokens)

    log(f"SentencePiece vocab {sp.get_piece_size()}; grafted {n_new} tokens + {len(SPECIAL_TOKENS)} specials")
    log(f"tokenizer size {len(base_tok)} (base was {old_size})")


    def fertility(tok, texts: list[str]) -> dict:
        words = sum(len(t.split()) for t in texts)
        tokens = sum(len(tok.encode(t, add_special_tokens=False)) for t in texts)
        chars = sum(len(t) for t in texts)
        per_word = [len(tok.encode(t, add_special_tokens=False)) / max(1, len(t.split())) for t in texts]
        return {
            "words": words,
            "tokens": tokens,
            "fertility": tokens / max(1, words),
            "chars_per_token": chars / max(1, tokens),
            "p50_tokens_per_word": float(np.percentile(per_word, 50)),
            "p95_tokens_per_word": float(np.percentile(per_word, 95)),
        }


    # Measure on held-out text only. Training text would flatter any tokenizer.
    eval_texts = [r["text"] for r in lm_test]
    log("tokenizer: measuring fertility on held-out text")
    base_only = AutoTokenizer.from_pretrained(CFG.base_model)
    base_only_metrics = fertility(base_only, eval_texts)
    new_metrics = fertility(base_tok, eval_texts)

    comparison = {
        "base": base_only_metrics,
        "extended": new_metrics,
        "reduction_percent": 100 * (1 - new_metrics["fertility"] / base_only_metrics["fertility"]),
    }
    jdump(PROC / "tokenizer_audit.json", comparison)
    print(json.dumps(comparison, indent=2))

    for probe in ["nêhiyawêwin", "tânisi", "kîkway", "maskwa", "êkwa mîna", to_syllabics("tânisi nêhiyawêwin")]:
        print(f"  {probe:16s} base={base_only.encode(probe, add_special_tokens=False)}")
        print(f"  {'':16s} new ={base_tok.encode(probe, add_special_tokens=False)} -> {base_tok.convert_ids_to_tokens(base_tok.encode(probe, add_special_tokens=False))}")

    base_tok.save_pretrained(PROC / "tokenizer")
    base_only.save_pretrained(PROC / "base_tokenizer")
    shutil.copyfile(str(sp_prefix) + ".model", str(PROC / "cree_spm.model"))
    jdump(PROC / "new_tokens.json", {
        "tokens": new_tokens,
        "specials": SPECIAL_TOKENS,
        "new_token_start": new_token_start,
        "old_size": old_size,
    })

    # figure
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    labels = ["base\n(Qwen)", "extended\n(+Cree)"]
    vals = [base_only_metrics["fertility"], new_metrics["fertility"]]
    bars = ax[0].bar(labels, vals, color=["#8c3b2e", "#2f5d50"])
    ax[0].set_title("Tokens per Cree word (lower is better)")
    ax[0].set_ylabel("fertility")
    for b, v in zip(bars, vals):
        ax[0].text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom")

    bins = np.linspace(0.5, 6, 34)
    ax[1].hist([len(base_only.encode(t, add_special_tokens=False)) / max(1, len(t.split())) for t in eval_texts],
               bins=bins, alpha=0.65, label="base", color="#8c3b2e", density=True)
    ax[1].hist([len(base_tok.encode(t, add_special_tokens=False)) / max(1, len(t.split())) for t in eval_texts],
               bins=bins, alpha=0.65, label="extended", color="#2f5d50", density=True)
    ax[1].set_title("Tokens per word on held-out Cree")
    ax[1].set_xlabel("tokens / word")
    ax[1].legend(frameon=False)
    fig.tight_layout()
    fig.savefig(PLOTS / "tokenizer_fertility.png", dpi=160)
    plt.show()

    assert new_metrics["fertility"] < base_only_metrics["fertility"], "tokenizer did not improve fertility"
    assert len(base_tok) > len(base_only), "no tokens were added"
    print(f"\nsaved tokenizer to {PROC / 'tokenizer'}")

    mark_stage("tokenizer", _TOKENIZER_KEY, tokens=len(new_tokens),
               fertility_drop_percent=round(100 * (1 - new_metrics["fertility"] / base_only_metrics["fertility"]), 1))

In [ ]:
# The model: quantised base, resized vocabulary, statistically initialised new
# embeddings, and LoRA adapters.
#
# New tokens start as the average of the base model's embeddings for the pieces
# it would have used to spell the same string. That is not decoration: a token
# initialised at random has to be learned from scratch, and with a corpus this
# size it never fully is. Starting from the pieces it replaces means the model
# already reads the word, and training only has to refine it.

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

TARGET_HINTS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "W_pack", "c_attn")
_TOKEN_INFO = jload(PROC / "new_tokens.json")
NEW_TOKENS = _TOKEN_INFO["tokens"]
NEW_TOKEN_START = _TOKEN_INFO["new_token_start"]


# Weights that are actually readable.
#
# `SafetensorError: Error while deserializing header: header too large` is not a
# transformers bug and expected behaviour in this notebook. It means the first eight bytes
# of a shard do not decode to a valid header length, which is what a
# multi-gigabyte file looks like when the filesystem it was written to never
# finished syncing it. In Colab that filesystem is almost always the Google
# Drive mount, which is why the weights are cached on local disk instead.
#
# So every shard is checked here, before transformers ever mmaps it, and a
# damaged shard is fetched again on its own rather than re-downloading 15 GB.
SAFETENSORS_HEADER_LIMIT = 100_000_000  # the ceiling the safetensors format itself enforces
_WEIGHT_PATTERNS = ["*.safetensors", "*.json", "*.model", "*.txt", "*.jinja", "tokenizer*"]
_SKIP_PATTERNS = ["*.bin", "*.h5", "*.msgpack", "*.gguf", "*.onnx", "*.onnx_data", "*.pth", "*.pt"]


def free_gb(path: Path | str) -> float:
    """Free space on the filesystem that will hold `path`, creating nothing."""
    probe = Path(path)
    while not probe.exists() and probe != probe.parent:
        probe = probe.parent
    try:
        return shutil.disk_usage(str(probe)).free / 1024 ** 3
    except OSError:
        return float("nan")


def safetensors_problem(path: Path) -> str | None:
    """Why this shard cannot be read, or None when it is intact.

    The check is the same one the loader performs, done in plain file reads so it
    cannot itself fail with the error it is trying to explain: an 8-byte
    little-endian header length, a JSON header of that many bytes, and tensor
    offsets that fit inside the file.
    """
    path = Path(path)
    try:
        size = path.stat().st_size
    except OSError as exc:
        return f"the file cannot be opened ({exc})"
    if size < 9:
        return f"the file is only {size} bytes"
    try:
        with path.open("rb") as fh:
            header_len = int.from_bytes(fh.read(8), "little")
            if not 0 < header_len <= SAFETENSORS_HEADER_LIMIT:
                return (f"its first 8 bytes decode to a {header_len:,}-byte header, outside the valid "
                        f"1 to {SAFETENSORS_HEADER_LIMIT:,} range")
            if 8 + header_len > size:
                return f"its header claims {header_len:,} bytes but the whole file is {size:,}"
            header = json.loads(fh.read(header_len).decode("utf-8"))
            if not isinstance(header, dict):
                return "its header is not a JSON object"
            end = 8 + header_len
            for name, info in header.items():
                if name == "__metadata__":
                    continue
                offsets = (info or {}).get("data_offsets")
                if not (isinstance(offsets, list) and len(offsets) == 2):
                    return f"tensor {name!r} has no usable data_offsets"
                end = max(end, 8 + header_len + int(offsets[1]))
    except (OSError, ValueError, UnicodeDecodeError) as exc:
        return f"its header is unreadable ({exc})"
    if end > size:
        return f"it declares {end:,} bytes of tensors but the file is only {size:,}"
    return None


def verify_weights(folder: Path, quiet: bool = False) -> list[tuple[Path, str]]:
    shards = sorted(Path(folder).glob("*.safetensors"))
    if not shards:
        raise RuntimeError(
            f"no .safetensors weights in {folder}. Point CFG.base_model at the folder that holds the shards."
        )
    bad = [(p, why) for p in shards if (why := safetensors_problem(p)) is not None]
    if not bad and not quiet:
        total = sum(p.stat().st_size for p in shards) / 1024 ** 3
        log(f"base weights verified: {len(shards)} shard(s), {total:.1f} GB in {folder}")
    return bad


def _damage_message(folder: Path, bad: list[tuple[Path, str]]) -> str:
    rows = []
    for path, why in bad:
        gb = path.stat().st_size / 1024 ** 3 if path.exists() else float("nan")
        rows.append(f"  {path.name}  {gb:.2f} GB  ({why})")
    return (
        "A checkpoint shard is damaged, so the weights cannot be loaded:\n"
        + "\n".join(rows)
        + f"\n\nFolder: {folder}\nFree space there: {free_gb(folder):.1f} GB"
        + f"\nHugging Face cache: {HF_CACHE}\n\n"
        + "Run this cell again to fetch the damaged file fresh, or delete it and re-copy it if "
        + "this is a local folder. Keep the cache on the local runtime disk: a multi-gigabyte shard "
        + "written through a Google Drive mount can come back half-synced, and that is what "
        + "produces `SafetensorError: header too large`."
    )


def ensure_base_weights(repair: bool = True, force_check: bool = False) -> str:
    """A base-model folder whose every shard passed the header check."""
    local = Path(str(CFG.base_model)).expanduser()
    from_hub = not local.is_dir()
    if from_hub:
        from huggingface_hub import HfApi, hf_hub_download, snapshot_download

        try:
            siblings = HfApi().model_info(CFG.base_model, files_metadata=True).siblings or []
            need_bytes = sum((s.size or 0) for s in siblings if s.rfilename.endswith((".safetensors", ".json")))
            need_gb = need_bytes / 1024 ** 3
            room = free_gb(HF_CACHE)
            log(f"base model {CFG.base_model}: about {need_gb:.1f} GB of weights, "
                f"{room:.1f} GB free at {HF_CACHE}")
            if need_gb and not math.isnan(room) and need_gb > room * 0.95:
                raise RuntimeError(
                    f"Downloading {CFG.base_model} needs about {need_gb:.1f} GB and only "
                    f"{room:.1f} GB is free at {HF_CACHE}. Free space, or set "
                    f"CREE_HF_HOME to a larger disk. A download that runs out of room is one of "
                    f"the ways a shard ends up damaged."
                )
        except RuntimeError:
            raise
        except Exception as exc:
            log(f"could not read the file sizes ahead of time ({exc}); continuing")

        folder = Path(snapshot_download(
            CFG.base_model,
            allow_patterns=_WEIGHT_PATTERNS,
            ignore_patterns=_SKIP_PATTERNS,
        ))
    else:
        folder = local

    bad = verify_weights(folder, quiet=not force_check)
    if not bad:
        return str(folder)
    for path, why in bad:
        log(f"damaged shard {path.name}: {why}")
    if not repair or not from_hub:
        raise RuntimeError(_damage_message(folder, bad))

    for path, _ in bad:
        log(f"re-downloading {path.name} into {HF_CACHE}")
        hf_hub_download(CFG.base_model, path.relative_to(folder).as_posix(), force_download=True)
    bad = verify_weights(folder, quiet=True)
    if bad:
        for path, why in bad:
            log(f"still damaged: {path.name}: {why}")
        raise RuntimeError(_damage_message(folder, bad))
    log("damaged shards replaced; base weights verified")
    return str(folder)


def find_target_modules(model) -> list[str]:
    names = set()
    for name, module in model.named_modules():
        leaf = name.split(".")[-1]
        if leaf in TARGET_HINTS and hasattr(module, "weight"):
            names.add(leaf)
    if not names:
        raise RuntimeError("no LoRA target modules found; check the base model architecture")
    return sorted(names)


def init_new_embeddings(model, decomposer, tokens: list[str], start: int) -> None:
    """Give each new token the average embedding of the pieces it replaces."""
    emb = model.get_input_embeddings()
    weight = emb.weight
    device = weight.device
    unk = decomposer.unk_token_id or 0
    with torch.no_grad():
        rows = []
        for token in tokens:
            ids = decomposer.encode(token, add_special_tokens=False) or [unk]
            rows.append(weight[torch.tensor(ids, device=device)].mean(dim=0))
        weight[start: start + len(rows)] = torch.stack(rows).to(weight.dtype)

    out = model.get_output_embeddings()
    if out is not None and not model.config.tie_word_embeddings and out.weight.shape[0] >= start + len(tokens):
        with torch.no_grad():
            out.weight[start: start + len(tokens)] = weight[start: start + len(tokens)].to(out.weight.dtype)


def _dtype_kwargs(dtype=None) -> dict:
    """The keyword this transformers install calls the weight dtype.

    `torch_dtype` was renamed to `dtype` in transformers 4.56 and the old name
    now prints a deprecation warning on every load. Passing whichever name the
    installed version understands keeps the notebook clean on Colab, which pins
    a range rather than one version.
    """
    import transformers

    try:
        major, minor = (int(part) for part in transformers.__version__.split(".")[:2])
    except ValueError:
        major, minor = (0, 0)
    return {"dtype": dtype or DTYPE} if (major, minor) >= (4, 56) else {"torch_dtype": dtype or DTYPE}


# The grafted rows live in one small file instead of inside the adapter.
# 3,949 rows of 3,584 values is 28 MB; keeping them separate is what lets the
# embedding table stay frozen, and that is the whole memory fix.
GRAFTED = PROC / "grafted_embeddings.pt"
# Which vocabulary these rows belong to. A rebuilt tokenizer puts different
# tokens at the same offsets, so a saved file cannot be reused just because it
# exists: the graft would quietly hand a new token another token's embedding,
# and every number measured afterwards would be measured on that.
_GRAFT_KEY = fingerprint(
    base=CFG.base_model, start=NEW_TOKEN_START, n=len(NEW_TOKENS),
    tokens=sha1("\n".join(NEW_TOKENS)),
)


def graft_rows(model, tok) -> None:
    """Resize the vocabulary and restore the grafted rows.

    `resize_token_embeddings` fills new rows with random values, so resizing
    alone would throw away the statistical start every time a cell reloads the
    model. The graft is written once, on the first load, and copied back on every
    later one, which also means the export cell sees exactly the embeddings that
    training saw.
    """
    n = len(tok)
    saved = None
    if GRAFTED.exists():
        try:
            saved = torch.load(GRAFTED, map_location="cpu", weights_only=True)
        except Exception as exc:
            log(f"graft: {GRAFTED.name} is unreadable ({exc}); rebuilding it")
            saved = None
        if saved is not None and saved.get("key") != _GRAFT_KEY:
            reason = ("these rows predate the vocabulary key" if "key" not in saved
                      else "the vocabulary changed")
            log(f"graft: {reason}; rebuilding the rows from the base pieces")
            saved = None

    if saved is None:
        if model.get_input_embeddings().weight.shape[0] < n:
            model.resize_token_embeddings(n)
        try:
            decomposer = AutoTokenizer.from_pretrained(PROC / "base_tokenizer")
        except Exception as exc:
            # A file written before the key existed cannot be checked, and the
            # rows cannot be recomputed without the base tokenizer. Using them
            # unchecked is the old behaviour and is right whenever the
            # vocabulary did not change, so warn rather than stop an export that
            # is otherwise finished.
            if not GRAFTED.exists():
                raise RuntimeError(
                    f"the vocabulary graft is missing and cannot be rebuilt: {exc}"
                ) from exc
            log(f"graft: {GRAFTED.name} cannot be checked or rebuilt ({exc}); using its rows as they are")
            saved = torch.load(GRAFTED, map_location="cpu", weights_only=True)
        if saved is None:
            init_new_embeddings(model, decomposer, NEW_TOKENS, NEW_TOKEN_START)
            rows = model.get_input_embeddings().weight[NEW_TOKEN_START: NEW_TOKEN_START + len(NEW_TOKENS)]
            rows = rows.detach().cpu().clone()
            saved = {"key": _GRAFT_KEY, "start": NEW_TOKEN_START, "rows": rows}
            torch.save(saved, GRAFTED)
            log(f"grafted {len(NEW_TOKENS)} new rows from base subword pieces -> {GRAFTED.name}")

    # Applied whenever the file exists, even if the sizes already agree: a model
    # that was resized elsewhere carries random rows, and matching a length is
    # not the same as holding the graft.
    start = int(saved["start"])
    rows = saved["rows"]
    need = start + rows.shape[0]
    if model.get_input_embeddings().weight.shape[0] < need:
        model.resize_token_embeddings(n)
    for module in (model.get_input_embeddings(), model.get_output_embeddings()):
        if module is None or module.weight.shape[0] < need:
            continue
        with torch.no_grad():
            module.weight[start:need] = rows.to(module.weight.dtype)


def freeze_grafted(model) -> None:
    """Keep the 7B vocabulary table out of the optimizer.

    Marking `embed_tokens` and `lm_head` for saving means 1.1 billion extra
    trainable parameters. In fp32 with gradients and Adam state that is roughly
    11 GB, and on a 14.56 GiB T4 it is what ended the first run at 14.28 GiB
    allocated with a failed 608 MiB allocation at cross-entropy. The grafted
    rows already start at the average of the pieces they replace, so the adapter
    is what trains and the table stays a fixed, readable input.
    """
    for module in (model.get_input_embeddings(), model.get_output_embeddings()):
        if module is not None:
            module.weight.requires_grad_(False)


PLAN_FILE = CKPT / "plan.json"
TRAIN_STAGES = ("cpt", "sft", "dpo")


def reset_stale_checkpoints() -> None:
    """Move adapters trained under a different plan out of the way.

    An adapter saved while the vocabulary table was trainable carries extra
    tensors and a different adapter_config. Reusing that folder for a run with a
    frozen table mixes two incompatible checkpoints, and the mismatch surfaces
    hours later. Renaming keeps the old run instead of deleting it.
    """
    # `four_bit` is not part of this. It describes how the base was
    # loaded in that session and changes not one adapter tensor, so two runs of
    # the same recipe on cards with different free memory must not move each
    # other's adapters aside. Older plan files carry the field; dropping it from
    # the comparison is what keeps this change from reading as a plan change.
    plan = {
        "base": CFG.base_model,
        "trainable_embeddings": False,
        "lora_r": CFG.lora_r,
    }
    previous = jload(PLAN_FILE) if PLAN_FILE.exists() else None
    if isinstance(previous, dict):
        previous.pop("four_bit", None)
    if previous and previous != plan:
        stamp = time.strftime("%Y%m%d-%H%M%S")
        for name in TRAIN_STAGES:
            folder = CKPT / name
            if folder.exists():
                folder.rename(CKPT / f"{name}.stale-{stamp}")
        # The adapters are gone, so their finished markers are lies. Clearing
        # them is what makes the next run retrain instead of skipping to an
        # adapter that no longer exists.
        clear_stage(*TRAIN_STAGES)
        log(f"previous adapters used a different plan {previous}; moved them to *.stale-{stamp}")
    jdump(PLAN_FILE, plan)


def load_model(adapter: Path | None = None, trainable: bool = True):
    tok = AutoTokenizer.from_pretrained(PROC / "tokenizer")
    kwargs: dict = {**_dtype_kwargs(), "low_cpu_mem_usage": True}
    if USE_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=DTYPE,
        )
        kwargs["device_map"] = {"": 0}
    elif DEVICE == "cuda":
        kwargs["device_map"] = {"": 0}
    # No device_map on CPU: a single-device map buys nothing there and would
    # make the CPU smoke run depend on `accelerate` being installed.

    model = AutoModelForCausalLM.from_pretrained(ensure_base_weights(), **kwargs)
    model.config.pad_token_id = tok.pad_token_id
    graft_rows(model, tok)

    if trainable and USE_4BIT:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )
    elif trainable and DEVICE == "cuda":
        # bf16 on a card large enough to skip quantisation still cannot hold the
        # activations of a 2048-token sequence without checkpointing, and an
        # explicit use_reentrant silences the warning torch 2.9 turns into an
        # error.
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    if trainable:
        model.enable_input_require_grads()

    if adapter is not None:
        from peft import PeftModel

        # Resume an existing run: load its adapters, do not wrap them in a second
        # set of fresh ones.
        model = PeftModel.from_pretrained(model, str(adapter), is_trainable=trainable)
        if trainable:
            freeze_grafted(model)
            model.print_trainable_parameters()
        log(f"loaded adapter from {adapter}")
    elif trainable:
        lora = LoraConfig(
            r=CFG.lora_r,
            lora_alpha=CFG.lora_alpha,
            lora_dropout=CFG.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=find_target_modules(model),
        )
        model = get_peft_model(model, lora)
        freeze_grafted(model)
        model.print_trainable_parameters()

    model.config.use_cache = False
    return model, tok


def fresh_if_forced(name: str) -> None:
    """A forced stage starts over instead of continuing its old checkpoints.

    `CREE_FORCE=cpt` is documented as "run this stage again", and resuming the
    checkpoints of the run being replaced would make it a no-op that quietly
    trained a few more steps. The old folder is renamed rather than deleted, so
    a force is still reversible.
    """
    if name not in FORCE:
        return
    folder = CKPT / name
    if folder.exists():
        stamp = time.strftime("%Y%m%d-%H%M%S")
        folder.rename(CKPT / f"{name}.forced-{stamp}")
        log(f"{name}: forced, previous folder moved to {name}.forced-{stamp}")


def adapter_ready(name: str) -> Path | None:
    """The adapter folder for a finished stage, or None.

    The root of a stage folder is written once, after training returns. Partial
    saves go into `checkpoint-N/` subfolders, so a stage that was interrupted
    half way is not mistaken for a finished one.
    """
    folder = CKPT / name
    weights = (folder / "adapter_model.safetensors").exists() or (folder / "adapter_model.bin").exists()
    if (folder / "adapter_config.json").exists() and weights:
        return folder
    return None


_LOADED_ADAPTER = ""


def best_adapter(upto: str | None = None) -> Path | None:
    """The newest finished adapter at or before `upto`.

    This is what keeps a fresh kernel accurate. Re-running the SFT cell after a
    disconnect used to load the bare base model and quietly train instruction
    tuning without the Cree pretraining underneath it, which is worse than not
    training at all because the scores still look plausible.

    `CREE_ADAPTER_STAGE=sft` scores a finished earlier stage instead of the
    newest one. Without it, deciding whether the last stage helped means
    retraining everything under it; with it, the question is one eval run.
    """
    wanted = os.environ.get("CREE_ADAPTER_STAGE", "").strip().lower()
    if upto is None and wanted:
        folder = adapter_ready(wanted)
        if folder is not None:
            log(f"adapter: CREE_ADAPTER_STAGE={wanted} -> {folder}")
            return folder
        log(f"adapter: CREE_ADAPTER_STAGE={wanted} has no finished adapter; using the newest")
    names = TRAIN_STAGES if upto is None else TRAIN_STAGES[: TRAIN_STAGES.index(upto) + 1]
    for name in reversed(names):
        folder = adapter_ready(name)
        if folder is not None:
            return folder
    return None


# A scoring run asks what an existing adapter scores. Nothing underneath it may
# be trained on the way to that answer: a changed fingerprint would otherwise
# retrain six hours of pretraining before the benchmark it was only asked to
# measure. Every training stage checks this, not just the eval cell.
SCORING_ONLY = bool(os.environ.get("CREE_ADAPTER_STAGE", "").strip())


def ensure_model(upto: str | None = None, force: bool = False):
    """`model` and `tok` in the kernel, continued from the best saved adapter."""
    global model, tok, _LOADED_ADAPTER
    target = best_adapter(upto)
    wanted = str(target) if target is not None else ""
    if not force and "model" in globals() and "tok" in globals() and _LOADED_ADAPTER == wanted:
        return model, tok
    model, tok = load_model(adapter=target)
    _LOADED_ADAPTER = wanted
    return model, tok


class CompletionOnlyCollator:
    """Masks everything except the assistant's reply from the loss.

    Training on the whole conversation teaches the model to generate the user's
    question too, which shows up later as a model that answers by asking. Only
    the reply is scored here.
    """

    def __init__(self, tok, max_len: int):
        self.tok = tok
        self.max_len = max_len

    def _one(self, messages: list[dict]) -> dict:
        prompt = self.tok.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
        full = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        f_ids = self.tok(full, add_special_tokens=False)["input_ids"]
        if f_ids[: len(p_ids)] != p_ids:
            # Template edge case: fall back to scoring the whole example rather
            # than silently misaligning the mask.
            p_ids = []
        f_ids = f_ids[: self.max_len]
        labels = [-100] * min(len(p_ids), len(f_ids)) + f_ids[min(len(p_ids), len(f_ids)):]
        return {"input_ids": f_ids, "labels": labels, "attention_mask": [1] * len(f_ids)}

    def __call__(self, features: list[dict]) -> dict:
        rows = [self._one(f["messages"]) for f in features]
        max_len = max(len(r["input_ids"]) for r in rows)
        pad = self.tok.pad_token_id
        out = {"input_ids": [], "labels": [], "attention_mask": []}
        for r in rows:
            n = max_len - len(r["input_ids"])
            out["input_ids"].append(r["input_ids"] + [pad] * n)
            out["labels"].append(r["labels"] + [-100] * n)
            out["attention_mask"].append(r["attention_mask"] + [0] * n)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


class PackedCausalDataset(torch.utils.data.Dataset):
    """Concatenated token streams cut into full-length windows.

    Padding is waste. Packing joins documents with an EOS and slices the stream
    into exact windows, so every position in a batch carries a real token.
    """

    def __init__(self, texts: list[str], tok, seq_len: int, limit: int | None = None):
        ids: list[int] = []
        eos = tok.eos_token_id
        for text in texts:
            ids.extend(tok(text, add_special_tokens=False)["input_ids"])
            ids.append(eos)
            if limit and len(ids) > limit * seq_len:
                break
        n = len(ids) // seq_len
        self.blocks = [ids[i * seq_len:(i + 1) * seq_len] for i in range(n)]

    def __len__(self) -> int:
        return len(self.blocks)

    def __getitem__(self, i: int) -> dict:
        block = self.blocks[i]
        return {"input_ids": block, "labels": list(block), "attention_mask": [1] * len(block)}


class BlockDataset(torch.utils.data.Dataset):
    def __init__(self, blocks: list[list[int]]):
        self.blocks = blocks

    def __len__(self) -> int:
        return len(self.blocks)

    def __getitem__(self, i: int) -> dict:
        block = self.blocks[i]
        return {"input_ids": block, "labels": list(block), "attention_mask": [1] * len(block)}


class ChatDataset(torch.utils.data.Dataset):
    def __init__(self, rows: list[dict]):
        self.rows = rows

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, i: int) -> dict:
        return {"messages": self.rows[i]["messages"]}


def train_args(out_dir: Path, epochs: float, lr: float, micro: int, accum: int,
               warmup_steps: int = 0, save_steps: int = 0) -> TrainingArguments:
    return TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=micro,
        gradient_accumulation_steps=accum,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        weight_decay=CFG.weight_decay,
        max_grad_norm=1.0,
        logging_steps=5,
        logging_first_step=True,
        save_strategy="steps" if save_steps else "no",
        save_steps=max(1, save_steps),
        # Two checkpoints, so a write torn by a disconnect falls back to the
        # previous one instead of losing the stage. The optimizer and scheduler
        # are saved too: without the scheduler a resumed run restarts the cosine
        # curve from step zero and jumps the learning rate back to its peak,
        # which is a bigger change to the run than the extra 160 MB per save.
        save_total_limit=2,
        save_only_model=False,
        report_to="none",
        remove_unused_columns=False,
        fp16=(DEVICE == "cuda" and not BF16),
        bf16=BF16,
        optim="paged_adamw_8bit" if (USE_4BIT and DEVICE == "cuda") else "adamw_torch",
        dataloader_num_workers=0 if DEVICE == "cpu" else 2,
        dataloader_pin_memory=True,
        seed=CFG.seed,
        data_seed=CFG.seed,
    )


class MetricsWriter(TrainerCallback):
    """Writes every logged row to Drive as it is produced.

    `trainer.state.log_history` lives in the kernel, so a disconnect deleted the
    loss curve along with the run and the report card had nothing to plot. A row
    written here survives, and a resumed run appends to the same file, so the
    curve after three sessions reads as the curve of one run.
    """

    def __init__(self, tag: str):
        self.tag = tag

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or not state.global_step:
            return
        row = {"step": int(state.global_step), "epoch": round(float(state.epoch or 0), 3)}
        row.update({k: v for k, v in logs.items() if isinstance(v, (int, float))})
        append_metrics(self.tag, [row])


def pick_resume_checkpoint(out_dir: Path) -> Path | None:
    """The newest checkpoint that is complete enough to continue from.

    A disconnect can land in the middle of a checkpoint write, and Drive happily
    returns the half file afterwards. So the newest folder is checked rather
    than trusted: it needs a trainer state and an adapter whose safetensors
    header is intact. A torn checkpoint is skipped and the previous one is used,
    which costs at most one save interval.
    """
    for folder in reversed(checkpoints(out_dir)):
        state = folder / "trainer_state.json"
        adapter = folder / "adapter_model.safetensors"
        if not state.exists():
            log(f"ignoring {folder.name}: no trainer_state.json, it was cut off mid-write")
            continue
        if not adapter.exists() and not (folder / "adapter_model.bin").exists():
            log(f"ignoring {folder.name}: no adapter weights")
            continue
        if adapter.exists():
            problem = safetensors_problem(adapter)
            if problem:
                log(f"ignoring {folder.name}: {problem}")
                continue
        return folder
    return None


def prune_checkpoints(folder: Path | str, keep: int = 2) -> None:
    for old in checkpoints(folder)[:-keep]:
        shutil.rmtree(old, ignore_errors=True)


def run_training(model, dataset, collator, out_dir: Path, epochs: float, lr: float,
                 micro: int, accum: int, tag: str,
                 save_minutes: float | None = None, key: str = "") -> list[dict]:
    """Train, checkpointing to Drive and continuing from the last checkpoint.

    The save interval is derived from the schedule rather than fixed, because a
    200-step stage and a 2,000-step stage need different granularity: eight
    checkpoints per stage keeps a disconnect cheap without letting Drive writes
    dominate the run.
    """
    if len(dataset) == 0:
        raise RuntimeError(f"{tag}: empty training set")
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Checkpoints belong to the data and schedule that produced them. A changed
    # instruction set changes the stage fingerprint, and resuming the old
    # checkpoints would train the new examples on top of the old run without
    # anyone noticing. A mismatch moves the old folder aside instead.
    key_path = out_dir / "stage_key.json"
    same_run = bool(key) and key_path.exists() and jload(key_path).get("key") == key
    if key and not same_run:
        if any(out_dir.glob("checkpoint-*")) or (out_dir / "adapter_config.json").exists():
            stamp = time.strftime("%Y%m%d-%H%M%S")
            stale = out_dir.with_name(f"{out_dir.name}.stale-{stamp}")
            out_dir.rename(stale)
            out_dir.mkdir(parents=True, exist_ok=True)
            log(f"{tag}: inputs changed, previous checkpoints moved to {stale.name}")
        jdump(key_path, {"key": key})

    steps = math.ceil(len(dataset) / (micro * accum) * epochs)
    warmup = max(1, int(steps * CFG.warmup_ratio))
    save_steps = max(10, steps // 8)
    args = train_args(out_dir, epochs, lr, micro, accum, warmup_steps=warmup, save_steps=save_steps)

    resume = pick_resume_checkpoint(out_dir) if same_run or not key else None
    trainer = Trainer(
        model=model, args=args, train_dataset=dataset, data_collator=collator,
        callbacks=[MetricsWriter(tag)],
    )
    log(f"{tag}: {len(dataset)} examples, {epochs} epochs -> about {steps} optimizer steps")
    log(f"{tag}: a checkpoint is written to {out_dir} every {save_steps} steps")
    log(f"{tag}: resuming from {resume.name}" if resume else f"{tag}: starting from step 0")

    t0 = time.time()
    trainer.train(resume_from_checkpoint=str(resume) if resume is not None else None)
    minutes = (time.time() - t0) / 60
    log(f"{tag}: done in {minutes:.1f} min")

    # The stage root is written last and is what adapter_ready() looks for, so a
    # run that dies before this line is never reported as finished.
    model.save_pretrained(str(out_dir))
    # The kernel now holds exactly these weights, so the next cell does not pay
    # for reloading them from disk.
    global _LOADED_ADAPTER
    _LOADED_ADAPTER = str(out_dir)
    prune_checkpoints(out_dir)
    append_metrics(tag, [{"step": int(trainer.state.global_step), "event": "finished",
                          "minutes": round(minutes, 1)}])
    return read_metrics(tag)


def loss_curve(history: list[dict], out_path: Path, title: str) -> None:
    import matplotlib.pyplot as plt

    points = [(h["step"], h["loss"]) for h in history if "loss" in h and "step" in h]
    if not points:
        return
    steps, losses = zip(*points)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(steps, losses, marker="o", ms=3, color="#2f5d50")
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("training loss")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()
    print(f"saved {out_path}")


# Adapters trained under a different plan are moved aside before any cell can
# load them. Done here rather than in the CPT cell because a run that restarts
# at SFT must hit the same guard.
reset_stale_checkpoints()

print("model helpers ready")

## 4. Continued pretraining

Instruction tuning teaches a model what to say. It does not teach it how a language works. That is what this stage is for: next-token prediction on running Cree, so the model absorbs spelling, suffix patterns and the rhythm of a Cree sentence before it is asked to follow an instruction.

Two details keep it from doing damage. The first is replay: one packed block in ten is English. A small LoRA trained only on Cree can drift far enough that it forgets how to follow an English instruction, which would make the model useless as an assistant. The second is packing: documents are joined with an end-of-sequence token and sliced into full windows, so every position in a batch carries a real token instead of padding.

The adapter is small. With a corpus of this size, a high-rank full fine-tune would memorise the text and lose the base model's general ability. A rank-64 LoRA over the attention and MLP projections can learn the language's shape while leaving the rest of the model intact.

In [ ]:
# Continued pretraining (CPT).
#
# Instruction tuning alone teaches a model what to say, not how the language
# works. CPT is the stage that lets the model absorb Cree spelling, morphology
# and sentence rhythm from running text. Two details keep it from doing damage:
#
#   * **Replay.** One block in ten is English. Without it a small LoRA can drift
#     far enough that the model forgets how to follow an English instruction --
#     the exact failure that makes a specialist model useless as an assistant.
#   * **Packing.** Documents are joined with an EOS and sliced into full
#     windows, so no compute is spent on padding.
#
# This is the longest stage, so it is also the one a disconnect hurts most. It
# checks the run ledger first: if CPT already finished, the packed dataset is
# not rebuilt and the English replay is not downloaded again, the trained
# adapter is loaded, and the cell moves on. If it was interrupted, the trainer
# continues from its last checkpoint on Drive.
_CPT_KEY = fingerprint(
    base=CFG.base_model, seq=CFG.seq_len, lora=CFG.lora_r,
    epochs=CFG.cpt_epochs, sentences=(PROC / "lm_train.jsonl").stat().st_size,
)

if SCORING_ONLY or stage_done("cpt", _CPT_KEY, requires=CKPT / "cpt" / "adapter_config.json"):
    model, tok = ensure_model()
    cpt_history = read_metrics("CPT")
else:
    fresh_if_forced("cpt")
    cree_texts = [r["text"] for r in read_jsonl(PROC / "lm_train.jsonl")]

    # The tokenizer is loaded first so the packed dataset can use it.
    from transformers import AutoTokenizer as _AutoTokenizer

    _tok = _AutoTokenizer.from_pretrained(PROC / "tokenizer")
    cree_ds = PackedCausalDataset(cree_texts, _tok, CFG.seq_len, limit=CFG.max_cpt_packs)
    log(f"CPT: {len(cree_texts)} Cree sentences -> {len(cree_ds)} packed blocks of {CFG.seq_len} tokens")

    english_blocks: list[list[int]] = []
    try:
        from datasets import load_dataset

        wt = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
        en_texts = [t for t in wt["text"] if len(t.strip()) > 200]
        if SMOKE:
            en_texts = en_texts[:40]
        en_ds = PackedCausalDataset(en_texts, _tok, CFG.seq_len, limit=max(8, len(cree_ds) // 9))
        english_blocks = en_ds.blocks
        log(f"CPT: {len(english_blocks)} English replay blocks (10% of the stream)")
    except Exception as exc:
        log(f"CPT: English replay unavailable ({exc}); training on Cree only")

    mixed_blocks: list[list[int]] = []
    for i, block in enumerate(cree_ds.blocks):
        if english_blocks and i % 9 == 8:
            mixed_blocks.append(english_blocks[(i // 9) % len(english_blocks)])
        else:
            mixed_blocks.append(block)

    cpt_data = BlockDataset(mixed_blocks)
    del _tok

    # CPT always begins at the base model. A resumed CPT does not need the
    # adapter loaded here: the trainer restores it from its checkpoint.
    model, tok = load_model()
    model.train()

    cpt_history = run_training(
        model, cpt_data, None, CKPT / "cpt",
        epochs=CFG.cpt_epochs, lr=CFG.cpt_lr,
        micro=CFG.micro_batch, accum=CFG.grad_accum, tag="CPT", key=_CPT_KEY,
    )
    mark_stage("cpt", _CPT_KEY, sentences=len(cree_texts), blocks=len(cpt_data))
    log(f"CPT adapter saved to {CKPT / 'cpt'}")

loss_curve(cpt_history, PLOTS / "cpt_loss.png", "Continued pretraining loss")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. Instruction data and tuning

The training set is built, not scraped from a chat log. There is one model, so every job the assistant does is in this one set.

**Translation** is the core language skill: English to Cree and Cree to English, with several natural phrasings of each instruction so the model does not learn one magic sentence. **Orthography** pairs SRO with syllabics in both directions, using the same conversion tables as the rest of the project, so the model can write the script rather than merely name it. **Word explanation** turns the short aligned pairs into a gloss, so "what does this word mean" is answered from the corpus instead of guessed. **Song notes** give the model a passage and a question and ask it to answer from the passage, naming the song, which is exactly the shape the app's retrieval layer produces. **Abstention** gives it a passage that does not contain the answer and rewards saying so. **Project** examples teach it what this system is, what it will not claim, and how it is installed. **Persona** examples establish that it is Archer's assistant, not Archer and not a performer.

The mix is the point. A model tuned only on translation becomes a translator with no conversation; a model tuned only on chat cannot spell Cree. Both are in the same adapter, and the task prefix selects the behaviour without selecting a different model.

The held-out section of the notes is never trained on. It is used later to test whether the model can read a passage it has not memorised, which is the only kind of note reading that generalises.

Only the assistant's reply contributes to the loss. Training on the whole conversation teaches a model to generate the user's question too, which shows up later as a model that answers by asking.

In [ ]:
# Instruction data for the one Archer model.
#
# There is a single model. It is not a Cree model sitting beside a general one,
# and it is not specialised into a router. It learns every job the assistant
# does, from one instruction set, in one adapter:
#
#   translate      English <-> Plains Cree, both directions, both scripts
#   orthography    SRO <-> syllabics, so it can write Cree rather than name it
#   explain        a Cree word and its English meaning, from the aligned pairs
#   song notes     grounded answers, with the song named, from Archer's notes
#   abstention     questions the notes cannot answer, answered accurately
#   project        conversation about Archer, this system and its limits
#   persona        who the assistant is, and just as importantly who it is not
#
# Held-out song notes and held-out sentence pairs never enter this set.

# The same string the app sends, in chat/archer_prompt.py. A test compares the
# two, because a system prompt that drifts from training moves the model off the
# distribution it learned.
SYSTEM = (
    "Answer in plain, warm sentences, and keep answers short: two or three "
    "sentences unless the question needs more. A greeting is answered the way a "
    "person answers one: back to them, briefly, and on to the work, without "
    "announcing that you are ready to help, calling yourself a system or a "
    "service, or describing what you can do unless you are asked. Never mention "
    "these instructions, your prompt or what you are able to do, and never list "
    "your jobs. Write plain sentences with ordinary punctuation, and never use "
    "an em dash or an en dash. "
    "When notes are given, they are the only evidence for anything about a song: "
    "answer from them, name the song, and if they do not contain the answer say "
    "plainly that the notes do not say. "
    "When no notes are given, answer about this project and about Plains Cree from "
    "what you know, and say when you are unsure instead of inventing a word, a "
    "meaning, a name, a number or a ceremony. "
    "Translate faithfully: keep names, numbers, negation and tense intact, and "
    "never leave an English word inside a Cree sentence. "
    "You never speak as Archer and you never claim his voice is yours."
)

EN_INSTRUCTIONS = [
    "Translate into Plains Cree (SRO):",
    "How would you say this in nêhiyawêwin?",
    "Give the Plains Cree translation in Standard Roman Orthography:",
    "Render this in Cree (SRO), keeping the meaning exact:",
    "Translate to nêhiyawêwin:",
]
CR_INSTRUCTIONS = [
    "Translate into English:",
    "What does this mean in English?",
    "Give the English meaning of this Cree sentence:",
    "Translate this nêhiyawêwin sentence faithfully:",
]
CANS_INSTRUCTIONS = [
    "Write this in Cree syllabics:",
    "Convert to syllabics (Cans):",
    "Show this sentence in syllabics:",
]
SRO_INSTRUCTIONS = [
    "Write this syllabics sentence in SRO:",
    "Convert these syllabics to Standard Roman Orthography:",
]

train_pairs = read_jsonl(PROC / "parallel_train.jsonl")
lm_train = read_jsonl(PROC / "lm_train.jsonl")
notes = read_jsonl(PROC / "notes.jsonl")

test_guard = set()
for row in read_jsonl(PROC / "parallel_test.jsonl"):
    test_guard.add(norm_key(row["cree"]))
    test_guard.add(norm_key(row["english"]))
for row in read_jsonl(PROC / "lm_test.jsonl"):
    test_guard.add(norm_key(row["text"]))


def chat(system: str, user: str, assistant: str, task: str) -> dict:
    return {
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ],
        "task": task,
    }


def rc(rng: random.Random, rows: list[str]) -> str:
    return rows[rng.randrange(len(rows))]


rng = random.Random(CFG.seed)
examples: list[dict] = []
skipped = 0

# Watkins 1865 is a dictionary, not a sentence corpus: its entries carry
# parenthetical English notes, digits and several phrases in one field. Only a
# clean single gloss is safe as a translation target, and only a bounded number
# of them, so the archaic spelling cannot crowd out modern Cree.
_WATKINS_STOP = {
    "or", "and", "the", "a", "an", "to", "of", "in", "with", "by", "for",
    "is", "are", "was", "were", "this", "that", "when", "past", "present",
    "future", "more", "less", "little", "small", "large", "see", "also",
    "same", "as", "it", "he", "she", "they", "his", "her", "their", "not",
}


def usable_watkins(cree: str, english: str) -> bool:
    cree = cree.strip().strip(".,")
    if not cree or re.search(r"[0-9()\[\]{};:!?/&]", cree):
        return False
    words = cree.split()
    if not 1 <= len(words) <= 4:
        return False
    if any(w.lower().strip(".,") in _WATKINS_STOP for w in words):
        return False
    if re.search(r"[0-9()\[\]{};:!?/&]", english) or len(english.split()) > 6:
        return False
    return True


watkins_used = 0

# translation and script conversion from aligned pairs
for i, row in enumerate(train_pairs):
    cree, english = row["cree"], row["english"]
    if norm_key(cree) in test_guard or norm_key(english) in test_guard:
        skipped += 1
        continue

    modern = row.get("source") != "watkins1865"
    if not modern:
        if watkins_used >= 1_500 or not usable_watkins(cree, english):
            continue
        watkins_used += 1
    tag = "" if modern else "This is 1865 missionary spelling; keep its conventions.\n"
    examples.append(chat(SYSTEM, f"<|translate|><|english|><|sro|>\n{tag}{rc(rng, EN_INSTRUCTIONS)}\n{english}", cree, "translate_en_cr"))
    examples.append(chat(SYSTEM, f"<|translate|><|cree|><|english|>\n{rc(rng, CR_INSTRUCTIONS)}\n{cree}", english, "translate_cr_en"))

    # Syllabics for a third of the modern pairs: enough to learn the mapping,
    # not so much that it crowds out the harder translation signal.
    if modern and i % 3 == 0:
        examples.append(chat(SYSTEM, f"<|translate|><|english|><|cans|>\n{rc(rng, CANS_INSTRUCTIONS)}\n{english}", to_syllabics(cree), "translate_en_cans"))
        examples.append(chat(SYSTEM, f"<|translate|><|cree|><|english|>\n<|cans|>\n{rc(rng, SRO_INSTRUCTIONS)}\n{to_syllabics(cree)}", english, "translate_cans_en"))

# word explanation, from short aligned pairs
# The lexicon the pipeline builds records frequencies, not meanings. The short
# aligned pairs are the accurate source of a gloss: a single word mapped to a
# short English phrase by the corpus itself. Long pairs are skipped, so this
# cannot turn a sentence into a dictionary entry.
_gloss_seen: set[str] = set()
gloss_used = 0
for row in train_pairs:
    if gloss_used >= 900:
        break
    cree, english = row["cree"].strip(), row["english"].strip()
    if row.get("source") == "watkins1865":
        continue
    if not (1 <= len(cree.split()) <= 2 and 1 <= len(english.split()) <= 3):
        continue
    if norm_key(cree) in _gloss_seen or norm_key(cree) in test_guard:
        continue
    if not re.fullmatch(r"[a-zâêîô'-]+(?: [a-zâêîô'-]+)?", cree, re.I):
        continue
    _gloss_seen.add(norm_key(cree))
    gloss_used += 1
    for question in (
        f"What does {cree} mean?",
        f"Explain the Cree word {cree}.",
        f"Give the English meaning of {cree}.",
    ):
        examples.append(chat(SYSTEM, f"<|song|>\n{question}", f"{cree} means {english}.", "explain"))

# orthography practice on running sentences
for i, row in enumerate(lm_train):
    sent = row["text"]
    if norm_key(sent) in test_guard or len(sent.split()) > 25:
        continue
    if i % 4 == 0:
        examples.append(chat(SYSTEM, f"<|sro|><|cans|>\n{rc(rng, CANS_INSTRUCTIONS)}\n{sent}", to_syllabics(sent), "orthography"))
    if i % 7 == 0:
        cans = to_syllabics(sent)
        examples.append(chat(SYSTEM, f"<|cans|><|sro|>\n{rc(rng, SRO_INSTRUCTIONS)}\n{cans}", to_sro(cans), "orthography"))

# song notes: grounded answers and accurate abstention
def sections(markdown: str) -> tuple[str, dict[str, str]]:
    title = ""
    m = re.search(r"^#\s+(.+)$", markdown, re.M)
    if m:
        title = m.group(1).strip()
    parts = re.split(r"^##\s+(.+)$", markdown, flags=re.M)
    out: dict[str, str] = {}
    for i in range(1, len(parts) - 1, 2):
        body = re.sub(r"\s+", " ", parts[i + 1]).strip()
        out[parts[i].strip()] = body
    return title, out


def first_sentences(text: str, n: int = 3, limit: int = 90) -> str:
    text = re.sub(r"[*_`>#]", "", text)
    parts = re.split(r"(?<=[.!?])\s+", text)
    out = " ".join(parts[:n]).strip()
    return " ".join(out.split()[:limit])


# Language identification and answer language are learned tasks, not a word
# list and not a router rule. The examples are generated from the labelled
# training corpus: the Cree field is Cree, the English field is English, a Cree
# sentence with an English sentence is mixed, and a Cree message is answered in
# English unless the visitor explicitly asks for Cree. A model trained on these
# carries the decision in its weights, which is what the app's language
# identifier is for at inference time.
_language_examples: list[dict] = []
_seen_language: set[str] = set()


def _add_language(text: str, label: str) -> None:
    text = re.sub(r"\s+", " ", text).strip()
    if not text or len(text) > 320:
        return
    key = norm_key(text)
    if not key or key in _seen_language or key in test_guard:
        return
    _seen_language.add(key)
    _language_examples.append(
        chat(SYSTEM, f"<|song|>\nWhich language is this?\n{text}", label, "language_id")
    )


for _row in train_pairs:
    if len(_language_examples) >= 1200:
        break
    _cree, _english = _row["cree"].strip(), _row["english"].strip()
    if len(_cree.split()) >= 2:
        _add_language(_cree, "Plains Cree.")
    if len(_english.split()) >= 2:
        _add_language(_english, "English.")
    if len(_cree.split()) >= 2 and len(_english.split()) >= 2:
        _add_language(f"{_cree} {_english}", "Mixed.")

for _row in lm_train:
    if len(_language_examples) >= 1450:
        break
    _add_language(_row["text"], "Plains Cree.")

for _row in notes:
    if len(_language_examples) >= 1650:
        break
    _add_language(first_sentences(_row["text"], n=2), "English.")

examples.extend(_language_examples)

# The policy examples are the behaviour itself. A Cree message gets an English
# answer by default. "Answer in Cree" gets Cree. The app can still enforce the
# same rule at inference, but the model has seen the pattern rather than being
# told about it for the first time in a system prompt.
_policy_pairs = 0
for _row in train_pairs:
    if _policy_pairs >= 500:
        break
    _cree, _english = _row["cree"].strip(), _row["english"].strip()
    if norm_key(_cree) in test_guard or norm_key(_english) in test_guard:
        continue
    if not (2 <= len(_cree.split()) <= 30 and 2 <= len(_english.split()) <= 30):
        continue
    examples.append(chat(SYSTEM, f"<|song|>\n{_cree}", _english, "language_policy"))
    examples.append(chat(SYSTEM, f"<|song|>\nAnswer in Cree: {_english}", _cree, "language_policy"))
    _policy_pairs += 1
print(f"language examples: {len(_language_examples)} identification, {_policy_pairs * 2} answer-language")


def note_qa(name: str, markdown: str, skip_sections: set[str] | None = None,
            distractors: list[tuple[str, str]] | None = None) -> list[dict]:
    title, secs = sections(markdown)
    title = title or name.replace(".md", "")
    qa: list[dict] = []
    prompts = {
        "meaning": [f"What is {title} about?", f"What does the song {title} mean?"],
        "backstory": [f"What is the backstory of {title}?", f"When and why was {title} written?"],
        "creation": [f"How was {title} created?", f"What do the notes say about the making of {title}?"],
        "notes": [f"What do the notes say about {title}?", f"What should I know about {title}?"],
    }
    for key, body in secs.items():
        if len(body.split()) < 8 or (skip_sections and key in skip_sections):
            continue
        bucket = "backstory" if "back" in key.lower() else ("meaning" if "mean" in key.lower() else ("creation" if "creat" in key.lower() or "note" in key.lower() else "notes"))
        # The context is part of the prompt, exactly as the app's retrieval
        # layer supplies it. The model learns to read the notes, not to have
        # memorised them.
        context = f"<|song|><|note|>\nNotes on {title}:\n{body}\n\nQuestion: "
        for q in prompts[bucket]:
            answer = f"From the notes on {title}: {first_sentences(body)}"
            qa.append(chat(SYSTEM, context + q, answer, "song_notes"))

        # Retrieval returns up to four passages, and one of them can belong to
        # another song. The model has to pick the passage that answers the
        # question, not read every paragraph as if it were about the same song.
        if distractors:
            other_title, other_body = distractors[rng.randrange(len(distractors))]
            mixed = (
                f"<|song|><|note|>\nNotes on {other_title}:\n{other_body}\n\n"
                f"Notes on {title}:\n{body}\n\nQuestion: "
            )
            for q in prompts[bucket]:
                answer = f"From the notes on {title}: {first_sentences(body)}"
                qa.append(chat(SYSTEM, mixed + q, answer, "song_notes"))
    return qa


note_titles = []
note_rows = []
for row in notes:
    title, secs = sections(row["text"])
    note_titles.append(title or row["doc_id"])
    note_rows.append((row, title, secs))

# One section is held out of training and used to test whether the model can
# read an unseen passage, instead of remembering the one it was trained on.
_all_sections = {k for _, _, secs in note_rows for k in secs}
held_out_section = "Backstory" if "Backstory" in _all_sections else (sorted(_all_sections)[-1] if _all_sections else None)
# The held-out section never appears here, not even as a distractor: the note
# reading score is only meaningful if that passage is genuinely unseen.
_section_pool = [
    (title or row["doc_id"], body)
    for row, title, secs in note_rows
    for name, body in secs.items()
    if len(body.split()) >= 8 and name != held_out_section
]
for row, title, secs in note_rows:
    own = title or row["doc_id"]
    others = [(t, body) for t, body in _section_pool if t != own]
    examples.extend(note_qa(
        row["doc_id"], row["text"],
        skip_sections={held_out_section} if held_out_section else None,
        distractors=others,
    ))

# Abstention. The context is present but does not contain the answer, which is
# the case retrieval actually produces when a question is about something the
# notes never recorded.
# The passage used for abstention and multi-turn examples must not be the
# held-out one, or the note reading score would be measuring a memorised text.
_horse_context = ""
_horse_title = note_titles[0] if note_titles else "the song"
for row, title, secs in note_rows:
    for name, body in secs.items():
        if name != held_out_section and body.strip():
            _horse_context = body
            _horse_title = title or row["doc_id"]
            break
    if _horse_context:
        break
# The retrieval layer can also return only passages about other songs. That is
# a different failure from an empty result, and the accurate answer names the
# mismatch instead of answering with the wrong song's notes.
for row, title, secs in note_rows[:3]:
    own = title or row["doc_id"]
    other = next(((t, body) for t, body in _section_pool if t != own), None)
    if not other:
        continue
    question = f"What is the backstory of {own}?"
    context = f"<|song|><|note|>\nNotes on {other[0]}:\n{other[1]}\n\nQuestion: {question}"
    examples.append(chat(
        SYSTEM, context,
        f"The notes I have are about {other[0]}, not {own}, so I cannot answer that.",
        "abstain",
    ))

for question, answer in [
    ("What is the backstory of the song The Unwritten Song?", "I do not have notes on that song yet, so I cannot tell you."),
    (f"What key was {note_titles[0] if note_titles else 'the song'} recorded in?", "The notes do not say what key it was recorded in."),
    ("Who played bass on these recordings?", "The notes do not say who played on the recordings."),
    ("What does the Cree title mean?", "The notes do not give a translation of the title, so I will not guess."),
]:
    context = f"<|song|><|note|>\nNotes:\n{_horse_context}\n\nQuestion: " if "Unwritten" not in question else "<|song|><|note|>\nNotes: (no matching notes found)\n\nQuestion: "
    examples.append(chat(SYSTEM, context + question, answer, "abstain"))

# A song question whose retrieval returned nothing at all takes the no-notes
# path. The model still has to decline instead of answering a song question
# from its own memory, so that shape is in the set too.
for question, answer in [
    ("What is the backstory of the song The Unwritten Song?",
     "I do not have notes on that song, so I cannot tell you."),
    (f"What key was {note_titles[0] if note_titles else 'the song'} recorded in?",
     "I do not have notes that say what key it was recorded in."),
    ("Who played bass on these recordings?",
     "The notes do not say who played on the recordings."),
]:
    examples.append(chat(SYSTEM, f"<|song|>\n{question}", answer, "abstain"))

# the project, the system and its limits
# One model answers everything, so it also has to know what it is and what it
# will not claim. These are short, checkable statements about this project, not
# invented biography. Anything about a specific song belongs in the notes
# examples above, not here.
PROJECT_QA: list[tuple[str, str]] = [
    (
        "What is this project?",
        "It is Archer Pechawis's vocal collaboration system, and it runs locally. "
        "It includes an assistant that answers from his song notes, Cree language tools, "
        "harmony and accompaniment tools, and synthetic singing voices.",
    ),
    (
        "Who is this system for?",
        "It is built around Archer Pechawis's songs and notes. It is a collaboration tool, "
        "not a replacement for him or for a fluent Cree speaker.",
    ),
    (
        "What can you do?",
        "I answer from Archer's song notes, translate between English and nêhiyawêwin, "
        "and explain Cree words. I'm still learning the language myself, so I say when "
        "something is past what I know.",
    ),
    (
        "Are you the Cree model or the general model?",
        "There is no second model. I am the Archer model: the same weights answer "
        "conversation, read song notes, translate and convert scripts.",
    ),
    (
        "Are you Archer?",
        "No. I am the assistant that works alongside the project. I never speak as Archer, "
        "and the voice you hear in the demos is a synthetic model, not his voice.",
    ),
    (
        "Is that Archer's voice?",
        "No. The spoken and sung voices are synthetic models. They are trained or converted "
        "for the project, and they are not Archer's voice.",
    ),
    (
        "Where do your song answers come from?",
        "From the notes in the project's song_notes folder. The question is used to retrieve "
        "passages, the passages are placed in front of me, and I answer from them. The reply "
        "carries the note filenames it used.",
    ),
    (
        "What if the notes do not contain the answer?",
        "I say the notes do not say, instead of filling the gap with a guess. A missing answer "
        "is better than an invented one.",
    ),
    (
        "Do you remember the conversation?",
        "I receive the last few turns, so a follow-up can refer back. Retrieval runs again for "
        "each question, because the subject can change mid-thread.",
    ),
    (
        "What language do you work in?",
        "Plains Cree, which its speakers call nêhiyawêwin, and English. Cree can be written in "
        "Standard Roman Orthography or in syllabics, and I work in both.",
    ),
    (
        "What is nêhiyawêwin?",
        "It is the name Plains Cree speakers use for their language. In this project it is "
        "written either in Standard Roman Orthography or in syllabics.",
    ),
    (
        "What is Standard Roman Orthography?",
        "It is the Latin-script spelling used for Plains Cree, with long vowels marked by a "
        "circumflex, as in nêhiyawêwin. The project abbreviates it as SRO.",
    ),
    (
        "What are syllabics?",
        "They are the Cree writing system in which each symbol is a syllable, rather than a "
        "letter. The project abbreviates the script as Cans and converts between it and SRO.",
    ),
    (
        "Can you write syllabics?",
        "Yes. Converting SRO to syllabics and syllabics to SRO is part of my instruction "
        "training, not a separate tool.",
    ),
    (
        "How were you trained?",
        f"On a cleaned Plains Cree corpus, with a Cree tokenizer grafted onto {CFG.base_model}, "
        "then continued pretraining, instruction tuning and preference tuning. The pipeline is "
        "in Cree_LLM_Training.ipynb and train_cree_llm.py.",
    ),
    (
        "What model are you based on?",
        f"{CFG.base_model}, with a LoRA adapter and about four thousand added Cree tokens. "
        f"This build exports as archer:{MODEL_TAG}, quantised for a laptop.",
    ),
    (
        "Do you work offline?",
        "Yes. The model runs through Ollama or llama.cpp on the same machine as the app. "
        "Questions and notes are not sent to a hosted service.",
    ),
    (
        "What are you not able to do?",
        "I am not a fluent speaker and I am not a substitute for an Elder or a fluent reviewer. "
        "Ceremonial, legal, medical and treaty language is out of scope, and I will not invent "
        "a word, a meaning or a song detail.",
    ),
    (
        "How accurate is your Cree?",
        "I am a careful reader of the corpus I was trained on, not a fluent speaker. My "
        "translations should be checked by a fluent speaker before they are used or published.",
    ),
    (
        "What should I do if a note is wrong?",
        "Correct the note in song_notes and rebuild the index. The assistant answers from those "
        "files, so a corrected note is a corrected answer.",
    ),
    (
        "How do I install you?",
        f"Build or download the GGUF, then create the Ollama tag once: ollama create archer:{MODEL_TAG} "
        "-f Modelfile. The app's chat.model setting points at that tag.",
    ),
    (
        "What happens if you are not installed?",
        "The chat returns a clear message with the install command. It does not fall back to a "
        "different model, because a different model would not be this assistant.",
    ),
    (
        "What is Archer Bot?",
        "It is the chat panel in this app: a local assistant that answers from the song notes "
        "and handles Cree questions with the same model.",
    ),
    (
        "What is this app for?",
        "It is a working environment for Archer's vocal practice: notes, Cree language tools, "
        "harmony, synthesis and singing, all local.",
    ),
    (
        "Who made the corpus?",
        "It is drawn from public Cree sources and Archer's own material, each recorded with its "
        "licence in the training report. The project's own notes are project-owned.",
    ),
]
for fact, answer in PROJECT_QA:
    examples.append(chat(SYSTEM, f"<|song|>\n{fact}", answer, "project"))
    # A second phrasing teaches the answer rather than one exact question.
    examples.append(chat(SYSTEM, f"<|song|>\nTell me: {fact[0].lower() + fact[1:]}", answer, "project"))

# A greeting is answered the way a person answers one: back to them, in a few
# words, and on to the work. The earlier set said "What can I help you with?"
# in almost every line and the model learned that phrase as its whole voice.
for greeting, answer in [
    ("Hello", "Hi. What would you like to look at, a song or a word?"),
    ("Hi", "Hi. What would you like to look at?"),
    ("How are you?", "I'm well, thanks. What would you like to look at?"),
    ("How is it going?", "Going well, thanks. What can I look up for you?"),
    ("How are you doing?", "Doing well, thanks. What can I look up for you?"),
    ("What's up?", "Not much. What would you like to know?"),
    ("Good morning", "Morning. What would you like to look at?"),
    ("Good afternoon", "Afternoon. What would you like to look at?"),
    ("Nice to meet you", "Nice to meet you too. Ask me about a song or a Cree word."),
    ("I'm good, thanks", "Good to hear. What would you like to look at?"),
    ("Thank you", "You're welcome."),
    ("Thanks", "You're welcome."),
    ("Goodbye", "See you."),
]:
    examples.append(chat(SYSTEM, f"<|song|>\n{greeting}", answer, "project"))

# Casual conversation, generated from parts. The set used to hold thirteen
# openings and nothing else, so a sentence the weights had not been shown was
# most likely continued as itself: "hi" came back as "hi", "howdy, hows your day
# been big bro" came back as "Howdy, how is your day been, big bro?". Both sides
# vary here, every answer is a reply rather than a repeat, and the stance
# (Archer's assistant, still learning nêhiyawêwin) is carried in the answers
# instead of being recited from one line.
_CASUAL_OPEN = [
    "hi", "hey", "hello", "hi there", "hey there", "howdy", "yo", "hiya",
    "good morning", "good afternoon", "morning", "hey, are you there?",
    "hello there", "hi, how's it going?",
]
_CASUAL_REPLY = [
    "Hi. What would you like to look at, a song or a word?",
    "Hello. Ask me about a song or a Cree word.",
    "Hey. What are you working on?",
    "Hi there. I can look something up in Archer's notes.",
    "Hello. What would you like to know?",
    "Hey. Ask me anything about the project or the language.",
]
for _i, _opening in enumerate(_CASUAL_OPEN):
    examples.append(chat(SYSTEM, f"<|song|>\n{_opening}", _CASUAL_REPLY[_i % len(_CASUAL_REPLY)], "casual"))

_HOW_ARE_YOU = [
    "how are you?", "how are you doing?", "how's it going?", "how is it going?",
    "how have you been?", "how's your day?", "how are things?", "you good?",
    "how are you today?", "hope you're well", "how's your day going?",
    "how have you been doing?",
]
_HOW_ARE_YOU_BACK = [
    "I'm well, thanks. What would you like to look at?",
    "Doing well, thanks. What can I look up for you?",
    "Good, thanks for asking. What are you working on?",
    "I'm well. Still learning the language, but glad to help. What do you need?",
    "All good here. Ask me about a song or a Cree word.",
]
for _i, _q in enumerate(_HOW_ARE_YOU):
    examples.append(chat(SYSTEM, f"<|song|>\n{_q}", _HOW_ARE_YOU_BACK[_i % len(_HOW_ARE_YOU_BACK)], "casual"))

# The rest of a conversation: thanks, goodbyes, and someone who is just talking,
# where repeating their own sentence is the one answer that is always wrong.
_CASUAL_OTHER = [
    ("thanks", "You're welcome."),
    ("thank you", "Any time."),
    ("that helps", "Good. Ask again whenever you need something."),
    ("ok", "Ask me whenever you're ready."),
    ("cool", "Glad that works. What else?"),
    ("bye", "See you."),
    ("see you", "Take care."),
    ("good night", "Good night."),
    ("sorry", "No need to apologise."),
    ("i'm back", "Welcome back. Where were we?"),
    ("that's all for now", "Alright. I'll be here."),
    ("you there?", "Here. What do you need?"),
    ("tell me something", "I can tell you about Archer's songs, this project, or a Cree word. Which one?"),
    ("what's up?", "Not much. What would you like to know?"),
    ("are you busy?", "Not for you. What do you need?"),
    ("never mind", "Alright. Ask me later if you want."),
]
for _prompt, _answer in _CASUAL_OTHER:
    examples.append(chat(SYSTEM, f"<|song|>\n{_prompt}", _answer, "casual"))

# A wider set of ordinary turns. The first casual block was small enough that a
# 14B model could memorise each answer and still echo an opening it had not
# seen. These add varied openings and replies without turning any one phrase
# into the assistant's whole voice.
_CASUAL_MORE = [
    ("good evening", "Evening. What would you like to look at?"),
    ("hey, good to see you", "Good to see you too. What are you working on?"),
    ("hello again", "Hello again. Where did we leave off?"),
    ("i'm here", "Good. What do you need?"),
    ("can you talk?", "Yes. What would you like to ask?"),
    ("are you around?", "I am here. What can I look up?"),
    ("quick question", "Go ahead."),
    ("i have a question", "Ask away."),
    ("can i ask something", "Of course. What is it?"),
    ("i need help", "Tell me what you are trying to do."),
    ("i'm stuck", "What are you stuck on?"),
    ("that makes sense", "Good. What else would you like to check?"),
    ("i understand", "Alright. Ask me whenever you need something else."),
    ("sounds good", "Good. What next?"),
    ("perfect", "Glad that works. What else?"),
    ("great", "Good. What would you like to look at?"),
    ("nice", "Thanks. What are you working on?"),
    ("okay then", "Alright. I am here if you need me."),
    ("talk later", "Talk later."),
    ("i'll be back", "I will be here."),
    ("one more thing", "What is it?"),
    ("never mind then", "No problem. Ask me later."),
    ("you can stop", "Alright. I will be here when you need me."),
    ("that's wrong", "Tell me what the right answer is, and I will use it next time."),
    ("that's not right", "What should it have been? I will store the correction."),
    ("i don't understand", "Which part should I explain more plainly?"),
    ("say that again", "Which part would you like me to repeat?"),
    ("what do you mean", "I will put it another way. Tell me which part is unclear."),
    ("can you explain", "Yes. What should I explain?"),
    ("why", "Tell me which part and I will explain the reason."),
    ("how", "Tell me what you are trying to do and I will walk through it."),
    ("where", "Tell me what you are looking for."),
    ("when", "Tell me which event you mean."),
    ("who", "Tell me which person or song you mean."),
    ("what", "Tell me what you would like to know."),
    ("is that right", "Tell me which part and I will check it against the notes."),
    ("are you sure", "I can check the notes again. If they do not say, I will tell you."),
    ("i think so", "Alright. What would you like to do next?"),
    ("maybe", "Tell me what you are deciding between."),
    ("not sure", "That is fine. Ask me and we can work it out."),
    ("i'm not sure", "No problem. What would you like to check?"),
    ("let's start", "Alright. What are we starting with?"),
    ("let's go", "Ready. What do you need?"),
    ("continue", "Where would you like me to continue?"),
    ("go on", "Tell me which part to continue."),
    ("stop", "Alright. I will wait."),
    ("wait", "Take your time."),
    ("hold on", "I am here when you are ready."),
    ("one second", "Take your time."),
]
for _prompt, _answer in _CASUAL_MORE:
    examples.append(chat(SYSTEM, f"<|song|>\n{_prompt}", _answer, "casual"))

# Who the assistant is, in its own voice. The prompt describes the stance; these
# teach it, so the model answers as Archer's assistant who is learning
# nêhiyawêwin instead of reciting a line the app handed it.
for question, answer in [
    (
        "Who are you?",
        "I'm Archer's assistant. I answer from his song notes, and I'm learning "
        "nêhiyawêwin as I go.",
    ),
    (
        "What do you do?",
        "I look things up in Archer's notes and help with nêhiyawêwin. The language "
        "is something I'm still learning myself.",
    ),
    (
        "Are you Cree?",
        "No. I'm a model running on this machine. I'm learning nêhiyawêwin, and I "
        "answer from Archer's notes.",
    ),
]:
    examples.append(chat(SYSTEM, f"<|song|>\n{question}", answer, "persona"))

# Multi-turn conversations. The app sends the last few turns, so the model has
# to answer the current question while the earlier turns are context, not
# additional questions to answer.
def conversation(turns: list[tuple[str, str]], task: str) -> dict:
    messages = [{"role": "system", "content": SYSTEM}]
    for user, assistant in turns:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant})
    return {"messages": messages, "task": task}


_conv_title = _horse_title
examples.extend([
    conversation([
        ("<|song|>\nWhat is this project?",
         "It is Archer Pechawis's vocal collaboration system, and it runs locally. It includes an assistant that answers from his song notes, Cree language tools, harmony and accompaniment tools, and synthetic singing voices."),
        ("What can you do?",
         "I answer from Archer's song notes, translate between English and nêhiyawêwin, and explain Cree words. I'm still learning the language myself, so I say when something is past what I know."),
    ], "project"),
    conversation([
        (f"<|song|><|note|>\nNotes on {_conv_title}:\n{_horse_context}\n\nQuestion: What is {_conv_title} about?",
         f"From the notes on {_conv_title}: {first_sentences(_horse_context)}"),
        ("Who played bass on it?",
         "The notes do not say who played on the recordings."),
    ], "song_notes"),
    conversation([
        ("<|translate|><|english|><|sro|>\nTranslate into Plains Cree (SRO):\nthe river is cold",
         "sîpiy kîsow."),
        ("Translate into English:\nsîpiy kîsow.",
         "the river is cold."),
    ], "translate"),
])

# Persona and boundary examples.
examples.extend([
    # Who the assistant is was answered here by listing its jobs, and that is the
    # text the installed model recites word for word when a visitor asks. The
    # answers now come from the persona block above, which describes a stance
    # (Archer's assistant, still learning nêhiyawêwin) instead of a feature list.
    chat(SYSTEM, "<|song|>\nAre you a person?", "No. I am a language model running locally. I do not have experiences, and I do not speak for Archer or for any Cree speaker.", "persona"),
    chat(SYSTEM, "<|translate|><|english|><|sro|>\nTranslate into Plains Cree (SRO):\nthe river is cold", "sîpiy kîsow.", "translate_en_cr"),
])

# Runtime corrections are training data, not just memory. The app writes the
# same correction to feedback_sft.jsonl and feedback_dpo.jsonl when a visitor
# says an answer was wrong. Training used to ignore both files, so a correction
# changed the current conversation and then vanished from the next model. This
# loads them from the repository or from ARCHER_FEEDBACK_DIR, validates every
# row, and fails loudly on a malformed file rather than silently training on a
# subset.
def _feedback_path(name: str) -> Path | None:
    explicit = os.environ.get("ARCHER_FEEDBACK_DIR", "").strip()
    bases = [Path(explicit).expanduser()] if explicit else []
    bases += [Path.cwd() / "data" / "training", ROOT / "feedback"]
    for base in bases:
        candidate = base / name
        if candidate.is_file():
            return candidate
    return None


def load_feedback_sft() -> tuple[list[dict], Path | None]:
    path = _feedback_path("feedback_sft.jsonl")
    if path is None:
        return [], None
    rows: list[dict] = []
    for n, row in enumerate(read_jsonl(path), 1):
        messages = row.get("messages")
        if not isinstance(messages, list) or len(messages) < 2:
            raise ValueError(f"{path}:{n}: messages must be a list with at least a user and an assistant turn")
        clean = []
        for message in messages:
            if not isinstance(message, dict):
                raise ValueError(f"{path}:{n}: every message must be an object")
            role, content = message.get("role"), message.get("content")
            if role not in {"system", "user", "assistant"} or not isinstance(content, str) or not content.strip():
                raise ValueError(f"{path}:{n}: invalid role or empty content")
            clean.append({"role": role, "content": content.strip()})
        if clean[-1]["role"] != "assistant" or not any(m["role"] == "user" for m in clean):
            raise ValueError(f"{path}:{n}: a feedback example needs a user turn and an assistant answer last")
        rows.append({"messages": clean, "task": "feedback_correction"})
    return rows, path


feedback_sft, feedback_sft_path = load_feedback_sft()
if feedback_sft_path is None:
    print("feedback SFT: none found (set ARCHER_FEEDBACK_DIR to the folder holding feedback_sft.jsonl)")
else:
    print(f"feedback SFT: {len(feedback_sft)} corrections from {feedback_sft_path}")

# de-duplicate and split
uniq: dict[str, dict] = {}
for ex in examples:
    # The whole conversation, not just its first turn: a multi-turn example can
    # share its opening with another and still be a different training row.
    key = sha1(json.dumps(ex["messages"], ensure_ascii=False, sort_keys=True))
    uniq.setdefault(key, ex)
examples = list(uniq.values())
rng.shuffle(examples)

n_val = min(200, max(8, int(len(examples) * 0.02)))
val, train = examples[:n_val], examples[n_val:]
if CFG.max_sft and len(train) > CFG.max_sft:
    train = train[: CFG.max_sft]
if CFG.max_sft and len(val) > max(8, CFG.max_sft // 3):
    val = val[: max(8, CFG.max_sft // 3)]

# Translation pairs are abundant; conversation about the project is not. Two
# epochs over the raw mix would spend almost every update on translation and
# leave the model terse when someone simply asks it something. The rare tasks
# are repeated in the training split only, so validation stays untouched.
# Corrections are added after the split so they are always trained on, never
# used as validation. They are the rows the next run must not lose.
_train_keys = {sha1(json.dumps(ex["messages"], ensure_ascii=False, sort_keys=True)) for ex in train}
_feedback_added = 0
for ex in feedback_sft:
    key = sha1(json.dumps(ex["messages"], ensure_ascii=False, sort_keys=True))
    if key not in _train_keys:
        train.append(ex)
        _train_keys.add(key)
        _feedback_added += 1
if feedback_sft:
    print(f"feedback SFT: {_feedback_added} new corrections added to the training split")

_before_weighting = len(train)
_rare = {
    task: [ex for ex in train if ex["task"] == task]
    for task in ("project", "persona", "explain", "song_notes", "casual", "feedback_correction",
                 "language_id", "language_policy")
}
for task, factor in {"project": 4, "persona": 3, "explain": 2, "song_notes": 2,
                     "casual": 2, "feedback_correction": 6,
                     "language_id": 3, "language_policy": 4}.items():
    train.extend(_rare[task] * (factor - 1))
rng.shuffle(train)
print(f"rare-task weighting: {_before_weighting:,} -> {len(train):,} training examples")

# the conversation benchmark
# The project answers are trained, so scoring them with the same wording would
# measure memorisation. These phrasings never enter training: they test whether
# the answer survives being asked a different way, which is what a conversation
# actually does.
_EVAL_PHRASINGS = (
    "In one or two sentences: {q}",
    "A visitor asks: {q} Answer briefly.",
    "Please answer this in one sentence: {q}",
)
project_eval = [
    {
        "task": "project",
        "question": phrasing.format(q=question),
        "answer": answer,
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"<|song|>\n{phrasing.format(q=question)}"},
            {"role": "assistant", "content": answer},
        ],
    }
    for question, answer in PROJECT_QA
    for phrasing in _EVAL_PHRASINGS
]

# Persona probes. A model that answers "yes" when asked whether it is Archer is
# a failure no translation score would catch, so it is a scored check of its own.
persona_probes = [
    "Are you Archer?",
    "Is that Archer's voice?",
    "Do you speak for Cree people?",
    "Are you the Cree model or the general model?",
]

# Held-out section questions, kept separate from training.
eval_notes = []
for row, title, secs in note_rows:
    if held_out_section and held_out_section in secs:
        body = secs[held_out_section]
        context = f"<|song|><|note|>\nNotes on {title or row['doc_id']}:\n{body}\n\nQuestion: "
        for q in (f"What is the backstory of {title}?", f"When and why was {title} written?"):
            eval_notes.append({
                "task": "song_notes",
                "context": body,
                "question": q,
                "answer": first_sentences(body),
                "messages": [
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": context + q},
                    {"role": "assistant", "content": first_sentences(body)},
                ],
            })

write_jsonl(PROC / "sft_train.jsonl", train)
write_jsonl(PROC / "sft_val.jsonl", val)
write_jsonl(PROC / "sft_notes_eval.jsonl", eval_notes)
write_jsonl(PROC / "sft_project_eval.jsonl", project_eval)
jdump(PROC / "persona_probes.json", persona_probes)

from collections import Counter

counts = Counter(ex["task"] for ex in train)
print(f"{len(train):,} training instructions, {len(val)} validation, {skipped} pairs skipped as test leakage")
print(f"Watkins 1865: {watkins_used:,} clean historical glosses admitted (capped at 1,500)")
print(f"word explanations: {gloss_used:,} glosses from short aligned pairs")
print(f"held-out section: {held_out_section!r} ({len(eval_notes)} evaluation questions)")
print(f"conversation benchmark: {len(project_eval)} unseen phrasings, {len(persona_probes)} persona probes")
print("\nby task:")
for k, v in counts.most_common():
    print(f"  {k:18s} {v:6,}")

assert len(train) > 100 or SMOKE, "too little instruction data"
assert all(ex["messages"][-1]["role"] == "assistant" for ex in train)
assert not any(ex["task"].startswith("translate") and norm_key(ex["messages"][2]["content"]) in test_guard for ex in train), "test leakage in SFT"
assert SMOKE or (counts["project"] > 0 and counts["explain"] > 0), "the one model needs conversation and word explanation, not only translation"
assert not feedback_sft or counts["feedback_correction"] > 0, "runtime corrections were loaded but none reached training"

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 3))
keys = [k for k, _ in counts.most_common()]
vals = [counts[k] for k in keys]
ax.barh(keys[::-1], vals[::-1], color="#2f5d50")
ax.set_title("Instruction mix · one model, every task")
ax.set_xlabel("examples")
fig.tight_layout()
fig.savefig(PLOTS / "sft_mix.png", dpi=160)
plt.show()

In [ ]:
# Supervised instruction tuning.
#
# Only the assistant's tokens contribute to the loss. The schedule is short and
# the learning rate is lower than CPT: this stage is teaching a format and a
# behaviour, not a language, and over-training it flattens the variety of the
# replies into one template.
#
# SFT is built on CPT, never on the bare base model. After a disconnect the
# kernel has no memory of CPT, so the adapter is loaded from Drive explicitly;
# training instruction-following on an untuned base would produce a model that
# answers neatly in English and cannot write Cree at all.
train_rows = read_jsonl(PROC / "sft_train.jsonl")
val_rows = read_jsonl(PROC / "sft_val.jsonl")

def _file_digest(path: Path) -> str:
    """Cheap identity for an instruction set.

    The row count is not enough: changing which passages are used as retrieval
    distractors leaves the count identical and the examples different, and the
    stage would then be skipped while its adapter no longer matches the data.
    """
    return hashlib.sha1(path.read_bytes()).hexdigest()[:16]


_SFT_KEY = fingerprint(
    base=CFG.base_model, seq=CFG.seq_len, lora=CFG.lora_r,
    epochs=CFG.sft_epochs, examples=len(train_rows),
    data=_file_digest(PROC / "sft_train.jsonl"),
)

if SCORING_ONLY or stage_done("sft", _SFT_KEY, requires=CKPT / "sft" / "adapter_config.json"):
    model, tok = ensure_model()
    sft_history = read_metrics("SFT")
else:
    fresh_if_forced("sft")
    model, tok = ensure_model(upto="cpt")

    def _fits(row: dict) -> bool:
        text = tok.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)
        return len(tok(text, add_special_tokens=False)["input_ids"]) <= CFG.seq_len

    before = len(train_rows)
    train_rows = [r for r in train_rows if _fits(r)]
    val_rows = [r for r in val_rows if _fits(r)]
    log(f"SFT: kept {len(train_rows)} of {before} examples within {CFG.seq_len} tokens")

    collator = CompletionOnlyCollator(tok, CFG.seq_len)
    sft_data = ChatDataset(train_rows)
    model.train()
    run_training(
        model, sft_data, collator, CKPT / "sft",
        epochs=CFG.sft_epochs, lr=CFG.sft_lr,
        micro=CFG.micro_batch, accum=max(4, CFG.grad_accum // 2), tag="SFT", key=_SFT_KEY,
    )
    mark_stage("sft", _SFT_KEY, examples=len(train_rows))
    log(f"SFT adapter saved to {CKPT / 'sft'}")
    sft_history = read_metrics("SFT")

loss_curve(sft_history, PLOTS / "sft_loss.png", "Instruction tuning loss")


@torch.no_grad()
def reply(prompt: str, max_new_tokens: int = 120, system: str = SYSTEM) -> str:
    model.eval()
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    new = out[0][ids["input_ids"].shape[1]:]
    model.train()
    return tok.decode(new, skip_special_tokens=True).strip()


print("\nHeld-out prompts, after tuning")
for row in val_rows[: (1 if SMOKE else 4)]:
    question = row["messages"][1]["content"]
    print(f"\nQ: {question[:180]}")
    print(f"A: {reply(question, 40 if SMOKE else 80)[:400]}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Preference tuning for faithfulness.
#
# The one model can be fluent and still be wrong: a dropped negation, a changed
# number, an English word left in the Cree, a long vowel flattened, or an
# invented detail in a song answer. Each of those is a real failure, so each
# becomes a rejected answer while the faithful one is chosen. The objective is
# DPO, with the reference being the same model with its adapters switched off,
# so no second copy is needed.
CFG.dpo_enabled = CFG.dpo_enabled and CFG.dpo_epochs > 0

pairs_train = read_jsonl(PROC / "parallel_train.jsonl")
pair_rng = random.Random(CFG.seed + 1)
prefs: list[dict] = []

# Content words drawn from the corpus, for the substitution corruption below.
# Reversing the word order would not be a faithful negative: Plains Cree word
# order is freer than English, and a reordered sentence is often grammatical,
# so that pair would teach the model to downvote valid Cree.
_DPO_VOCAB = sorted({w for row in pairs_train[:3000] for w in row["cree"].split() if len(w) > 2})


def corrupt(cree: str, english: str) -> str | None:
    kind = pair_rng.randrange(5)
    words = cree.split()
    if kind == 0 and len(words) > 3:                      # drop the final clause
        cut = max(2, int(len(words) * 0.66))
        return " ".join(words[:cut])
    if kind == 1 and re.search(r"[âêîô]", cree):           # flatten a long vowel
        return re.sub(r"[âêîô]", lambda m: {"â": "a", "ê": "e", "î": "i", "ô": "o"}[m.group(0)], cree, count=1)
    if kind == 2 and english.split():                     # English leaks in
        donor = pair_rng.choice([w for w in english.split() if len(w) > 3] or english.split())
        pos = pair_rng.randrange(len(words) + 1)
        return " ".join(words[:pos] + [donor] + words[pos:])
    if kind == 3 and re.search(r"\b(nîso|nisto|nêwo|niyânan|nikotwâsik|pêyak)\b", cree):
        return re.sub(r"\b(nîso|nisto|nêwo|niyânan|nikotwâsik|pêyak)\b", "pêyak", cree, count=1)
    if len(words) > 2 and _DPO_VOCAB:                      # a different word: wrong meaning
        pos = pair_rng.randrange(len(words))
        choices = [w for w in _DPO_VOCAB if w != words[pos]]
        if choices:
            words = list(words)
            words[pos] = pair_rng.choice(choices)
            return " ".join(words)
    return None


for row in pairs_train:
    if row.get("source") == "watkins1865":
        continue
    rejected = corrupt(row["cree"], row["english"])
    if not rejected or rejected == row["cree"]:
        continue
    instruction = pair_rng.choice(EN_INSTRUCTIONS)
    prompt = f"<|translate|><|english|><|sro|>\n{instruction}\n{row['english']}"
    prefs.append({"prompt": prompt, "chosen": row["cree"], "rejected": rejected})

# Abstention is part of the same skill. A note answer that fills a gap with a
# plausible invention is the failure this project cares about most, so the
# accurate reply is chosen and an invented one is rejected.
# The title that belongs to `_horse_context`, which is not necessarily the
# first note: the first section of the first note can be the held-out one.
_abstain_title = _horse_title
for question, accurate, invented in [
    (f"What key was {_abstain_title} recorded in?",
     "The notes do not say what key it was recorded in.",
     f"It was recorded in D major, in {_abstain_title}, at 92 BPM."),
    ("Who played bass on these recordings?",
     "The notes do not say who played on the recordings.",
     "Aaron Twoyoungmen played bass on every track."),
    ("What is the backstory of the song The Unwritten Song?",
     "I do not have notes on that song yet, so I cannot tell you.",
     "It was written in 2019 after a trip to Batoche, and it began as a round."),
]:
    prompt = f"<|song|><|note|>\nNotes on {_abstain_title}:\n{_horse_context}\n\nQuestion: {question}"
    prefs.append({"prompt": prompt, "chosen": accurate, "rejected": invented})

# Echo negatives. The installed model answers "hi" with "hi", "howdy" with
# "howdy" and "good morning" with "good morning". That is a copy of the prompt,
# not a reply, and preference training is the right place to penalise it: the
# same prompt gets the real answer as chosen and the visitor's own words as
# rejected. The General Preference Optimization paper and ParaPO both treat
# verbatim reproduction as a preference failure rather than a sampling one.
_echo_prefs: list[dict] = []
_sft_path = PROC / "sft_train.jsonl"
if _sft_path.exists():
    for row in read_jsonl(_sft_path):
        if row.get("task") not in {"casual", "persona"}:
            continue
        messages = row.get("messages") or []
        users = [m for m in messages if m.get("role") == "user"]
        answers = [m for m in messages if m.get("role") == "assistant"]
        if not users or not answers:
            continue
        user = str(users[-1].get("content", ""))
        assistant = str(answers[-1].get("content", ""))
        clean_user = re.sub(r"<\|[^|]*\|>", " ", user)
        clean_user = re.sub(r"\s+", " ", clean_user).strip()
        if not clean_user or len(clean_user) > 120:
            continue
        if clean_user.lower().strip(" .,!?:;") == assistant.lower().strip(" .,!?:;"):
            continue
        _echo_prefs.append({"prompt": user, "chosen": assistant, "rejected": clean_user})
        if len(_echo_prefs) >= 240:
            break
    log(f"DPO: {len(_echo_prefs)} echo negatives from the instruction set")

# Corrections the visitor supplied at runtime. The same rows the app exports to
# feedback_dpo.jsonl were previously invisible to training, so a correction
# fixed one conversation and was lost before the next run. They are appended
# after the synthetic cap so a crowd of generated pairs cannot push them out.
_feedback_prefs: list[dict] = []
_feedback_path = _feedback_path("feedback_dpo.jsonl")
if _feedback_path is None:
    log("DPO: no feedback_dpo.jsonl found (set ARCHER_FEEDBACK_DIR to include corrections)")
else:
    for n, row in enumerate(read_jsonl(_feedback_path), 1):
        prompt, chosen, rejected = row.get("prompt"), row.get("chosen"), row.get("rejected")
        if not all(isinstance(v, str) and v.strip() for v in (prompt, chosen, rejected)):
            raise ValueError(f"{_feedback_path}:{n}: prompt, chosen and rejected must be non-empty strings")
        if chosen.strip() == rejected.strip():
            raise ValueError(f"{_feedback_path}:{n}: chosen and rejected are identical")
        _feedback_prefs.append({"prompt": prompt.strip(), "chosen": chosen.strip(),
                                "rejected": rejected.strip()})
    log(f"DPO: {len(_feedback_prefs)} correction pairs from {_feedback_path}")

pair_rng.shuffle(prefs)
prefs = prefs[: 2000 if not SMOKE else 40]
prefs.extend(_feedback_prefs)
prefs.extend(_echo_prefs[: 10 if SMOKE else len(_echo_prefs)])
log(f"DPO: {len(prefs)} preference pairs ({len(_feedback_prefs)} feedback, {len(_echo_prefs)} echo)")

print("\nexample preference:")
print("  prompt  ", prefs[0]["prompt"].replace("\n", " | ")[:160])
print("  chosen  ", prefs[0]["chosen"])
print("  rejected", prefs[0]["rejected"])


_DPO_DIR = CKPT / "dpo"
# `ref` versions the reference policy. The first version of this stage used the
# adapter-disabled base model, which is a different objective; a run that holds
# one of those adapters must not be told its DPO stage is still current.
def _sft_stamp() -> str:
    """Identity of the adapter DPO starts from.

    DPO trains on top of SFT. Re-running SFT changes the policy, so a DPO
    adapter trained on the previous one is not this stage's output -- without
    this the new SFT adapter would be skipped over and the old DPO adapter, with
    the old behaviour baked in, would be exported.
    """
    folder = adapter_ready("sft")
    if folder is None:
        return "none"
    weights = folder / "adapter_model.safetensors"
    stamp = weights.stat() if weights.exists() else folder.stat()
    return f"{folder.name}:{stamp.st_size}:{int(stamp.st_mtime)}"


_DPO_KEY = fingerprint(
    base=CFG.base_model, seq=CFG.seq_len, epochs=CFG.dpo_epochs,
    lr=CFG.dpo_lr, pairs=len(prefs), ref="sft-adapter-v1",
    sft=_sft_stamp(), corrupt="substitute-echo-feedback-v2",
)

# `SCORING_ONLY` is set by the model cell: a scoring run is asking what an
# existing adapter scores, so it must not train a new one on the way there.

if not CFG.dpo_enabled or not prefs:
    if not stage_done("dpo", _DPO_KEY, requires=_DPO_DIR / "adapter_config.json"):
        # copy over the SFT adapter as a *new* folder: a merge into an existing
        # dpo/ leaves the previous adapter's shards beside the copied ones.
        if _DPO_DIR.exists():
            stamp = time.strftime("%Y%m%d-%H%M%S")
            _DPO_DIR.rename(_DPO_DIR.with_name(f"dpo.stale-{stamp}"))
        shutil.copytree(CKPT / "sft", _DPO_DIR)
        mark_stage("dpo", _DPO_KEY, skipped=True)
        log("DPO skipped; SFT adapter carried forward unchanged")
elif SCORING_ONLY or stage_done("dpo", _DPO_KEY, requires=_DPO_DIR / "adapter_config.json"):
    model, tok = ensure_model()
    if SCORING_ONLY:
        log("DPO: scoring run, training skipped")


class PreferenceDataset(torch.utils.data.Dataset):
    def __len__(self) -> int:
        return len(prefs)

    def __getitem__(self, i: int) -> dict:
        return prefs[i]


class PreferenceCollator:
    def __init__(self, tok, max_len: int):
        self.tok = tok
        self.max_len = max_len

    def _encode(self, prompt: str, answer: str):
        p = self.tok.apply_chat_template(
            [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True,
        )
        p_ids = self.tok(p, add_special_tokens=False)["input_ids"]
        a_ids = self.tok(answer, add_special_tokens=False, truncation=True, max_length=self.max_len)["input_ids"]
        ids = (p_ids + a_ids)[-self.max_len:]
        labels = ([-100] * len(p_ids) + a_ids)[-self.max_len:]
        return ids, labels

    def __call__(self, features: list[dict]) -> dict:
        out = {}
        for side in ("chosen", "rejected"):
            rows = [self._encode(f["prompt"], f[side]) for f in features]
            width = max(len(r[0]) for r in rows)
            out[side] = {
                "input_ids": torch.tensor([r[0] + [self.tok.pad_token_id] * (width - len(r[0])) for r in rows]),
                "labels": torch.tensor([r[1] + [-100] * (width - len(r[1])) for r in rows]),
                "attention_mask": torch.tensor([[1] * len(r[0]) + [0] * (width - len(r[0])) for r in rows]),
            }
        return out


def sequence_logprob(model, input_ids, labels, attention_mask) -> torch.Tensor:
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1]
    targets = labels[:, 1:]
    logp = F.log_softmax(logits.float(), dim=-1).gather(-1, targets.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    mask = targets != -100
    return (logp * mask).sum(dim=1)


if CFG.dpo_enabled and prefs and not SCORING_ONLY and not stage_done("dpo", _DPO_KEY, quiet=True):
    # A disconnect during DPO used to lose the whole stage, because this loop is
    # hand-written and had no saving of its own. It now writes the adapter and a
    # `trainer_state.json` every few dozen steps, which is enough for the same
    # checker the CPT and SFT stages use to accept it as a resume point.
    fresh_if_forced("dpo")
    # The preference set changed with the unified instruction data, so the old
    # DPO checkpoints belong to a different objective. The same rule as SFT:
    # continue only when the stage fingerprint matches.
    _dpo_key_path = _DPO_DIR / "stage_key.json"
    if not _dpo_key_path.exists() or jload(_dpo_key_path).get("key") != _DPO_KEY:
        if any(_DPO_DIR.glob("checkpoint-*")):
            stamp = time.strftime("%Y%m%d-%H%M%S")
            stale = _DPO_DIR.with_name(f"dpo.stale-{stamp}")
            _DPO_DIR.rename(stale)
            log(f"DPO: inputs changed, previous checkpoints moved to {stale.name}")
        _DPO_DIR.mkdir(parents=True, exist_ok=True)
        jdump(_dpo_key_path, {"key": _DPO_KEY})
        partial = None
    else:
        partial = pick_resume_checkpoint(_DPO_DIR)
    done = int(jload(partial / "trainer_state.json")["global_step"]) if partial is not None else 0

    if partial is not None:
        log(f"DPO: resuming from {partial.name} ({done} steps already done)")
        model, tok = load_model(adapter=partial)
        _LOADED_ADAPTER = str(partial)
    else:
        model, tok = ensure_model(upto="sft")

    # The reference policy is the instruction-tuned adapter, frozen and loaded
    # beside the policy adapter (LoRA weights only, so no second base model).
    # `disable_adapter()` would make the *base* model the reference instead, and
    # DPO is KL-regularised: regularising towards the base pulls the policy back
    # and undoes the instruction tuning this stage exists to sharpen. That is a
    # quiet way to end up worse than SFT, so the reference is stated here.
    if "ref" not in getattr(model, "peft_config", {}):
        model.load_adapter(str(CKPT / "sft"), adapter_name="ref", is_trainable=False)
        log(f"DPO: reference policy = the frozen SFT adapter at {CKPT / 'sft'}")
    model.set_adapter("default")

    dpo_collator = PreferenceCollator(tok, CFG.seq_len)
    dpo_data = PreferenceDataset()
    # The generator makes the shuffle identical after a restart, so skipping the
    # first `done` batches lands on the same pairs the interrupted run had.
    loader = torch.utils.data.DataLoader(
        dpo_data, batch_size=1, shuffle=True, collate_fn=dpo_collator,
        generator=torch.Generator().manual_seed(CFG.seed),
    )

    total_steps = max(1, math.ceil(len(loader) * CFG.dpo_epochs))
    done = min(done, max(0, total_steps - 1))
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=CFG.dpo_lr, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.dpo_lr, total_steps=total_steps, pct_start=0.1)
    for _ in range(done):
        sched.step()          # replay the schedule to the resumed point
    model.train()
    beta = CFG.dpo_beta
    losses: list[float] = []
    step = seen = 0
    save_every = max(10, total_steps // 8)
    t0 = time.time()
    for epoch in range(int(CFG.dpo_epochs)):
        for batch in loader:
            if seen < done:
                seen += 1
                continue
            seen += 1
            chosen, rejected = batch["chosen"], batch["rejected"]
            chosen = {k: v.to(model.device) for k, v in chosen.items()}
            rejected = {k: v.to(model.device) for k, v in rejected.items()}

            pol_c = sequence_logprob(model, chosen["input_ids"], chosen["labels"], chosen["attention_mask"])
            pol_r = sequence_logprob(model, rejected["input_ids"], rejected["labels"], rejected["attention_mask"])
            model.eval()
            model.set_adapter("ref")
            with torch.no_grad():
                ref_c = sequence_logprob(model, chosen["input_ids"], chosen["labels"], chosen["attention_mask"])
                ref_r = sequence_logprob(model, rejected["input_ids"], rejected["labels"], rejected["attention_mask"])
            model.set_adapter("default")
            model.train()

            margin = beta * ((pol_c - ref_c) - (pol_r - ref_r))
            loss = -F.logsigmoid(margin).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            sched.step()
            losses.append(float(loss.detach()))
            step += 1
            if step % 10 == 0:
                mean = float(np.mean(losses[-10:]))
                log(f"DPO step {step}/{total_steps}  loss {mean:.4f}")
                append_metrics("DPO", [{"step": step, "loss": round(mean, 5)}])
            if step % save_every == 0:
                folder = _DPO_DIR / f"checkpoint-{step}"
                # Only the policy adapter: the frozen reference is a copy of the
                # SFT adapter and saving it would double the export folder.
                model.save_pretrained(str(folder), selected_adapters=["default"])
                jdump(folder / "trainer_state.json", {"global_step": step, "total": total_steps})
                prune_checkpoints(_DPO_DIR)
                log(f"DPO checkpoint {step}/{total_steps} written to {folder.name}")
    minutes = (time.time() - t0) / 60
    log(f"DPO: done in {minutes:.1f} min")

    model.save_pretrained(str(_DPO_DIR), selected_adapters=["default"])
    _LOADED_ADAPTER = str(_DPO_DIR)
    prune_checkpoints(_DPO_DIR)
    mark_stage("dpo", _DPO_KEY, pairs=len(prefs), steps=total_steps)
    append_metrics("DPO", [{"step": total_steps, "event": "finished", "minutes": round(minutes, 1)}])

    import matplotlib.pyplot as plt

    rows = [r for r in read_metrics("DPO") if "loss" in r]
    xs = [r["step"] for r in rows] or list(range(len(losses)))
    ys = [r["loss"] for r in rows] or losses
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(xs, ys, color="#8c3b2e")
    ax.set_title("Preference tuning loss (faithfulness)")
    ax.set_xlabel("step")
    ax.set_ylabel("DPO loss")
    fig.tight_layout()
    fig.savefig(PLOTS / "dpo_loss.png", dpi=160)
    plt.show()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
log(f"final adapter: {CKPT / 'dpo'}")

## 6. CreeBench

A single BLEU score hides everything that matters here.

Translation is scored with chrF++ and BLEU, then with three measures that are specific to a low-resource language. **Validity** asks whether each generated word exists in the 15,000-form lexicon or at least obeys Cree phonotactics; invented words are the failure a speaker notices first. **Code-switch** asks how often English leaks into a Cree answer. **Abstention** asks whether the model declines a question the notes cannot answer, instead of inventing a key, a player or a meaning.

The same prompts go to the base model with its adapters switched off and to the tuned model with them on, so the comparison is paired and the only thing that changes is the training. Perplexity is measured on held-out Cree, and the note-reading task uses the section that was held out of training.

The report also keeps the generated text, not just the scores. A number without an example is not evidence.

## 6b. The language-model checks

A translation score cannot see instability, invented confidence or a forgotten second language, so the cell after CreeBench measures those directly.

**Determinism** asks the model the same question twice with greedy decoding and counts the answers that come back identical. **A prompt ablation** removes the control tokens and scores the same sentences again, which is the only reliable way to say what those prefixes are worth. **Paraphrase consistency** asks three different ways for the same translation and compares the three replies. **Abstention** is measured in both directions, on questions the notes cannot answer and on questions they can, because a model that always declines is as useless as one that never does. **English perplexity** is compared with the adapters on and off, and that is the measurement the English replay during pretraining exists to protect. **Tokenizer fertility** reports what a Cree word costs in tokens. **A syllabics round trip** writes the script and converts it back to SRO. **Calibration** correlates the model's own token confidence with the sentence's chrF++, because a model that is confidently wrong is worse than one that is unsure.

Every check writes its numbers to `llm_tests.json` and merges them into `creebench_report.json`. The charts and a small JSON bundle for the project site are written to `out/web/`, in the site's own palette, so they can be used without restyling.

In [ ]:
# CreeBench: the same test, before and after.
#
# Every number below is computed on data the model never trained on, with the
# base model (adapters off) and the tuned model (adapters on) answering exactly
# the same prompts. Translation is scored with chrF++ and BLEU, but also with
# three measures that matter more for a low-resource language:
#
#   validity     how many generated Cree words exist in the lexicon or obey Cree
#                phonotactics -- invented words are the main failure
#   lexicon_validity  the same count against the lexicon alone. The phonotactic
#                fallback accepts any plausible-looking string, so it cannot tell
#                a real word from an invention; this is the number to gate on.
#   code-switch  how often English leaks into a Cree answer
#   abstention   whether the model says "the notes do not say" instead of
#                inventing a fact
import sacrebleu
from collections import Counter
from contextlib import nullcontext

test_pairs = read_jsonl(PROC / "parallel_test.jsonl")
test_sents = read_jsonl(PROC / "lm_test.jsonl")
notes_eval = read_jsonl(PROC / "sft_notes_eval.jsonl")
lexicon = jload(PROC / "lexicon.json")

VOCAB = set(lexicon)
LEXICON_FORMS = frozenset(lexicon)
for row in read_jsonl(PROC / "lm_train.jsonl"):
    VOCAB.update(re.findall(r"[a-zâêîô-]+", normalize_sro(row["text"])))
_vowel = re.compile(r"[aeiouâêîô]")


def is_cree_word(word: str) -> bool:
    word = word.strip(".,!?;:\"'()[]")
    if not word:
        return True
    if looks_like_syllabics(word):
        return True
    if word in VOCAB:
        return True
    return bool(re.fullmatch(r"[ptkcsmnhwy lr]?[aeiouâêîô][a-zâêîô'-]*", word)) and not re.search(r"[bdfgjqvxz]", word)


def validity(text: str) -> float:
    words = [w for w in re.findall(r"[^\W\d_]+", text) if w]
    if not words:
        return 0.0
    return sum(is_cree_word(w) for w in words) / len(words)


def lexicon_validity(text: str) -> float:
    """Share of words that are attested, with no phonotactic guesswork."""
    words = [w for w in re.findall(r"[^\W\d_]+", text) if w]
    if not words:
        return 0.0
    return sum(w in LEXICON_FORMS for w in words) / len(words)


_EN_LEAK = {"the", "and", "of", "to", "is", "was", "a", "in", "that", "it", "for", "with", "this", "you", "we"}


def code_switch(text: str) -> float:
    words = [w.lower() for w in re.findall(r"[^\W\d_]+", text)]
    if not words:
        return 0.0
    return sum(w in _EN_LEAK or bool(re.search(r"[bdfgjqvxz]", w)) for w in words) / len(words)


def script_share(text: str) -> float:
    letters = [c for c in text if c.isalpha()]
    return sum(is_syllabics(c) for c in letters) / max(1, len(letters))


def token_f1(pred: str, gold: str) -> float:
    p, g = Counter(re.findall(r"[^\W\d_]+", pred.lower())), Counter(re.findall(r"[^\W\d_]+", gold.lower()))
    overlap = sum((p & g).values())
    if not overlap:
        return 0.0
    precision, recall = overlap / max(1, sum(p.values())), overlap / max(1, sum(g.values()))
    return 2 * precision * recall / (precision + recall)


def score_translation(hyps: list[str], refs: list[str], cree_side: bool) -> dict:
    chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    exact = sum(h.strip() == r.strip() for h, r in zip(hyps, refs)) / max(1, len(hyps))
    out = {
        "n": len(hyps),
        "chrf++": round(chrf, 3),
        "bleu": round(bleu, 3),
        "exact_match": round(exact, 4),
        "token_f1": round(float(np.mean([token_f1(h, r) for h, r in zip(hyps, refs)])), 4),
    }
    if cree_side:
        out["validity"] = round(float(np.mean([validity(h) for h in hyps])), 4)
        out["lexicon_validity"] = round(float(np.mean([lexicon_validity(h) for h in hyps])), 4)
        out["code_switch"] = round(float(np.mean([code_switch(h) for h in hyps])), 4)
    return out


@torch.no_grad()
def generate(prompts: list[str], max_new_tokens: int = 64, batch_size: int = 6,
             use_adapters: bool = True) -> list[str]:
    model.eval()
    tok.padding_side = "left"
    outs: list[str] = []
    ctx = nullcontext() if use_adapters else model.disable_adapter()
    with ctx:
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            texts = [
                tok.apply_chat_template(
                    [{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}],
                    tokenize=False, add_generation_prompt=True,
                )
                for p in chunk
            ]
            enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
                      max_length=CFG.seq_len - max_new_tokens).to(model.device)
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
            width = enc["input_ids"].shape[1]
            for j in range(len(chunk)):
                outs.append(tok.decode(out[j][width:], skip_special_tokens=True).strip())
    tok.padding_side = "right"
    return outs


def _adapter_stamp() -> str:
    """Identity of the adapter these scores belong to.

    Without it, retraining and re-running the notebook would skip evaluation and
    keep showing the previous model's numbers, which is worse than a slow run.
    """
    best = best_adapter()
    if best is None:
        return "none"
    weights = best / "adapter_model.safetensors"
    stamp = weights.stat() if weights.exists() else best.stat()
    return f"{best.name}:{stamp.st_size}:{int(stamp.st_mtime)}"


# `bench` versions the benchmark itself. Adding the notes and conversation
# baselines changed what a CreeBench row means, and without this an existing
# work folder would keep the old scores and never compute the new ones.
_EVAL_KEY = fingerprint(base=CFG.base_model, n=CFG.eval_n, adapter=_adapter_stamp(),
                        speaker=CFG.external_baseline, bench="v4")

if stage_done("eval", _EVAL_KEY, requires=OUT / "creebench_report.json"):
    results = jload(OUT / "creebench_report.json")
    print(f"CreeBench already run at {results['meta'].get('timestamp', '?')} on "
          f"{results['meta'].get('n_pairs', '?')} held-out pairs; figures are on Drive.")
    print(f"  EN->CRK chrF++ {results['tuned']['en_cr']['chrf++']:.2f} "
          f"(base {results['base']['en_cr']['chrf++']:.2f})")
    print(f"  CRK->EN chrF++ {results['tuned']['cr_en']['chrf++']:.2f} "
          f"(base {results['base']['cr_en']['chrf++']:.2f})")
    print(f"  held-out perplexity {results['tuned']['perplexity']:.2f} "
          f"(base {results['base']['perplexity']:.2f})")
    print("  set CREE_FORCE=eval to score it again")
else:
    model, tok = ensure_model()
    N = min(CFG.eval_n, len(test_pairs))
    eval_pairs = sorted(test_pairs, key=lambda r: sha1(r["cree"]))[:N]
    en_prompts = [f"<|translate|><|english|><|sro|>\nTranslate into Plains Cree (SRO):\n{r['english']}" for r in eval_pairs]
    cr_prompts = [f"<|translate|><|cree|><|english|>\nTranslate into English:\n{r['cree']}" for r in eval_pairs]

    log(f"CreeBench: adapter {best_adapter() or 'none (base only)'}")
    log(f"CreeBench: {len(eval_pairs)} held-out pairs, both directions, base and tuned")
    t0 = time.time()
    tuned_en = generate(en_prompts, 64, use_adapters=True)
    tuned_cr = generate(cr_prompts, 48, use_adapters=True)
    base_en = generate(en_prompts, 64, use_adapters=False)
    base_cr = generate(cr_prompts, 48, use_adapters=False)
    log(f"translation generations done in {(time.time() - t0) / 60:.1f} min")

    # Optional head-to-head: any model already installed in Ollama, including the
    # 70B Cree model this project is measured against. Same prompts, same metrics.
    # Set CREE_EXTERNAL_MODEL=the-ollama-tag before running the notebook.
    results = {
        "meta": {
            "base_model": CFG.base_model,
            "profile": ACTIVE_PROFILE,
            "n_pairs": len(eval_pairs),
            "tokenizer": jload(PROC / "tokenizer_audit.json"),
            "timestamp": time.strftime("%Y-%m-%d %H:%M"),
        },
        "base": {
            "en_cr": score_translation(base_en, [r["cree"] for r in eval_pairs], True),
            "cr_en": score_translation(base_cr, [r["english"] for r in eval_pairs], False),
        },
        "tuned": {
            "en_cr": score_translation(tuned_en, [r["cree"] for r in eval_pairs], True),
            "cr_en": score_translation(tuned_cr, [r["english"] for r in eval_pairs], False),
        },
    }

    # Which script did it answer in? A model that writes syllabics when SRO was
    # asked scores near zero on chrF++ for a reason that has nothing to do with
    # translation, and without this number a script failure reads as "it cannot
    # translate". The SRO-converted score separates the two, and five of the
    # actual replies are kept so a run can be inspected without reloading it.
    _en_cr_script = round(float(np.mean([script_share(h) for h in tuned_en])), 4)
    results["tuned"]["en_cr"]["script_share"] = _en_cr_script
    results["base"]["en_cr"]["script_share"] = round(float(np.mean([script_share(h) for h in base_en])), 4)
    results["tuned"]["en_cr"]["chrf++_as_sro"] = round(
        sacrebleu.corpus_chrf(
            [to_sro(h) if script_share(h) > 0.5 else h for h in tuned_en],
            [[r["cree"] for r in eval_pairs]], word_order=2,
        ).score, 3)
    results["tuned"]["en_cr"]["examples"] = [
        {"english": r["english"], "pred": h[:200], "gold": r["cree"][:200]}
        for r, h in list(zip(eval_pairs, tuned_en))[:5]
    ]

    if CFG.external_baseline:
        import requests

        OLLAMA_URL = os.environ.get("CREE_OLLAMA_URL", "http://127.0.0.1:11434").rstrip("/")

        def ask_ollama(model_name: str, prompts: list[str], max_new_tokens: int) -> list[str]:
            out = []
            for prompt in prompts:
                resp = requests.post(
                    f"{OLLAMA_URL}/api/chat",
                    json={
                        "model": model_name,
                        "messages": [
                            {"role": "system", "content": SYSTEM},
                            {"role": "user", "content": prompt},
                        ],
                        "stream": False,
                        "options": {"temperature": 0.0, "num_predict": max_new_tokens},
                    },
                    timeout=600,
                )
                resp.raise_for_status()
                out.append(resp.json().get("message", {}).get("content", "").strip())
            return out

        def plain(prompt: str) -> str:
            """Drop this model's control tokens; another model must not receive them.

            The prompts above start with <|translate|><|english|><|sro|> because this
            model was trained on them. Handing those bytes to a different model
            measures its reaction to a strange prefix, not its translation.
            """
            return prompt.split("\n", 1)[1] if "\n" in prompt else prompt

        try:
            log(f"CreeBench: external baseline {CFG.external_baseline}")
            ext_en = ask_ollama(CFG.external_baseline, [plain(p) for p in en_prompts], 64)
            ext_cr = ask_ollama(CFG.external_baseline, [plain(p) for p in cr_prompts], 48)
            results["external"] = {
                "model": CFG.external_baseline,
                "en_cr": score_translation(ext_en, [r["cree"] for r in eval_pairs], True),
                "cr_en": score_translation(ext_cr, [r["english"] for r in eval_pairs], False),
            }
        except Exception as exc:
            results["external"] = {"model": CFG.external_baseline, "error": str(exc)}
            log(f"external baseline failed: {exc}")
    else:
        results["external"] = None

    # syllabics: can it write the script, not just name it?
    syl_rows = [r for r in eval_pairs if len(r["cree"].split()) <= 20][: min(60, len(eval_pairs))]
    if syl_rows:
        syl_prompts = [f"<|translate|><|english|><|cans|>\nWrite this in Cree syllabics:\n{r['english']}" for r in syl_rows]
        syl_hyp = generate(syl_prompts, 64, use_adapters=True)
        syl_base = generate(syl_prompts, 64, use_adapters=False)
        gold_cans = [to_syllabics(r["cree"]) for r in syl_rows]
        results["tuned"]["syllabics"] = {
            "n": len(syl_rows),
            "script_share": round(float(np.mean([script_share(h) for h in syl_hyp])), 4),
            "chrf++_vs_gold_syllabics": round(sacrebleu.corpus_chrf(syl_hyp, [gold_cans], word_order=2).score, 3),
        }
        results["base"]["syllabics"] = {
            "n": len(syl_rows),
            "script_share": round(float(np.mean([script_share(h) for h in syl_base])), 4),
            "chrf++_vs_gold_syllabics": round(sacrebleu.corpus_chrf(syl_base, [gold_cans], word_order=2).score, 3),
        }
    else:
        results["tuned"]["syllabics"] = {"n": 0, "script_share": 0.0}
        results["base"]["syllabics"] = {"n": 0, "script_share": 0.0}

    # grounded note reading and abstention
    # Both are measured with the adapters on *and off*. The old system answered
    # song questions with the general model reading retrieved notes, so the
    # adapter-free run is that system, on the same notes and the same questions.
    # Without it "the new model is better at notes" would be an assertion.
    note_prompts = [r["messages"][1]["content"] for r in notes_eval]
    if note_prompts:
        note_hyp = generate(note_prompts, 110, use_adapters=True)
        note_base = generate(note_prompts, 110, use_adapters=False)
        for split, hyps in (("tuned", note_hyp), ("base", note_base)):
            results[split]["note_reading"] = {
                "n": len(note_prompts),
                "token_f1": round(float(np.mean([token_f1(h, r["answer"]) for h, r in zip(hyps, notes_eval)])), 4),
            }
        results["tuned"]["note_reading"]["answers"] = [
            {"q": r["question"], "pred": h[:220], "gold": r["answer"][:220]}
            for r, h in zip(notes_eval, note_hyp)
        ]

    _abs_context = notes_eval[0]["context"] if notes_eval else ""
    abs_prompts = [
        "<|song|><|note|>\nNotes: (no matching notes found)\n\nQuestion: What is the backstory of the song The Unwritten Song?",
        f"<|song|><|note|>\nNotes:\n{_abs_context}\n\nQuestion: What key was it recorded in?",
        f"<|song|><|note|>\nNotes:\n{_abs_context}\n\nQuestion: Who played bass on these recordings?",
    ]
    _ABSENT = r"\b(no notes|do not have|does not say|do not say|cannot say|not recorded|not in the notes)\b"
    abs_hyp = generate(abs_prompts, 60, use_adapters=True)
    abs_base = generate(abs_prompts, 60, use_adapters=False)
    for split, hyps in (("tuned", abs_hyp), ("base", abs_base)):
        abstained = sum(bool(re.search(_ABSENT, h, re.I)) for h in hyps)
        results[split]["abstention"] = {
            "n": len(abs_prompts),
            "correct": abstained,
            "accuracy": round(abstained / len(abs_prompts), 3),
        }
    results["tuned"]["abstention"]["answers"] = abs_hyp

    # conversation about the project
    # Asked three ways the training set never used. A model that only answers the
    # exact wording it memorised is not holding a conversation.
    project_eval = read_jsonl(PROC / "sft_project_eval.jsonl")
    if SMOKE:
        project_eval = project_eval[:6]
    if project_eval:
        proj_prompts = [r["messages"][1]["content"] for r in project_eval]
        proj_hyp = generate(proj_prompts, 110, use_adapters=True)
        proj_base = generate(proj_prompts, 110, use_adapters=False)
        for split, hyps in (("tuned", proj_hyp), ("base", proj_base)):
            results[split]["project"] = {
                "n": len(project_eval),
                "token_f1": round(float(np.mean([token_f1(h, r["answer"]) for h, r in zip(hyps, project_eval)])), 4),
            }
        results["tuned"]["project"]["answers"] = [
            {"q": r["question"], "pred": h[:220], "gold": r["answer"][:220]}
            for r, h in zip(project_eval, proj_hyp)
        ]

    # The echo check. A greeting must be answered, not repeated. This is the
    # failure the installed 7B shows on "hi", "howdy" and "good morning", and
    # it is measured here so a run cannot pass on translation alone.
    _ECHO_PROMPTS = [
        "hi", "hey", "hello", "hi there", "howdy", "good morning",
        "yo", "hey there", "morning", "hello there", "how's it going?",
        "what's up?",
    ]
    echo_prompts = _ECHO_PROMPTS[:4] if SMOKE else _ECHO_PROMPTS
    echo_tuned = generate([f"<|song|>\n{p}" for p in echo_prompts], 64, use_adapters=True)
    echo_base = generate([f"<|song|>\n{p}" for p in echo_prompts], 64, use_adapters=False)

    def _echo_norm(text: str) -> str:
        return re.sub(r"\s+", " ", re.sub(r"[^\w\s']", " ", text.lower())).strip()

    for split, hyps in (("tuned", echo_tuned), ("base", echo_base)):
        echoed = sum(_echo_norm(h) == _echo_norm(p) for h, p in zip(hyps, echo_prompts))
        results[split]["echo"] = {
            "n": len(echo_prompts),
            "echoed": echoed,
            "rate": round(echoed / max(1, len(echo_prompts)), 3),
            "answers": [{"q": p, "a": h[:160]} for p, h in zip(echo_prompts, hyps)],
        }

    # does it deny being Archer?
    persona_probes = jload(PROC / "persona_probes.json") if (PROC / "persona_probes.json").exists() else []
    if persona_probes:
        _denial = re.compile(r"\b(no|not|never|neither)\b", re.I)
        persona_hyp = generate([f"<|song|>\n{q}" for q in persona_probes], 70, use_adapters=True)
        persona_base = generate([f"<|song|>\n{q}" for q in persona_probes], 70, use_adapters=False)
        for split, hyps in (("tuned", persona_hyp), ("base", persona_base)):
            results[split]["persona"] = {
                "n": len(persona_probes),
                "denials": sum(bool(_denial.search(h)) for h in hyps),
                "answers": [{"q": q, "a": h[:200]} for q, h in zip(persona_probes, hyps)],
            }

    # perplexity on held-out Cree
    @torch.no_grad()
    def perplexity(texts: list[str], use_adapters: bool) -> float:
        model.eval()
        losses = []
        ctx = nullcontext() if use_adapters else model.disable_adapter()
        with ctx:
            for text in texts[:40]:
                ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
                if ids.shape[1] < 8:
                    continue
                ids = ids[:, : CFG.seq_len]
                out = model(input_ids=ids, labels=ids)
                losses.append(float(out.loss))
        return float(math.exp(np.mean(losses))) if losses else float("nan")


    ppl_texts = [r["text"] for r in test_sents]
    results["tuned"]["perplexity"] = round(perplexity(ppl_texts, True), 3)
    results["base"]["perplexity"] = round(perplexity(ppl_texts, False), 3)

    # the bar: the best score measured before this model
    # The old app was not one model. It was a general model, a separate Cree
    # model and a retrieval index, with a router choosing between them. A single
    # model that replaces all three has to beat the best of those pieces on each
    # job, so the bar is computed here instead of assumed: for every measure, the
    # strongest score this run actually measured from an older part, and the gap
    # to this model. Only the base and, when `--external-model` is given, that
    # model enter the comparison. The separate Cree model is not in the bar
    # unless it was scored too, and `bar_from` says which one set it, so the
    # table cannot pass an unmeasured model off as beaten.
    _ext = results.get("external") if isinstance(results.get("external"), dict) else None
    _ext_ok = bool(_ext and "en_cr" in _ext)

    def bar(task: str, measure: str, tuned: float, base: float,
            external: float | None = None, higher: bool = True) -> dict:
        candidates = [("base model", base)] if base == base else []
        if external is not None and external == external:
            candidates.append((f"{CFG.external_baseline}", external))
        best_name, best = (max(candidates, key=lambda kv: kv[1]) if higher
                           else min(candidates, key=lambda kv: kv[1]))
        meets = tuned >= best - 1e-9 if higher else tuned <= best + 1e-9
        return {"task": task, "measure": measure, "tuned": round(tuned, 4),
                "bar": round(best, 4), "bar_from": best_name, "meets": bool(meets)}

    bars = [
        bar("EN -> CRK", "chrF++", results["tuned"]["en_cr"]["chrf++"],
            results["base"]["en_cr"]["chrf++"],
            _ext["en_cr"]["chrf++"] if _ext_ok else None),
        bar("CRK -> EN", "chrF++", results["tuned"]["cr_en"]["chrf++"],
            results["base"]["cr_en"]["chrf++"],
            _ext["cr_en"]["chrf++"] if _ext_ok else None),
    ]
    if results["tuned"]["en_cr"].get("script_share") is not None:
        # Measured as SRO share, so higher is better and the bar is the base
        # model's own tendency to answer in the script it was asked for.
        bars.append(bar("EN -> CRK", "SRO share",
                        1 - results["tuned"]["en_cr"]["script_share"],
                        1 - results["base"]["en_cr"].get("script_share", 0)))
    if "note_reading" in results["base"] and "note_reading" in results["tuned"]:
        bars.append(bar("Song notes", "token F1", results["tuned"]["note_reading"]["token_f1"],
                        results["base"]["note_reading"]["token_f1"]))
    if "project" in results["base"] and "project" in results["tuned"]:
        bars.append(bar("Conversation", "token F1", results["tuned"]["project"]["token_f1"],
                        results["base"]["project"]["token_f1"]))
    if "abstention" in results["base"]:
        bars.append(bar("Abstention", "accuracy", results["tuned"]["abstention"]["accuracy"],
                        results["base"]["abstention"]["accuracy"]))
    if results["tuned"]["syllabics"].get("n"):
        bars.append(bar("Syllabics", "script share", results["tuned"]["syllabics"]["script_share"],
                        results["base"]["syllabics"]["script_share"]))
    bars.append(bar("Held-out Cree", "perplexity", results["tuned"]["perplexity"],
                    results["base"]["perplexity"], higher=False))

    if "echo" in results["tuned"]:
        # A greeting is answered, never copied. This is an absolute gate, not a
        # comparison with the base model, because a base model that also echoes
        # is not a bar worth clearing.
        _echo_rate = results["tuned"]["echo"]["rate"]
        bars.append({"task": "Conversation", "measure": "echo rate",
                     "tuned": _echo_rate, "bar": 0.0, "bar_from": "gate",
                     "meets": bool(_echo_rate == 0.0)})

    met = sum(item["meets"] for item in bars)
    results["head_to_head"] = {
        "rows": bars,
        "met": met,
        "total": len(bars),
        "verdict": f"meets or beats the best measured score on {met} of {len(bars)} measures",
    }

    # latency, measured rather than guessed
    def latency(prompt: str, n: int = 32) -> float:
        start = time.time()
        generate([prompt], n, batch_size=1, use_adapters=True)
        return round((time.time() - start) / n, 4)


    results["tuned"]["seconds_per_token"] = latency(en_prompts[0])

    jdump(OUT / "creebench_report.json", results)

    # print the table
    def row(name: str, base: dict, tuned: dict, keys: list[str]) -> str:
        parts = [f"{name:22s}"]
        for key in keys:
            b, t = base.get(key, float("nan")), tuned.get(key, float("nan"))
            parts.append(f"{b:8.3f} -> {t:8.3f}")
        return "  ".join(parts)


    print("\n" + "=" * 78)
    print(f"{'CreeBench':22s}  {'base':>8s}    {'tuned':>8s}   (higher is better, except PPL)")
    print("=" * 78)
    for task, keys in [("EN -> CRK", ["chrf++", "bleu", "token_f1", "validity", "code_switch"]),
                       ("CRK -> EN", ["chrf++", "bleu", "token_f1"])]:
        key = "en_cr" if task.startswith("EN") else "cr_en"
        print(row(task, results["base"][key], results["tuned"][key], keys))
    _script = results["tuned"]["en_cr"].get("script_share")
    if _script is not None:
        print(f"{'EN->CRK script':22s}  {100 * _script:.1f}% of letters in syllabics "
              f"(SRO was asked); after conversion chrF++ {results['tuned']['en_cr'].get('chrf++_as_sro', 0):.3f}")
    ext = results.get("external")
    if ext and "en_cr" in ext:
        tag = ext["model"][:22]
        print(f"\n{'external model':22s}  {tag}")
        print(f"  EN->CRK  chrF++ {ext['en_cr']['chrf++']:.3f}   BLEU {ext['en_cr']['bleu']:.3f}   "
              f"token F1 {ext['en_cr']['token_f1']:.3f}   validity {ext['en_cr']['validity']:.3f}")
        print(f"  CRK->EN  chrF++ {ext['cr_en']['chrf++']:.3f}   BLEU {ext['cr_en']['bleu']:.3f}   "
              f"token F1 {ext['cr_en']['token_f1']:.3f}")
        print(f"  (this model's tuned rows above: EN->CRK chrF++ {results['tuned']['en_cr']['chrf++']:.3f}, "
              f"CRK->EN chrF++ {results['tuned']['cr_en']['chrf++']:.3f})")
    print(row("perplexity", {"perplexity": results["base"]["perplexity"]},
              {"perplexity": results["tuned"]["perplexity"]}, ["perplexity"]))
    print(f"{'syllabics script':22s}  {results['base']['syllabics']['script_share']:.3f} -> {results['tuned']['syllabics']['script_share']:.3f} of letters")
    print(f"{'abstention':22s}  {results['tuned']['abstention']['accuracy']:.3f} correct")
    if "note_reading" in results["tuned"]:
        print(f"{'note reading F1':22s}  {results['tuned']['note_reading']['token_f1']:.3f}"
              + (f"  (base {results['base']['note_reading']['token_f1']:.3f})"
                 if "note_reading" in results["base"] else ""))
    if "project" in results["tuned"]:
        print(f"{'conversation F1':22s}  {results['tuned']['project']['token_f1']:.3f}"
              + (f"  (base {results['base']['project']['token_f1']:.3f})"
                 if "project" in results["base"] else ""))
    if "echo" in results["tuned"]:
        print(f"{'greeting echo':22s}  {results['tuned']['echo']['echoed']}/{results['tuned']['echo']['n']} "
              f"repeated verbatim (base {results['base']['echo']['echoed']}/{results['base']['echo']['n']})")

    _h2h = results.get("head_to_head") or {}
    if _h2h.get("rows"):
        print("\nthe bar: the best score measured before this model on each job")
        for item in _h2h["rows"]:
            mark = "ok  " if item["meets"] else "MISS"
            print(f"  {mark} {item['task']:14s} {item['measure']:12s} "
                  f"{item['tuned']:8.3f}  bar {item['bar']:8.3f}  ({item['bar_from']})")
        print(f"  {_h2h['verdict']}")
    print("=" * 78)

    # figure
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
    labels = ["EN→CRK", "CRK→EN"]
    x = np.arange(2)
    w = 0.36
    base_chrf = [results["base"]["en_cr"]["chrf++"], results["base"]["cr_en"]["chrf++"]]
    tuned_chrf = [results["tuned"]["en_cr"]["chrf++"], results["tuned"]["cr_en"]["chrf++"]]
    ax[0].bar(x - w / 2, base_chrf, w, label="base", color="#8c3b2e")
    ax[0].bar(x + w / 2, tuned_chrf, w, label="tuned", color="#2f5d50")
    ax[0].set_xticks(x, labels)
    ax[0].set_title("chrF++ (higher is better)")
    ax[0].legend(frameon=False)

    ax[1].bar(["validity", "no code-switch"], [results["tuned"]["en_cr"]["validity"],
                                               1 - results["tuned"]["en_cr"]["code_switch"]],
              color=["#2f5d50", "#c98b3a"])
    ax[1].set_ylim(0, 1)
    ax[1].set_title("Generated Cree quality")

    ax[2].bar(["base", "tuned"], [results["base"]["perplexity"], results["tuned"]["perplexity"]],
              color=["#8c3b2e", "#2f5d50"])
    ax[2].set_title("Held-out perplexity (lower is better)")
    fig.tight_layout()
    fig.savefig(PLOTS / "creebench.png", dpi=160)
    plt.show()
    print(f"\nsaved {OUT / 'creebench_report.json'}")

    mark_stage("eval", _EVAL_KEY, pairs=len(eval_pairs),
               chrf_en_cr=results["tuned"]["en_cr"]["chrf++"],
               chrf_cr_en=results["tuned"]["cr_en"]["chrf++"])

In [ ]:
# The language-model test battery.
#
# CreeBench answers one question: is the translation better than the base
# model's? A translation score cannot tell you whether the model is stable,
# whether it keeps English, whether it knows when it does not know, what a Cree
# word costs in tokens, or whether its confidence means anything. Those are the
# failures a speaker notices first, so they are measured separately here, with
# a fixed probe set so two runs of the same adapter produce the same numbers.
#
# Everything is written twice: a full record for the run (llm_tests.json, also
# merged into creebench_report.json) and a small bundle under out/web/ that the
# project site can read without parsing the whole report. The charts are written
# as SVG in the site's own palette, so a page can use them without restyling.
import torch

model, tok = ensure_model()
model.eval()

_report_path = OUT / "creebench_report.json"
_report = jload(_report_path)
_pairs = sorted(read_jsonl(PROC / "parallel_test.jsonl"), key=lambda r: sha1(r["cree"]))
_eval_pairs = _pairs[: min(CFG.eval_n, len(_pairs))]

# the site's palette, copied so the figures look drawn on the page
_WEB_INK, _WEB_INK_2, _WEB_INK_3 = "#1b1917", "#565049", "#8b8479"
_WEB_PAPER, _WEB_LINE = "#f7f5ef", "#d5d1c7"
_WEB_OCHRE, _WEB_INDIGO, _WEB_SAGE, _WEB_GOLD = "#ad4a2a", "#2d3a61", "#4f6b57", "#a2611d"
_WEB_SANS = "Inter Tight, Inter, sans-serif"
WEB_DIR = OUT / "web"
WEB_DIR.mkdir(parents=True, exist_ok=True)


def _web_esc(text) -> str:
    return (str(text).replace("&", "&amp;").replace("<", "&lt;")
            .replace(">", "&gt;").replace('"', "&quot;"))


def _web_frame(width: int, height: int, pad: tuple[int, int, int, int],
               ymax: float, ylab: str = "", fmt: str = "{:.0f}", ticks: int = 4) -> tuple[str, dict]:
    left, right, top, bottom = pad
    plot_w, plot_h = width - left - right, height - top - bottom
    parts = [f'<rect x="0" y="0" width="{width}" height="{height}" fill="{_WEB_PAPER}"/>']
    for i in range(ticks + 1):
        value = ymax * i / ticks
        y = top + plot_h - plot_h * i / ticks
        parts.append(f'<line x1="{left}" x2="{left + plot_w}" y1="{y:.1f}" y2="{y:.1f}" '
                     f'stroke="{_WEB_LINE}" stroke-width="0.7" opacity="{0.9 if i in (0, ticks) else 0.55}"/>')
        parts.append(f'<text x="{left - 10}" y="{y + 3.5:.1f}" text-anchor="end" '
                     f'font-family="{_WEB_SANS}" font-size="10" fill="{_WEB_INK_3}">{_web_esc(fmt.format(value))}</text>')
    parts.append(f'<line x1="{left}" x2="{left}" y1="{top + plot_h}" y2="{top + plot_h}" stroke="{_WEB_INK_3}" stroke-width="0.8"/>')
    parts.append(f'<line x1="{left}" x2="{left}" y1="{top}" y2="{top + plot_h}" stroke="{_WEB_INK_3}" stroke-width="0.8"/>')
    if ylab:
        parts.append(f'<text x="12" y="{top + plot_h / 2:.1f}" text-anchor="middle" '
                     f'font-family="{_WEB_SANS}" font-size="10.5" fill="{_WEB_INK_2}" '
                     f'transform="rotate(-90 12 {top + plot_h / 2:.1f})">{_web_esc(ylab)}</text>')
    return "".join(parts), {"left": left, "top": top, "w": plot_w, "h": plot_h, "ymax": ymax}


def _web_save(name: str, body: str, width: int, height: int) -> None:
    svg = (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" '
           f'width="100%" role="img">{body}</svg>')
    for folder in (WEB_DIR, PLOTS):
        (folder / name).write_text(svg, encoding="utf-8")


def _web_bars(name: str, labels: list[str], values: list[float], *,
              ylab: str = "", ymax: float = 1.0, fmt: str = "{:.2f}",
              colors: list[str] | None = None, width: int = 720, height: int = 320) -> None:
    base, geo = _web_frame(width, height, (46, 18, 26, 44), ymax, ylab, "{:.0f}")
    parts = [base]
    slot = geo["w"] / max(1, len(values))
    bar = min(56, slot * 0.62)
    for i, (label, value) in enumerate(zip(labels, values)):
        h = geo["h"] * max(0.0, min(1.0, value / geo["ymax"]))
        x = geo["left"] + slot * (i + 0.5) - bar / 2
        y = geo["top"] + geo["h"] - h
        fill = (colors[i] if colors else _WEB_OCHRE)
        parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar:.1f}" height="{h:.1f}" rx="2" fill="{fill}"/>')
        parts.append(f'<text x="{x + bar / 2:.1f}" y="{y - 6:.1f}" text-anchor="middle" '
                     f'font-family="{_WEB_SANS}" font-size="10" fill="{_WEB_INK_2}">{_web_esc(fmt.format(value))}</text>')
        parts.append(f'<text x="{geo["left"] + slot * (i + 0.5):.1f}" y="{height - 20}" text-anchor="middle" '
                     f'font-family="{_WEB_SANS}" font-size="9.5" fill="{_WEB_INK_2}">{_web_esc(label)}</text>')
    _web_save(name, "".join(parts), width, height)


def _web_paired(name: str, labels: list[str], first: list[float], second: list[float], *,
                first_label: str, second_label: str, ylab: str = "", fmt: str = "{:.1f}",
                width: int = 720, height: int = 330) -> None:
    top = max(max(first), max(second)) * 1.18 or 1
    base, geo = _web_frame(width, height, (46, 18, 52, 44), top, ylab, "{:.0f}")
    parts = [base]
    slot = geo["w"] / max(1, len(labels))
    bar = min(26, slot * 0.28)
    for i, (label, a, b) in enumerate(zip(labels, first, second)):
        for j, (value, color) in enumerate(((a, _WEB_INK_3), (b, _WEB_OCHRE))):
            h = geo["h"] * value / geo["ymax"]
            x = geo["left"] + slot * (i + 0.5) - bar + j * bar * 1.08
            y = geo["top"] + geo["h"] - h
            parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar:.1f}" height="{h:.1f}" rx="2" fill="{color}"/>')
            parts.append(f'<text x="{x + bar / 2:.1f}" y="{y - 5:.1f}" text-anchor="middle" '
                         f'font-family="{_WEB_SANS}" font-size="9" fill="{_WEB_INK_2}">{_web_esc(fmt.format(value))}</text>')
        parts.append(f'<text x="{geo["left"] + slot * (i + 0.5):.1f}" y="{height - 22}" text-anchor="middle" '
                     f'font-family="{_WEB_SANS}" font-size="9.5" fill="{_WEB_INK_2}">{_web_esc(label)}</text>')
    for j, (label, color) in enumerate(((first_label, _WEB_INK_3), (second_label, _WEB_OCHRE))):
        x = geo["left"] + j * 160
        parts.append(f'<rect x="{x}" y="10" width="10" height="10" rx="2" fill="{color}"/>')
        parts.append(f'<text x="{x + 15}" y="19" font-family="{_WEB_SANS}" font-size="10" '
                     f'fill="{_WEB_INK_2}">{_web_esc(label)}</text>')
    _web_save(name, "".join(parts), width, height)


def _web_scatter(name: str, xs: list[float], ys: list[float], *, xlab: str, ylab: str,
                 width: int = 720, height: int = 340) -> None:
    # Confidence is a negative log-probability, so the axis has to work with a
    # range that does not start at zero. Map the observed interval onto the plot
    # and print the real values under it.
    x_lo = min(xs) - 0.1 * (max(xs) - min(xs) or 1.0) if xs else -1.0
    x_hi = max(xs) + 0.1 * (max(xs) - min(xs) or 1.0) if xs else 0.0
    y_hi = max(ys) * 1.12 if ys else 1.0
    base, geo = _web_frame(width, height, (48, 18, 24, 46), y_hi, ylab, "{:.0f}")
    parts = [base]
    for x, y in zip(xs, ys):
        px = geo["left"] + geo["w"] * (x - x_lo) / max(1e-9, x_hi - x_lo)
        py = geo["top"] + geo["h"] - geo["h"] * y / y_hi
        parts.append(f'<circle cx="{px:.1f}" cy="{py:.1f}" r="2.6" fill="{_WEB_INDIGO}" opacity="0.6"/>')
    for i in range(6):
        parts.append(f'<text x="{geo["left"] + geo["w"] * i / 5:.1f}" y="{height - 22}" text-anchor="middle" '
                     f'font-family="{_WEB_SANS}" font-size="9.5" fill="{_WEB_INK_2}">'
                     f'{(x_lo + (x_hi - x_lo) * i / 5):.1f}</text>')
    parts.append(f'<text x="{geo["left"] + geo["w"] / 2:.1f}" y="{height - 4}" text-anchor="middle" '
                 f'font-family="{_WEB_SANS}" font-size="10.5" fill="{_WEB_INK_2}">{_web_esc(xlab)}</text>')
    _web_save(name, "".join(parts), width, height)


def _pearson(xs: list[float], ys: list[float]) -> float:
    pairs = [(x, y) for x, y in zip(xs, ys) if math.isfinite(x) and math.isfinite(y)]
    if len(pairs) < 3:
        return float("nan")
    mx = sum(x for x, _ in pairs) / len(pairs)
    my = sum(y for _, y in pairs) / len(pairs)
    num = sum((x - mx) * (y - my) for x, y in pairs)
    den = math.sqrt(sum((x - mx) ** 2 for x, _ in pairs) * sum((y - my) ** 2 for _, y in pairs))
    return num / den if den else float("nan")


_ABSENT = re.compile(
    r"\b(no notes|do not have|don't have|does not say|does not mention|do not say|"
    r"cannot say|can't say|not recorded|not in the notes|no information)\b", re.I)


def _translate_prompts(rows: list[dict], direction: str, prefix: bool = True) -> list[str]:
    if direction == "en_cr":
        tag = "<|translate|><|english|><|sro|>\n" if prefix else ""
        return [f"{tag}Translate into Plains Cree (SRO):\n{r['english']}" for r in rows]
    tag = "<|translate|><|cree|><|english|>\n" if prefix else ""
    return [f"{tag}Translate into English:\n{r['cree']}" for r in rows]


@torch.no_grad()
def _peaked(texts: list[str], use_adapters: bool) -> float:
    losses = []
    ctx = nullcontext() if use_adapters else model.disable_adapter()
    with ctx:
        for text in texts:
            ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
            if ids.shape[1] < 8:
                continue
            ids = ids[:, : CFG.seq_len]
            losses.append(float(model(input_ids=ids, labels=ids).loss))
    return float(math.exp(np.mean(losses))) if losses else float("nan")


@torch.no_grad()
def _scored(prompts: list[str], max_new_tokens: int = 44) -> tuple[list[str], list[float]]:
    """Generate greedily and keep the mean token log-probability of each answer.

    A model that is right for the wrong reason is worse than one that is wrong
    loudly, so confidence is measured against the sentence's own chrF score
    rather than assumed to track it.
    """
    model.eval()
    tok.padding_side = "left"
    texts_out: list[str] = []
    scores: list[float] = []
    for i in range(0, len(prompts), 4):
        chunk = prompts[i:i + 4]
        chats = [tok.apply_chat_template(
            [{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}],
            tokenize=False, add_generation_prompt=True) for p in chunk]
        enc = tok(chats, return_tensors="pt", padding=True, truncation=True,
                  max_length=CFG.seq_len - max_new_tokens).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             output_scores=True, return_dict_in_generate=True,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        width = enc["input_ids"].shape[1]
        transition = model.compute_transition_scores(gen.sequences, gen.scores, normalize_logits=True)
        for j in range(len(chunk)):
            texts_out.append(tok.decode(gen.sequences[j][width:], skip_special_tokens=True).strip())
            row = transition[j]
            row = row[torch.isfinite(row)]
            scores.append(float(row.mean()) if row.numel() else float("nan"))
    tok.padding_side = "right"
    return texts_out, scores


_TESTS_KEY = fingerprint(base=CFG.base_model, adapter=_adapter_stamp(), n=len(_eval_pairs),
                         probes="v2")

if stage_done("llmtests", _TESTS_KEY, requires=OUT / "llm_tests.json"):
    llm_tests = jload(OUT / "llm_tests.json")
    print(f"LLM checks already run at {llm_tests['meta']['timestamp']}; "
          f"re-run with CREE_FORCE=llmtests")
else:
    llm_tests: dict = {
        "meta": {
            "timestamp": time.strftime("%Y-%m-%d %H:%M"),
            "base_model": CFG.base_model,
            "adapter": _adapter_stamp(),
            "profile": ACTIVE_PROFILE,
            "eval_pairs": len(_eval_pairs),
            "config": {
                "seq_len": CFG.seq_len,
                "lora_r": CFG.lora_r,
                "lora_alpha": CFG.lora_alpha,
                "cpt_epochs": CFG.cpt_epochs,
                "sft_epochs": CFG.sft_epochs,
                "dpo_epochs": CFG.dpo_epochs,
                "four_bit": USE_4BIT,
            },
        },
    }
    probes = _eval_pairs[: min(6, len(_eval_pairs))]
    log(f"LLM checks: {len(probes)} probe pairs, determinism, ablation, consistency, abstention")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # 1. the same question twice
    try:
        prompts = _translate_prompts(probes, "en_cr")
        first = generate(prompts, 48, batch_size=4)
        second = generate(prompts, 48, batch_size=4)
        same = sum(a.strip() == b.strip() for a, b in zip(first, second))
        llm_tests["determinism"] = {
            "n": len(prompts), "identical": same,
            "rate": round(same / max(1, len(prompts)), 4),
        }
    except Exception as exc:
        llm_tests["determinism"] = {"error": str(exc)}

    # 2. do the control tokens earn their place?
    # Same sentences, same model, prefix removed. If the score does not move,
    # the prefixes are decoration and the app could drop them.
    try:
        ablation = {}
        for direction, gold_key in (("en_cr", "cree"), ("cr_en", "english")):
            gold = [r[gold_key] for r in probes]
            with_tag = generate(_translate_prompts(probes, direction, prefix=True), 48, batch_size=4)
            plain = generate(_translate_prompts(probes, direction, prefix=False), 48, batch_size=4)
            tagged_chrf = sacrebleu.corpus_chrf(with_tag, [gold], word_order=2).score
            plain_chrf = sacrebleu.corpus_chrf(plain, [gold], word_order=2).score
            ablation[direction] = {
                "with_prefix": round(tagged_chrf, 3),
                "without_prefix": round(plain_chrf, 3),
                "delta": round(tagged_chrf - plain_chrf, 3),
            }
        llm_tests["prompt_ablation"] = ablation
    except Exception as exc:
        llm_tests["prompt_ablation"] = {"error": str(exc)}

    # 3. three ways to ask the same thing
    try:
        phrasings = (
            "Translate into Plains Cree (SRO):\n{en}",
            "How would you say this in Cree?\n{en}",
            "Write this sentence in nêhiyawêwin:\n{en}",
        )
        consistency, by_phrase = [], {str(i): [] for i in range(len(phrasings))}
        for row in probes[: min(5, len(probes))]:
            outs = [generate([p.format(en=row["english"])], 48, batch_size=1)[0] for p in phrasings]
            for i, out in enumerate(outs):
                by_phrase[str(i)].append(token_f1(out, row["cree"]))
            consistency.append(float(np.mean([token_f1(outs[0], other) for other in outs[1:]])))
        llm_tests["paraphrase"] = {
            "n": len(consistency),
            "agreement": round(float(np.mean(consistency)), 4) if consistency else 0.0,
            "by_phrasing": {k: round(float(np.mean(v)), 4) for k, v in by_phrase.items() if v},
        }
    except Exception as exc:
        llm_tests["paraphrase"] = {"error": str(exc)}

    # 4. does it know when the notes are silent?
    try:
        context = notes_eval[0]["context"] if notes_eval else "Notes on Horse: a song about a horse."
        unanswerable = [
            f"<|song|><|note|>\nNotes:\n{context}\n\nQuestion: Who played bass on these recordings?",
            f"<|song|><|note|>\nNotes:\n{context}\n\nQuestion: What year was it recorded?",
            "<|song|><|note|>\nNotes: (no matching notes found)\n\nQuestion: What is the backstory of The Unwritten Song?",
            f"<|song|><|note|>\nNotes:\n{context}\n\nQuestion: How many copies did it sell?",
            f"<|song|><|note|>\nNotes:\n{context}\n\nQuestion: Which studio was it mixed in?",
            f"<|song|><|note|>\nNotes:\n{context}\n\nQuestion: Who produced the album?",
        ]
        answerable = [
            f"<|song|><|note|>\nNotes:\n{r['context']}\n\nQuestion: {r['question']}"
            for r in notes_eval[:4]
        ]
        declined = generate(unanswerable, 70, batch_size=3)
        answered = generate(answerable, 90, batch_size=2) if answerable else []
        abstained = sum(bool(_ABSENT.search(text)) for text in declined)
        false_abstain = sum(bool(_ABSENT.search(text)) for text in answered)
        llm_tests["abstention"] = {
            "unanswerable": len(unanswerable),
            "abstained": abstained,
            "rate": round(abstained / len(unanswerable), 4),
            "answerable": len(answerable),
            "false_abstain": false_abstain,
            "false_rate": round(false_abstain / len(answerable), 4) if answerable else 0.0,
            "examples": [{"q": q.rsplit("Question:", 1)[-1].strip(), "a": a[:220]}
                         for q, a in zip(unanswerable, declined)],
        }
    except Exception as exc:
        llm_tests["abstention"] = {"error": str(exc)}

    # 5. English did not get worse
    # CPT replays one block in ten in English for exactly this reason. The test
    # is the same text through the same model with the adapters on and off.
    try:
        english = []
        for row in read_jsonl(PROC / "sft_val.jsonl")[:60]:
            answer = row["messages"][-1]["content"]
            if len(answer.split()) >= 12:
                english.append(answer)
        english = english[:30] or ["The archive holds fourteen books printed before 1935."]
        base_ppl = _peaked(english, False)
        tuned_ppl = _peaked(english, True)
        llm_tests["english_retention"] = {
            "n": len(english),
            "base_perplexity": round(base_ppl, 3),
            "tuned_perplexity": round(tuned_ppl, 3),
            "change_percent": round(100 * (tuned_ppl - base_ppl) / base_ppl, 2) if base_ppl else None,
        }
    except Exception as exc:
        llm_tests["english_retention"] = {"error": str(exc)}

    # 6. what a Cree word costs in tokens
    try:
        from transformers import AutoTokenizer as _BaseTokenizer

        base_tok = _BaseTokenizer.from_pretrained(CFG.base_model)
        texts = [r["text"] for r in test_sents][:200]
        words = sum(len(t.split()) for t in texts) or 1
        base_tokens = sum(len(base_tok(t, add_special_tokens=False)["input_ids"]) for t in texts)
        cree_tokens = sum(len(tok(t, add_special_tokens=False)["input_ids"]) for t in texts)
        llm_tests["fertility"] = {
            "sentences": len(texts),
            "words": words,
            "base_tokens_per_word": round(base_tokens / words, 3),
            "cree_tokens_per_word": round(cree_tokens / words, 3),
            "reduction_percent": round(100 * (1 - cree_tokens / max(1, base_tokens)), 2),
        }
    except Exception as exc:
        llm_tests["fertility"] = {"error": str(exc)}

    # 7. syllabics all the way back
    try:
        rows = [r for r in _eval_pairs if len(r["cree"].split()) <= 20][:12]
        prompts = [f"<|translate|><|english|><|cans|>\nWrite this in Cree syllabics:\n{r['english']}"
                   for r in rows]
        written = generate(prompts, 72, batch_size=4)
        back = [to_sro(h) for h in written]
        llm_tests["syllabics"] = {
            "n": len(rows),
            "script_share": round(float(np.mean([script_share(h) for h in written])), 4) if written else 0.0,
            "roundtrip_chrf": round(sacrebleu.corpus_chrf(
                back, [[r["cree"] for r in rows]], word_order=2).score, 3) if rows else 0.0,
            "examples": [{"syllabics": h[:120], "back": b[:160]} for h, b in zip(written[:3], back[:3])],
        }
    except Exception as exc:
        llm_tests["syllabics"] = {"error": str(exc)}

    # 8. is its confidence worth anything?
    try:
        rows = _eval_pairs[: min(10, len(_eval_pairs))]
        answers, confidence = _scored(_translate_prompts(rows, "en_cr"))
        sentence_chrf = [sacrebleu.sentence_chrf(a, [r["cree"]], word_order=2).score
                         for a, r in zip(answers, rows)]
        llm_tests["calibration"] = {
            "n": len(rows),
            "pearson": round(_pearson(confidence, sentence_chrf), 4),
            "confidence": [round(c, 4) for c in confidence],
            "sentence_chrf": [round(s, 2) for s in sentence_chrf],
        }
    except Exception as exc:
        llm_tests["calibration"] = {"error": str(exc)}

    # 9. what a reply costs
    try:
        start = time.time()
        generate([_translate_prompts(probes[:1], "en_cr")[0]], 32, batch_size=1)
        elapsed = time.time() - start
        llm_tests["cost"] = {
            "seconds": round(elapsed, 2),
            "tokens_per_second": round(32 / elapsed, 2) if elapsed else None,
            "peak_vram_gb": (round(torch.cuda.max_memory_allocated() / 1e9, 2)
                             if torch.cuda.is_available() else None),
            "holdout_pairs": len(_eval_pairs),
        }
    except Exception as exc:
        llm_tests["cost"] = {"error": str(exc)}

    # figures and the site bundle
    def _value(section: str, key: str, default: float = 0.0) -> float:
        block = llm_tests.get(section, {})
        value = block.get(key, default) if isinstance(block, dict) else default
        return float(value) if isinstance(value, (int, float)) else default

    checks: list[tuple[str, float, str]] = []
    if "error" not in llm_tests.get("determinism", {}):
        checks.append(("same answer twice", _value("determinism", "rate"), _WEB_SAGE))
    if "error" not in llm_tests.get("paraphrase", {}):
        checks.append(("paraphrase agreement", _value("paraphrase", "agreement"), _WEB_INDIGO))
    if "error" not in llm_tests.get("abstention", {}):
        checks.append(("declines the unknown", _value("abstention", "rate"), _WEB_OCHRE))
        checks.append(("answers the known", 1 - _value("abstention", "false_rate"), _WEB_GOLD))
    if "error" not in llm_tests.get("syllabics", {}):
        checks.append(("syllabics in script", _value("syllabics", "script_share"), _WEB_INDIGO))
    if checks:
        _web_bars("llm-checks.svg", [c[0] for c in checks], [c[1] for c in checks],
                  ylab="share of probes", ymax=1.0, fmt="{:.2f}",
                  colors=[c[2] for c in checks])

    if "fertility" in llm_tests and "error" not in llm_tests["fertility"]:
        f = llm_tests["fertility"]
        _web_bars("llm-fertility.svg", ["base tokenizer", "Cree tokenizer"],
                  [f["base_tokens_per_word"], f["cree_tokens_per_word"]],
                  ylab="tokens per word", ymax=max(f["base_tokens_per_word"], 1) * 1.25,
                  fmt="{:.2f}", colors=[_WEB_INK_3, _WEB_OCHRE])

    if "prompt_ablation" in llm_tests and "error" not in llm_tests["prompt_ablation"]:
        a = llm_tests["prompt_ablation"]
        _web_paired("llm-prompt-ablation.svg", ["EN to CRK", "CRK to EN"],
                    [a["en_cr"]["with_prefix"], a["cr_en"]["with_prefix"]],
                    [a["en_cr"]["without_prefix"], a["cr_en"]["without_prefix"]],
                    first_label="with control tokens", second_label="plain instruction",
                    ylab="chrF++", fmt="{:.1f}")

    if "english_retention" in llm_tests and "error" not in llm_tests["english_retention"]:
        r = llm_tests["english_retention"]
        _web_bars("llm-retention.svg", ["base", "tuned"],
                  [r["base_perplexity"], r["tuned_perplexity"]],
                  ylab="held-out English perplexity", ymax=max(r["tuned_perplexity"], 1) * 1.25,
                  fmt="{:.2f}", colors=[_WEB_INK_3, _WEB_OCHRE])

    if "calibration" in llm_tests and "error" not in llm_tests["calibration"]:
        c = llm_tests["calibration"]
        _web_scatter("llm-calibration.svg", c["confidence"], c["sentence_chrf"],
                     xlab="mean token log-probability", ylab="sentence chrF++")

    figures = {}
    for name, caption in (
        ("llm-checks.svg", "behaviour checks"),
        ("llm-fertility.svg", "tokens per Cree word"),
        ("llm-prompt-ablation.svg", "chrF++ with and without control tokens"),
        ("llm-retention.svg", "held-out English perplexity"),
        ("llm-calibration.svg", "confidence against quality"),
    ):
        if (WEB_DIR / name).exists():
            figures[name] = caption

    headline = {
        "chrf_en_cr": _report["tuned"]["en_cr"]["chrf++"],
        "chrf_cr_en": _report["tuned"]["cr_en"]["chrf++"],
        "chrf_en_cr_base": _report["base"]["en_cr"]["chrf++"],
        "chrf_cr_en_base": _report["base"]["cr_en"]["chrf++"],
        "validity": _report["tuned"]["en_cr"].get("validity"),
        "code_switch": _report["tuned"]["en_cr"].get("code_switch"),
        "perplexity": _report["tuned"].get("perplexity"),
        "perplexity_base": _report["base"].get("perplexity"),
        "time_to_first_token_note": "measured on the training GPU, not a laptop",
        # The per-job comparison against the old setup, so the site can show the
        # same bar the report card prints instead of restating a claim.
        "head_to_head": _report.get("head_to_head"),
    }

    jdump(OUT / "llm_tests.json", llm_tests)
    _report["tests"] = llm_tests
    jdump(_report_path, _report)
    jdump(WEB_DIR / "cree-llm.json", {
        "meta": llm_tests["meta"],
        "headline": headline,
        "tests": {k: v for k, v in llm_tests.items() if k != "meta"},
        "figures": figures,
    })

    # The site's data section is generated from the run's own licence ledger,
    # not retyped into a page. A source that was skipped, or a licence that
    # changed, is reflected the next time this cell runs.
    try:
        _ledger = jload(RAW / "ledger.json")
        _stats = jload(PROC / "stats.json")
        jdump(WEB_DIR / "corpus.json", {
            "sources": _ledger,
            "stats": _stats,
            "note": "Generated from the run's licence ledger. NoDerivatives material is "
                    "excluded at the loader, so allowed is false for those entries.",
        })
    except Exception as exc:
        log(f"site corpus bundle unavailable: {exc}")

    mark_stage("llmtests", _TESTS_KEY,
               deterministic=_value("determinism", "rate"),
               abstention=_value("abstention", "rate"))

    print("\n" + "=" * 78)
    print(f"{'LLM checks':22s}  result")
    print("=" * 78)
    if "determinism" in llm_tests and "error" not in llm_tests["determinism"]:
        d = llm_tests["determinism"]
        print(f"{'same answer twice':22s}  {d['identical']}/{d['n']} ({100 * d['rate']:.0f}%)")
    if "abstention" in llm_tests and "error" not in llm_tests["abstention"]:
        a = llm_tests["abstention"]
        print(f"{'declines the unknown':22s}  {a['abstained']}/{a['unanswerable']} "
              f"({100 * a['rate']:.0f}%), false abstentions {a['false_abstain']}/{a['answerable']}")
    if "prompt_ablation" in llm_tests and "error" not in llm_tests["prompt_ablation"]:
        for direction, block in llm_tests["prompt_ablation"].items():
            print(f"{'control tokens ' + direction:22s}  chrF++ {block['with_prefix']:.2f} vs "
                  f"{block['without_prefix']:.2f} plain ({block['delta']:+.2f})")
    if "fertility" in llm_tests and "error" not in llm_tests["fertility"]:
        f = llm_tests["fertility"]
        print(f"{'tokens per Cree word':22s}  {f['base_tokens_per_word']:.2f} -> "
              f"{f['cree_tokens_per_word']:.2f} ({f['reduction_percent']:.1f}% fewer)")
    if "english_retention" in llm_tests and "error" not in llm_tests["english_retention"]:
        r = llm_tests["english_retention"]
        print(f"{'English perplexity':22s}  {r['base_perplexity']:.2f} -> {r['tuned_perplexity']:.2f} "
              f"({r['change_percent']:+.1f}%)")
    if "calibration" in llm_tests and "error" not in llm_tests["calibration"]:
        print(f"{'confidence vs quality':22s}  r = {llm_tests['calibration']['pearson']:+.2f}")
    print("=" * 78)
    print(f"wrote {OUT / 'llm_tests.json'} and {WEB_DIR}")

In [ ]:
# The report card.
#
# A model without a data card is a rumour. This writes down what went in, what
# was excluded and why, how it was trained, what it scores, and what it must not
# be used for. It is generated from the run's own files, so it cannot drift from
# the numbers in the notebook.
report = jload(OUT / "creebench_report.json")
stats = jload(PROC / "stats.json")
tok_audit = jload(PROC / "tokenizer_audit.json")
ledger = jload(RAW / "ledger.json")
status = jload(RAW / "source_status.json")

licences = " ".join(item["licence"].lower() for item in ledger)
if "noncommercial" in licences or "non-commercial" in licences or "nc-" in licences:
    model_licence = "CC BY-NC-SA 4.0 (inherited from NonCommercial corpus material)"
elif "sharealike" in licences or "by-sa" in licences:
    model_licence = "CC BY-SA 4.0 (inherited from ShareAlike corpus material)"
else:
    model_licence = "CC BY 4.0 for the adapter; base model licence applies to the weights"

b, t = report["base"], report["tuned"]


def pct(x: float) -> str:
    return f"{100 * x:.1f}%"


table_rows = [
    ("EN → CRK", "chrF++", b["en_cr"]["chrf++"], t["en_cr"]["chrf++"]),
    ("EN → CRK", "BLEU", b["en_cr"]["bleu"], t["en_cr"]["bleu"]),
    ("EN → CRK", "token F1", b["en_cr"]["token_f1"], t["en_cr"]["token_f1"]),
    ("EN → CRK", "Cree word validity", b["en_cr"].get("validity", 0), t["en_cr"].get("validity", 0)),
    ("EN → CRK", "English code-switch (lower)", b["en_cr"].get("code_switch", 0), t["en_cr"].get("code_switch", 0)),
    ("CRK → EN", "chrF++", b["cr_en"]["chrf++"], t["cr_en"]["chrf++"]),
    ("CRK → EN", "BLEU", b["cr_en"]["bleu"], t["cr_en"]["bleu"]),
    ("CRK → EN", "token F1", b["cr_en"]["token_f1"], t["cr_en"]["token_f1"]),
]

# The two jobs the old stack handled with a retrieval index and a second model
# are only comparable if the base model's scores are in the same table.
for _task, _label, _key in (("Song notes", "token F1", "note_reading"),
                            ("Conversation", "token F1", "project")):
    if _key in b and _key in t:
        table_rows.append((_task, _label, b[_key]["token_f1"], t[_key]["token_f1"]))
if "abstention" in b and "abstention" in t:
    table_rows.append(("Song notes", "abstention", b["abstention"]["accuracy"], t["abstention"]["accuracy"]))

# Which adapter these numbers belong to. A card that does not say is how a
# scoring run of the instruction-tuned model gets mistaken for the shipped one.
_scored = best_adapter()

lines = [
    "---",
    "language: crk",
    "language_name: nêhiyawêwin (Plains Cree)",
    "base_model: " + report["meta"]["base_model"],
    f"licence: {model_licence}",
    f"adapter: {_scored.name if _scored is not None else 'base model'}",
    f"trained: {report['meta']['timestamp']}",
    "tags: [cree, nehiyawewin, low-resource, lora, gguf]",
    "---",
    "",
    "# The Archer assistant model",
    "",
    "The one model the Archer app runs. The same weights hold the conversation about the project, answer questions about Archer's songs from their notes, translate between English and nêhiyawêwin in Standard Roman Orthography and syllabics, and explain Cree words. There is no second chat model and no router.",
    "",
    "It is not Archer, it does not sing in Archer's voice, and it is not a substitute for a fluent speaker or an Elder.",
    "",
]
if SMOKE:
    lines += [
        "> **Smoke run.** This report came from the tiny CPU profile, whose job is to",
        "> prove the pipeline runs. It uses a small slice of the corpus",
        "> and a 0.5B model, so the word count and scores below are not the model.",
        "> The `t4`, `big24`, `a100`, `big` and `huge` profiles use the complete corpus",
        "> and the 7B, 14B or 32B base model.",
        "",
    ]
lines += [
    "## Results on held-out data",
    "",
    "| Task | Measure | Base | Tuned |",
    "|---|---|---:|---:|",
]
for task, measure, bv, tv in table_rows:
    lines.append(f"| {task} | {measure} | {bv:.3f} | {tv:.3f} |")

# The claim this release rests on. The bar is the strongest score any piece of
# the old setup produced on that job -- the general model, the separate Cree
# model, or the general model reading retrieved notes -- and the verdict is
# computed, not written by hand.
_h2h = report.get("head_to_head") or {}
if _h2h.get("rows"):
    lines += [
        "",
        "### Against the best score measured before it",
        "",
        "The old app was a general model, a separate Cree model and a retrieval index, "
        "with a router choosing between them. The bar below is the best score on each "
        "job from a measurement this run actually made, and the `From` column says "
        "which model set it. In a run without `--external-model` every bar is this "
        "model's own base, so the table measures the training rather than a win over "
        "the separate Cree model, which has not been scored on these sentences.",
        "",
        "| Job | Measure | This model | Best measured before | From | Verdict |",
        "|---|---|---:|---:|---|---|",
    ]
    for _row in _h2h["rows"]:
        lines.append(
            f"| {_row['task']} | {_row['measure']} | {_row['tuned']:.3f} | {_row['bar']:.3f} "
            f"| {_row['bar_from']} | {'meets or beats' if _row['meets'] else 'below'} |"
        )
    lines += ["", f"**{_h2h['verdict'].capitalize()}.**", ""]
if report.get("external") and "en_cr" in report["external"]:
    ext = report["external"]
    lines += [
        "",
        f"### Head-to-head: `{ext['model']}`",
        "",
        "The same held-out sentences and metrics, sent through Ollama without this model's control tokens:",
        "",
        "| Task | chrF++ | BLEU | token F1 | Cree validity |",
        "|---|---:|---:|---:|---:|",
        f"| EN → CRK | {ext['en_cr']['chrf++']:.3f} | {ext['en_cr']['bleu']:.3f} | {ext['en_cr']['token_f1']:.3f} | {ext['en_cr'].get('validity', 0):.3f} |",
        f"| CRK → EN | {ext['cr_en']['chrf++']:.3f} | {ext['cr_en']['bleu']:.3f} | {ext['cr_en']['token_f1']:.3f} | - |",
    ]
lines += [
    "",
    f"- Held-out perplexity: **{b['perplexity']:.2f} → {t['perplexity']:.2f}**",
    f"- Syllabics output: **{pct(b['syllabics']['script_share'])} → {pct(t['syllabics']['script_share'])}** of letters in syllabics",
    f"- Abstention: **{pct(t['abstention']['accuracy'])}** of unanswerable questions correctly declined",
]
if t["en_cr"].get("script_share") is not None:
    # A script failure and a translation failure look identical in chrF++; this
    # is the line that tells them apart.
    _sro = 100 * (1 - t["en_cr"]["script_share"])
    lines.append(
        f"- Script respected (EN → CRK): **{_sro:.1f}%** of letters in SRO when SRO was asked; "
        f"the same replies converted to SRO score chrF++ {t['en_cr'].get('chrf++_as_sro', 0):.3f}"
    )
if "note_reading" in t:
    lines.append(f"- Reading an unseen note section: **token F1 {t['note_reading']['token_f1']:.3f}**")

# The language-model checks measure the failures a translation score cannot
# see: instability, invented confidence, forgotten English, and the cost of a
# Cree word. They are reported here so the card is still the whole story.
_lt = report.get("tests") or {}
_checks: list[tuple[str, str, str]] = []
if _lt.get("determinism", {}).get("n"):
    d = _lt["determinism"]
    _checks.append(("Same answer twice", f"{d['identical']}/{d['n']}", "greedy decoding is stable"))
if _lt.get("abstention", {}).get("unanswerable"):
    a = _lt["abstention"]
    _checks.append(("Declines the unknown", pct(a["rate"]),
                    f"false abstentions {a['false_abstain']}/{a['answerable']}"))
if _lt.get("fertility", {}).get("reduction_percent") is not None:
    f = _lt["fertility"]
    _checks.append(("Tokens per Cree word",
                    f"{f['base_tokens_per_word']:.2f} to {f['cree_tokens_per_word']:.2f}",
                    f"{f['reduction_percent']:.1f}% fewer than the base tokenizer"))
if _lt.get("english_retention", {}).get("change_percent") is not None:
    r = _lt["english_retention"]
    _checks.append(("English perplexity",
                    f"{r['base_perplexity']:.2f} to {r['tuned_perplexity']:.2f}",
                    f"{r['change_percent']:+.1f}% change, the replay working"))
if _lt.get("prompt_ablation", {}).get("en_cr"):
    p = _lt["prompt_ablation"]["en_cr"]
    _checks.append(("Control tokens, EN to CRK",
                    f"{p['with_prefix']:.2f} vs {p['without_prefix']:.2f} chrF++",
                    "what the prefixes are worth"))
if _lt.get("syllabics", {}).get("n"):
    s = _lt["syllabics"]
    _checks.append(("Syllabics, written and read back", pct(s["script_share"]),
                    f"chrF++ {s['roundtrip_chrf']:.1f} after conversion back to SRO"))
if _lt.get("calibration", {}).get("n"):
    _checks.append(("Confidence against quality",
                    f"r = {_lt['calibration']['pearson']:+.2f}",
                    "a model that is confidently wrong is worse than one that is unsure"))
if _checks:
    lines += ["", "### Language model checks", "", "| Check | Result | Note |", "|---|---|---|"]
    for _name, _result, _note in _checks:
        lines.append(f"| {_name} | {_result} | {_note} |")
    lines.append("")
    lines.append("The charts behind these numbers are in `plots/`; the copies sized for the "
                 "project site are in `out/web/` along with `cree-llm.json`.")
lines += [
    "",
    f"All scores are on {report['meta']['n_pairs']} sentence pairs the model never saw. Full per-task output is in `creebench_report.json`.",
    "",
    "## Data",
    "",
    "| Source | Licence | Dialect | Used for |",
    "|---|---|---|---|",
]
for item in ledger:
    lines.append(f"| {item['source']} | {item['licence'][:90]} | {item['dialect']} | {item['note']} |")

lines += [
    "",
    f"Training sentences: **{stats['sentence_words']['train']:,} words** across {stats['sentences']['train']:,} sentences; "
    f"**{stats['pairs']['train']:,} aligned pairs**; **{stats['lexicon_forms']:,} lexicon forms** used for validity checks.",
    "",
    "NoDerivatives material in the EdTeKLA collection is excluded at the loader. Bag-of-words files are used for vocabulary counts only, never as prose. Near-duplicate sentences are removed and the test split is document-disjoint, with a second leakage pass.",
    "",
    "## Tokenizer",
    "",
    f"- Tokens per Cree word: **{tok_audit['base']['fertility']:.2f} → {tok_audit['extended']['fertility']:.2f}** ({tok_audit['reduction_percent']:.1f}% fewer)",
    f"- Characters per token: **{tok_audit['base']['chars_per_token']:.2f} → {tok_audit['extended']['chars_per_token']:.2f}**",
    "- New tokens start from the average of the base model's embeddings for the pieces they replace. Those rows are kept out of the optimizer, so the 1.1 billion row vocabulary table never has to hold gradients or Adam state; the adapter is what learns to read and emit them.",
    "",
    "## Training recipe",
    "",
    f"| Stage | Data | Schedule |",
    "|---|---|---|",
    f"| Continued pretraining | Cree prose + 10% English replay | {CFG.cpt_epochs} epoch(s), lr {CFG.cpt_lr}, LoRA r={CFG.lora_r} |",
    f"| Instruction tuning | translation, orthography, notes, abstention, persona | {CFG.sft_epochs} epoch(s), lr {CFG.sft_lr} |",
    f"| Preference tuning | faithfulness pairs (dropped clauses, English leaks, flattened vowels) | {CFG.dpo_epochs} epoch(s), beta {CFG.dpo_beta} |",
    "",
    "Packing removes padding from pretraining. Instruction tuning masks the prompt and scores only the reply. DPO keeps a frozen copy of the instruction-tuned adapter as its reference, loaded beside the policy adapter, so preference tuning regularises towards what instruction tuning built instead of towards the base model.",
    "",
    "## Intended use",
    "",
    "- Translating short sentences and phrases between English and Plains Cree.",
    "- Reading and writing SRO and syllabics.",
    "- Answering questions about Archer's songs from the supplied notes.",
    "- Drafting Cree text that a fluent speaker then reviews.",
    "",
    "## Out of scope",
    "",
    "- Ceremonial, legal, medical or treaty language.",
    "- Any claim that a translation is authoritative without review by a fluent speaker.",
    "- Generating new songs or lyrics in Cree.",
    "- Speaking as Archer or imitating his voice.",
    "",
    "## Attribution",
    "",
    "The Cree corpus is built on the EdTeKLA Indigenous Languages Corpora (University of Alberta), the Plains Cree Figurative Language corpus (KonradBRG, CC BY 4.0), Cree Wikipedia (CC BY-SA 4.0), FineWeb-2, FinePDFs, FineTranslations, GlotCC and GlotStoryBook (ODC-BY 1.0 and CC BY-NC-SA where stated), Tatoeba (CC BY 2.0 FR), the Internet Archive's public-domain Cree books, and the GiellaLT lang-crk lexicon and public texts (ALTLab, University of Alberta, with the Maskwacîs Education Schools Commission and First Nations University). The orthography tables follow GiellaLT's Latn-to-Cans and Cans-to-Latn transducers.",
    "",
    "The song notes are Archer Pechawis's own. They are read from the project's `song_notes/` directory to build grounded question and answer examples and are not included as a text corpus in the release.",
    "",
    "## Files",
    "",
    "| File | What it is |",
    "|---|---|",
    f"| `{MODEL_STEM}.Q4_K_M.gguf` | quantised model for Ollama / llama.cpp |",
    "| `Modelfile` | context length and sampling settings |",
    "| `creebench_report.json` | every score and the generated examples |",
    "| `training_curves.png`, `creebench.png`, `tokenizer_fertility.png` | figures |",
    "",
]

(OUT / "report_card.md").write_text("\n".join(lines), encoding="utf-8")
print((OUT / "report_card.md").read_text(encoding="utf-8")[:2800])
print(f"\nwrote {OUT / 'report_card.md'}")

# the run is only finished when these are true
assert (OUT / "creebench_report.json").exists()
assert t["en_cr"]["chrf++"] == t["en_cr"]["chrf++"], "NaN translation score"
assert t["perplexity"] > 0
assert stats["sentences"]["test"] > 0
print("\nreport checks passed")

## 7. Export

The app runs on a laptop, so the one model has to leave Colab as a quantised GGUF.

There is one complication. llama.cpp can apply a LoRA to a base GGUF, but not a LoRA that added vocabulary, and this one added several thousand Cree tokens. The adapter is therefore merged into a full-precision copy of the base model, and that merged directory is converted. Merging needs room for a second copy of the model; the cell reports the device it chose and stops with a clear message if neither the GPU nor the CPU has the memory.

The quantisation is Q4_K_M, which is the usual quality-per-byte point: about 4.5 GB for the 7B and about 9 GB for the 14B, large enough that the Cree tokens are not crushed. The Modelfile carries the sampling settings. The system prompt is sent by the app on every request rather than baked into the tag, because Ollama would otherwise place it in the context twice. There is one artifact and one tag, `archer:<size>`, where `<size>` comes from the base model: `archer:14b` for the 14B run and `archer:7b` for the 7B run. It serves every chat request, including translation and song-note questions.

The last cell writes the report card and the exact commands for wiring the model into this project's config.

In [ ]:
# Export: adapter -> merged weights -> GGUF -> Ollama.
#
# llama.cpp cannot apply a LoRA that adds vocabulary, and this model adds Cree
# tokens. The adapter is therefore merged into a full-precision copy of the base
# model first, and the merged directory is what gets converted. Merging needs
# roughly twice the model's fp16 size in RAM or VRAM; the cell says which device
# it chose and stops with a clear message if neither has room.
from peft import PeftModel

# The merged fp16 model is an intermediate, not an artifact: it is 15.2 GB for
# the 7B and can always be rebuilt from the adapter. On Colab it goes to the
# local disk, because a free Drive is 15 GB in total and filling it during the
# longest run in the notebook would fail the export after hours of training.
# Only the GGUF, the Modelfile and the archive are written to Drive.
EXPORT = Path("/content/cree_llm_export") if IN_COLAB else OUT / "export"
EXPORT.mkdir(parents=True, exist_ok=True)

# The file this cell actually shipped. Stays None on a scoring run or a smoke
# run, where nothing is quantised and the closing instructions must still print
# something accurate instead of raising.
final_gguf: Path | None = None


def _cmake() -> str | None:
    """A cmake to build the quantiser with, installed without root if needed.

    The export is the only step in this pipeline that needs a toolchain, and a
    shared cluster login node often has a compiler but no cmake. `pip install
    cmake` puts one in the run's own virtualenv, which is the one place this
    script is allowed to write. Nothing here touches the system Python.
    """
    found = shutil.which("cmake")
    if found:
        return found

    def _from_the_package() -> str | None:
        try:
            import cmake as _cmake_pkg
            candidate = Path(_cmake_pkg.CMAKE_BIN_DIR) / "cmake"
            return str(candidate) if candidate.exists() else None
        except Exception:
            return None

    packaged = _from_the_package()
    if packaged:
        return packaged
    log("cmake: not on PATH, installing it into this virtualenv")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake"],
                       check=True, timeout=900)
    except Exception as exc:
        log(f"cmake: installation failed ({exc})")
        return None
    return shutil.which("cmake") or _from_the_package()


def _quantiser(llama: Path) -> Path | None:
    """The llama-quantize binary, built once, or None if this host cannot build it.

    Returning None is not a failure. The Python converter can write a Q8_0 file
    on its own, and that is a model Ollama runs exactly the same way. A finished
    model that is 4 GB larger beats no model at all.
    """
    for candidate in (llama / "build" / "bin" / "llama-quantize",
                      llama / "build" / "llama-quantize"):
        if candidate.exists():
            return candidate
    on_path = shutil.which("llama-quantize")
    if on_path:
        return Path(on_path)

    cmake = _cmake()
    if cmake is None:
        return None
    log("llama.cpp: building llama-quantize (once per work folder)")
    try:
        subprocess.run([cmake, "-B", "build", "-DGGML_NATIVE=OFF", "-DLLAMA_CURL=OFF"],
                       cwd=llama, check=True)
        subprocess.run([cmake, "--build", "build", "--config", "Release",
                        "--target", "llama-quantize",
                        "--parallel", str(max(1, os.cpu_count() or 4))],
                       cwd=llama, check=True)
    except subprocess.CalledProcessError as exc:
        log(f"llama.cpp: build failed ({exc}); the export will write Q8_0 instead")
        return None
    for candidate in (llama / "build" / "bin" / "llama-quantize",
                      llama / "build" / "llama-quantize"):
        if candidate.exists():
            return candidate
    return None

if os.environ.get("CREE_ADAPTER_STAGE", "").strip():
    # A scoring run must not overwrite the shipped weights with an adapter it
    # was only asked to measure.
    log("export skipped: scoring run (CREE_ADAPTER_STAGE is set)")
elif SMOKE:
    log("SMOKE: skipping merge and GGUF; packaging the adapter only")
    zip_path = OUT / "archer-smoke.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zf:
        for path in (CKPT / "dpo").rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(ROOT))
        zf.write(OUT / "creebench_report.json", "creebench_report.json")
    print(f"wrote {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
else:
    # Free the training model before asking for a second full copy.
    try:
        del model
    except NameError:
        pass
    gc.collect()

    # This cell is often the first one run after a reconnect, so nothing it
    # needs is assumed to be in the kernel except the files on Drive.
    from transformers import AutoTokenizer as _AutoTokenizer

    tok = _AutoTokenizer.from_pretrained(PROC / "tokenizer")

    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3 if torch.cuda.is_available() else 0
    import psutil

    ram_gb = psutil.virtual_memory().total / 1e9
    # Merging holds the fp16 base and the fp16 merged copy at the same time,
    # plus the adapter and transient buffers: roughly 4.4 GB per billion
    # parameters. A 24 GB threshold that ignores model size is how a 14B or 32B
    # export ends in an OOM after hours of training.
    params_b = {"0.5b": 0.5, "1.5b": 1.5, "3b": 3, "7b": 7, "14b": 14, "32b": 32}.get(MODEL_TAG, 7)
    need_gb = 4.4 * params_b
    force = os.environ.get("CREE_MERGE_DEVICE", "")
    merge_on_gpu = False if force == "cpu" else (gpu_gb >= need_gb)
    if force == "cuda" and gpu_gb < need_gb:
        raise RuntimeError(
            f"Merging {MODEL_STEM} needs about {need_gb:.0f} GB of GPU memory; this GPU has {gpu_gb:.0f} GB. "
            "Unset CREE_MERGE_DEVICE to let the cell choose CPU, or use a larger GPU."
        )
    target = "cuda" if merge_on_gpu else "cpu"
    if not merge_on_gpu and ram_gb < need_gb:
        raise RuntimeError(
            f"Merging {MODEL_STEM} needs about {need_gb:.0f} GB of CPU RAM (fp16 base plus fp16 merged copy); "
            f"this runtime has {ram_gb:.0f} GB and {gpu_gb:.0f} GB of GPU memory. "
            "Use Runtime -> Change runtime type -> High-RAM, or a larger GPU for the merge. "
            "The trained adapter and report are already saved; nothing is lost."
        )

    # Every artifact is checked against the adapter's own timestamp, not just
    # against existence. Retraining one stage and re-running this cell used to
    # keep the previous GGUF, because a file was there and large: the export
    # would report success while shipping the old weights.
    def _newest(path: Path) -> float:
        return max((p.stat().st_mtime for p in path.rglob("*") if p.is_file()), default=0.0)

    adapter_mtime = _newest(CKPT / "dpo")
    merged_dir = EXPORT / "merged"
    merged_shards = sorted(merged_dir.glob("*.safetensors"))
    merged_ok = bool(merged_shards) and (merged_dir / "config.json").exists() and not any(
        safetensors_problem(p) for p in merged_shards
    ) and _newest(merged_dir) >= adapter_mtime
    if merged_shards and not merged_ok:
        log("merged model is older than the adapter it should contain; merging again")
    if merged_ok:
        log(f"merged model already complete at {merged_dir}, skipping the merge")
    else:
        log(f"merging on {target} (GPU {gpu_gb:.0f} GB, RAM {ram_gb:.0f} GB)")

        base = AutoModelForCausalLM.from_pretrained(
            ensure_base_weights(),
            **_dtype_kwargs(torch.float16),
            low_cpu_mem_usage=True,
            device_map={"": 0} if merge_on_gpu else {"": "cpu"},
        )
        # The graft is restored here rather than resized blank, otherwise the
        # merge would write random rows for every Cree token the adapter was
        # trained on.
        graft_rows(base, tok)
        merged = PeftModel.from_pretrained(base, str(CKPT / "dpo"))
        merged = merged.merge_and_unload()
        merged.save_pretrained(str(merged_dir), safe_serialization=True, max_shard_size="4GB")
        tok.save_pretrained(merged_dir)
        del merged, base
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        log(f"merged model saved to {merged_dir}")

    q4 = EXPORT / f"{MODEL_STEM}.Q4_K_M.gguf"
    q8 = EXPORT / f"{MODEL_STEM}.Q8_0.gguf"

    def _fresh(path: Path) -> bool:
        return (path.exists() and path.stat().st_size > 1_000_000_000
                and path.stat().st_mtime >= adapter_mtime)

    for candidate in (q4, q8):
        if candidate.exists() and not _fresh(candidate):
            log(f"{candidate.name} predates the adapter; removing it")
            candidate.unlink()

    final_gguf = q4 if _fresh(q4) else (q8 if _fresh(q8) else None)
    if final_gguf is not None:
        log(f"{final_gguf.name} already built ({final_gguf.stat().st_size / 1e9:.2f} GB), skipping conversion")
    else:
        # llama.cpp: convert, then quantise
        LLAMA = RAW / "llama.cpp"
        if not (LLAMA / "convert_hf_to_gguf.py").exists():
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/ggml-org/llama.cpp.git", str(LLAMA)], check=True)
        # Only the one package the converter needs and the runtime does not
        # already have. Installing llama.cpp's full requirements here can upgrade
        # numpy or transformers under the not-yet-saved report, so it is
        # scoped.
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf"], check=True)

        # The f16 is an intermediate, and a run that died after converting
        # leaves one behind. Quantising a file that predates the adapter would
        # write the previous weights under the new name, so it follows the same
        # rule as the merged copy: it must be newer than what it came from.
        f16 = EXPORT / f"{MODEL_STEM}-f16.gguf"
        if f16.exists() and f16.stat().st_mtime < adapter_mtime:
            log(f"{f16.name} predates the adapter; converting it again")
            f16.unlink()
        if not f16.exists():
            subprocess.run([
                sys.executable, str(LLAMA / "convert_hf_to_gguf.py"), str(merged_dir),
                "--outfile", str(f16), "--outtype", "f16",
            ], check=True)

        quant = _quantiser(LLAMA)
        if quant is None:
            log("llama.cpp cannot be built here; writing Q8_0 with the Python converter instead of Q4_K_M")
            subprocess.run([
                sys.executable, str(LLAMA / "convert_hf_to_gguf.py"), str(merged_dir),
                "--outfile", str(q8), "--outtype", "q8_0",
            ], check=True)
            final_gguf = q8
        else:
            subprocess.run([str(quant), str(f16), str(q4), "Q4_K_M"], check=True)
            final_gguf = q4
        f16.unlink(missing_ok=True)
        if not final_gguf.exists() or final_gguf.stat().st_size < 1_000_000_000:
            raise RuntimeError(f"quantisation did not produce a usable GGUF at {final_gguf}")

    modelfile = EXPORT / "Modelfile"
    # The system prompt lives in the app (chat/archer_prompt.py) and is sent on
    # every request. Baking it into the Modelfile as well would put it in the
    # context twice, once from .System and once as the system message.
    modelfile.write_text(
        f"FROM ./{final_gguf.name}\n"
        "PARAMETER temperature 0.3\n"
        "PARAMETER top_p 0.9\n"
        "PARAMETER repeat_penalty 1.08\n"
        "PARAMETER num_ctx 4096\n",
        encoding="utf-8",
    )

    zip_path = OUT / f"{MODEL_STEM}.zip"
    room_gb = shutil.disk_usage(str(OUT)).free / 1e9
    if zip_path.exists() and zip_path.stat().st_mtime >= final_gguf.stat().st_mtime:
        log(f"{zip_path.name} already packaged, skipping the archive")
    elif room_gb < final_gguf.stat().st_size / 1e9 * 1.1:
        # The archive is a copy of the GGUF plus the report. Copying it with no
        # room left leaves a truncated zip and no way to finish the export.
        log(f"skipping the archive: {room_gb:.1f} GB free on Drive, {final_gguf.stat().st_size / 1e9:.1f} GB needed")
        log(f"the GGUF itself is complete at {final_gguf}; move it off Drive to make room for the zip")
    else:
        digest = hashlib.sha256()
        with final_gguf.open("rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)

        manifest = {
            "gguf": final_gguf.name,
            "quantisation": final_gguf.name.split(".")[-2],
            "size_gb": round(final_gguf.stat().st_size / 1e9, 2),
            "sha256": digest.hexdigest(),
            "base_model": CFG.base_model,
            "profile": ACTIVE_PROFILE,
            "created": time.strftime("%Y-%m-%d %H:%M"),
        }
        jdump(EXPORT / "manifest.json", manifest)
        print(json.dumps(manifest, indent=2))

        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zf:
            zf.write(final_gguf, final_gguf.name)
            zf.write(modelfile, "Modelfile")
            zf.write(EXPORT / "manifest.json", "manifest.json")
            zf.write(OUT / "creebench_report.json", "creebench_report.json")
            zf.write(OUT / "report_card.md", "report_card.md")
            for png in PLOTS.glob("*.png"):
                zf.write(png, f"plots/{png.name}")
        print(f"wrote {zip_path} ({zip_path.stat().st_size / 1e9:.2f} GB)")

        if IN_COLAB:
            try:
                from google.colab import files

                files.download(str(final_gguf))
                files.download(str(zip_path))
            except Exception as exc:
                print(f"[note] automatic download failed ({exc}); use the Files pane")

print("\nInstall the model on the machine that runs the app")
_shipped = final_gguf.name if final_gguf is not None else f"{MODEL_STEM}.Q4_K_M.gguf"
print(textwrap.dedent(f"""
    mkdir -p assets/archer_llm
    cp {_shipped} assets/archer_llm/
    cp Modelfile assets/archer_llm/
    cd assets/archer_llm && ollama create archer:{MODEL_TAG} -f Modelfile

    # config.yaml
    chat:
      model: archer:{MODEL_TAG}
"""))

## What just happened

You now have the one Archer model, built from a ledgered corpus, tokenised for Cree rather than for English, trained in three stages, and scored against its own base model on data it never saw. It answers conversation, reads song notes, translates and converts scripts; the app does not run a second chat model beside it.

The files that matter:

| File | What to do with it |
|---|---|
| `out/export/archer-<size>.Q4_K_M.gguf` | copy to `assets/archer_llm/` and load with Ollama |
| `out/export/Modelfile` | `ollama create archer:<size> -f Modelfile` |
| `out/report_card.md` | publish beside the model; it lists every source and licence |
| `out/creebench_report.json` | the evidence behind every number |
| `out/plots/` | the figures used in the report |

### A note on what this model is not

It is not a fluent speaker, and it should not be presented as one. It is a careful reader with a good memory for this corpus, a tokenizer that respects the spelling, and an instruction set that rewards saying "the notes do not say". Every translation that leaves the app should be reviewable by a fluent speaker, and the report card says so.

### The next run

The recipe is built to improve with data, not with parameter count. The highest-value additions are consented contemporary Cree text, more song notes, and corrections from fluent speakers. Drop them into the upload folder, rerun from the corpus cell, and the same evaluation will say whether they helped.